# CHƯƠNG 4 — KNOWLEDGE GRAPH VÀ RAG

**Học phần:** Phát triển các hệ thống thông minh  
**Sinh viên:** Kiều Trường Giang  
**Mã sinh viên:** B23DCCN254  
**Tệp nộp:** `Tieuluan02_06_Nhom2_Giangkt.ipynb`

Notebook này là điểm chạy duy nhất của bài. Khi chọn **Run All**, notebook sẽ:

1. tải và xử lý dữ liệu tiểu đường, chứng khoán FPT và giá vàng;
2. xây dựng Knowledge Graph cho ba miền;
3. triển khai Vector RAG và GraphRAG không cần API key;
4. đánh giá, trực quan hóa và tạo ứng dụng dự đoán cùng chatbot RAG;
5. sinh báo cáo PDF `Tieuluan02_06_Nhom2_Giangkt.pdf`;
6. chuẩn bị thư mục `vercel_app/` để triển khai nhanh lên Vercel.

> Kết quả tài chính thay đổi theo thời điểm chạy. Nội dung y tế chỉ phục vụ minh họa học thuật,
> không thay thế chẩn đoán hoặc chỉ định của bác sĩ.

In [1]:
# 0. CẤU HÌNH VÀ PHỤ THUỘC
import base64
import importlib.util
import json
import math
import os
import subprocess
import sys
import textwrap
import warnings
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig_giangkt")
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
BASE_NAME = "Tieuluan02_06_Nhom2_Giangkt"
REPORT_PATH = ROOT / f"{BASE_NAME}.pdf"
ARTIFACT_DIR = ROOT / "artifacts_giangkt"
DATA_DIR = ARTIFACT_DIR / "data"
FIG_DIR = ARTIFACT_DIR / "figures"
WEB_DIR = ROOT / "vercel_app"
for folder in (ARTIFACT_DIR, DATA_DIR, FIG_DIR, WEB_DIR):
    folder.mkdir(parents=True, exist_ok=True)

STUDENT_NAME = "Kiều Trường Giang"
STUDENT_ID = "B23DCCN254"
GITHUB_URL = "https://github.com/jellalaz/RAG_HTTM"
DEPLOY_URL = "https://rag-httm-chatbot.vercel.app"
COURSE = "Phát triển các hệ thống thông minh"
SCHOOL = "HỌC VIỆN CÔNG NGHỆ BƯU CHÍNH VIỄN THÔNG"
FACULTY = "KHOA CÔNG NGHỆ THÔNG TIN 1"
RUN_AT = datetime.now().astimezone()
RANDOM_SEED = 42

# Nạp ReportLab từ môi trường dự phòng nếu kernel hiện tại chưa nhận thư viện.
# bằng kernel hệ thống nếu package chưa nằm trong sys.path.
bundled_site = Path("/home/jellalaz/.cache/codex-runtimes/codex-primary-runtime/dependencies/python")
if importlib.util.find_spec("reportlab") is None and bundled_site.exists():
    sys.path.append(str(bundled_site))

required = {
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
    "sklearn": "scikit-learn", "networkx": "networkx",
    "requests": "requests", "reportlab": "reportlab", "pypdf": "pypdf", "PIL": "Pillow"
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Đang cài các thư viện còn thiếu:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests
from sklearn.datasets import load_diabetes
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, KFold, TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

plt.rcParams.update({
    "font.family": "Liberation Serif", "font.size": 11,
    "figure.dpi": 180, "savefig.dpi": 240,
    "text.color": "black", "axes.labelcolor": "black",
    "axes.edgecolor": "black", "xtick.color": "black", "ytick.color": "black"
})
np.random.seed(RANDOM_SEED)
print(f"Khởi tạo thành công lúc {RUN_AT:%d/%m/%Y %H:%M:%S %Z}")
print("Thư mục làm việc:", ROOT)

Khởi tạo thành công lúc 16/10/2026 05:36:57 +07
Thư mục làm việc: /home/jellalaz/Documents/Jellalaz/DATA_CODE/PYTHON/RAG


## 1. Tải và xử lý dữ liệu

Ba miền được lựa chọn có đặc trưng rất khác nhau: dữ liệu y tế dạng bảng, chuỗi thời gian
cổ phiếu bằng VND và chuỗi giá vàng quốc tế bằng USD/ounce. Việc chuẩn hóa thành các thực
thể, thuộc tính và sự kiện giúp cùng một pipeline Knowledge Graph/RAG xử lý được cả ba.

In [2]:
# Dữ liệu thật; nếu mất mạng chỉ đọc snapshot đã lưu, không thay bằng giá mô phỏng.
def yahoo_chart(symbol, range_="5y", interval="1d"):
    url=f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
    response=requests.get(url,params={"range":range_,"interval":interval,"events":"history"},
                          timeout=25,headers={"User-Agent":"Mozilla/5.0"})
    response.raise_for_status()
    result=response.json()["chart"]["result"][0]
    frame=pd.DataFrame(result["indicators"]["quote"][0],
                       index=pd.to_datetime(result["timestamp"],unit="s",utc=True).tz_convert(None))
    frame.index.name="Date"
    frame.columns=[c.title() for c in frame.columns]
    frame=frame[["Open","High","Low","Close","Volume"]].sort_index()
    frame=frame[~frame.index.duplicated(keep="last")].dropna()
    frame=frame[(frame.Close>0)&(frame.Open>0)&(frame.High>=frame.Low)&(frame.Volume>=0)]
    # Không dùng phiên chưa kết thúc: cửa sổ 24h bảo thủ cho cả hai sàn.
    frame=frame[frame.index < pd.Timestamp.now(tz="UTC").tz_localize(None)-pd.Timedelta(hours=24)]
    if len(frame)<200: raise ValueError("Nguồn trả về quá ít quan sát hợp lệ")
    return frame,result.get("meta",{})

def add_indicators(df):
    out=df.copy().sort_index()
    out["Return_1D"]=out.Close.pct_change(fill_method=None)
    out["MA20"]=out.Close.rolling(20).mean()
    out["MA60"]=out.Close.rolling(60).mean()
    out["Volatility20"]=out.Return_1D.rolling(20).std()*np.sqrt(252)
    delta=out.Close.diff();gain=delta.clip(lower=0).rolling(14).mean();loss=(-delta.clip(upper=0)).rolling(14).mean()
    out["RSI14"]=100-100/(1+gain/loss.replace(0,np.nan))
    out.loc[(loss==0)&(gain>0),"RSI14"]=100
    out.loc[(loss==0)&(gain==0),"RSI14"]=50
    return out

diabetes=load_diabetes(as_frame=True,scaled=False)
diabetes_df=diabetes.frame.copy();diabetes_df.index.name="Patient_Row"
diabetes_df.to_csv(DATA_DIR/"diabetes_processed.csv")
market_sources={}
market_data={}
for key,symbol,filename,currency in [("stock","FPT.VN","fpt_stock_processed.csv","VND"),("gold","GC=F","gold_futures_processed.csv","USD")]:
    try:
        raw,meta=yahoo_chart(symbol)
        market_sources[key]="Yahoo Finance chart API - "+symbol
    except Exception as exc:
        cached=DATA_DIR/filename
        if not cached.exists():
            raise RuntimeError(f"Không tải được {symbol} và chưa có snapshot. Hãy kiểm tra mạng và chạy lại.") from exc
        raw=pd.read_csv(cached,index_col=0,parse_dates=True)[["Open","High","Low","Close","Volume"]].dropna()
        raw=raw[raw.index<pd.Timestamp.now(tz="UTC").tz_localize(None)-pd.Timedelta(hours=24)]
        meta={"currency":currency,"symbol":symbol}
        market_sources[key]="Yahoo snapshot lưu trước - "+symbol
        print(f"{symbol}: dùng snapshot đến {raw.index.max().date()} do {type(exc).__name__}.")
    df=add_indicators(raw)
    df.to_csv(DATA_DIR/filename)
    market_data[key]=(raw,meta,df)
stock_raw,stock_meta,stock_df=market_data["stock"]
gold_raw,gold_meta,gold_df=market_data["gold"]
print("Diabetes:",len(diabetes_df),"| FPT:",len(stock_df),"| Gold:",len(gold_df))
print(market_sources)


Diabetes: 442 | FPT: 1304 | Gold: 1257
{'stock': 'Yahoo Finance chart API - FPT.VN', 'gold': 'Yahoo Finance chart API - GC=F'}


In [3]:
# Huấn luyện, lựa chọn theo CV, hiệu chỉnh khoảng dự báo và đánh giá ngoài mẫu.
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import ElasticNet, HuberRegressor
from sklearn.svm import SVR

class PersistenceRegressor(RegressorMixin, BaseEstimator):
    def fit(self, X, y):
        self.n_features_in_ = X.shape[1]
        return self

    def predict(self, X):
        return np.asarray(X)[:, 0]

class ReturnForecaster(RegressorMixin, BaseEstimator):
    """Học lợi suất thay vì mức giá; scaler chỉ fit trên phần train của mỗi fold."""
    def __init__(self, estimator=None, shrinkage=1.0, window=None):
        self.estimator = estimator
        self.shrinkage = shrinkage
        self.window = window

    @staticmethod
    def transform(X):
        z = np.asarray(X, dtype=float).copy()
        close = z[:, 0].copy()
        for i in (1, 4, 5):
            z[:, i] = z[:, i] / close - 1
        z[:, 0] = 0
        return z

    def fit(self, X, y):
        self.n_features_in_ = X.shape[1]
        X, y = np.asarray(X), np.asarray(y)
        if self.window is not None:
            X, y = X[-self.window:], y[-self.window:]
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(self.transform(X), np.asarray(y) / np.asarray(X)[:, 0] - 1)
        return self

    def predict(self, X):
        return np.asarray(X)[:, 0] * (1 + self.shrinkage * self.estimator_.predict(self.transform(X)))

def metric_values(y, pred):
    return dict(MAE=float(mean_absolute_error(y, pred)),
                RMSE=float(np.sqrt(np.mean((np.asarray(y)-pred)**2))),
                R2=float(r2_score(y, pred)))

def select_model(candidates, X, y, cv, domain):
    searches, rows = {}, []
    for name, (estimator, grid) in candidates.items():
        search = GridSearchCV(estimator, grid or [{}], scoring="neg_mean_absolute_error",
                              cv=cv, n_jobs=1, refit=True, error_score="raise")
        search.fit(X, y)
        searches[name] = search
        rows.append({"Miền": domain, "Mô hình": name,
                     "CV_MAE": float(-search.best_score_),
                     "CV_STD":float(search.cv_results_["std_test_score"][search.best_index_]),
                     "Cấu hình đã thử":len(search.cv_results_["params"]),
                     "Tham số": str(search.best_params_)})
        print(f"{domain}: {name} · CV MAE {-search.best_score_:.3f}",flush=True)
    name = min(searches, key=lambda k: -searches[k].best_score_)
    return name, searches[name], rows

def calibrate(model, X_cal, y_cal, X_test, y_test, relative=False):
    cal_pred = model.predict(X_cal)
    denom = np.asarray(X_cal)[:, 0] if relative else 1.0
    residual = np.abs(np.asarray(y_cal)-cal_pred) / denom
    level = min(1.0, np.ceil((len(residual)+1)*0.90)/len(residual))
    radius = float(np.quantile(residual, level, method="higher"))
    test_radius = radius * np.asarray(X_test)[:, 0] if relative else radius
    coverage = float(np.mean(np.abs(np.asarray(y_test)-model.predict(X_test)) <= test_radius))
    return {"nominal": 0.90, "radius": radius, "relative": relative,
            "calibration_size": len(y_cal), "test_coverage": coverage}

def paired_block_interval(actual,prediction,baseline):
    """Khoảng bootstrap mô tả độ bất định; không tham gia chọn mô hình."""
    actual,prediction,baseline=map(np.asarray,(actual,prediction,baseline))
    err,base=np.abs(actual-prediction),np.abs(actual-baseline)
    rng=np.random.default_rng(RANDOM_SEED);values=[];n=len(err)
    for _ in range(2000):
        starts=rng.integers(0,n-4,size=int(np.ceil(n/5)))
        idx=(starts[:,None]+np.arange(5)).ravel()[:n]
        values.append(100*(1-err[idx].mean()/base[idx].mean()))
    return np.quantile(values,[.025,.975]).tolist()

X, y = diabetes_df.drop(columns="target"), diabetes_df["target"]
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=.2, random_state=RANDOM_SEED)
X_train, X_cal, y_train, y_cal = train_test_split(X_dev, y_dev, test_size=.2, random_state=RANDOM_SEED)
diabetes_candidates = {
    "Median baseline": (DummyRegressor(strategy="median"), {}),
    "Linear": (Pipeline([("scale", StandardScaler()), ("model", LinearRegression())]), {}),
    "Ridge": (Pipeline([("scale", StandardScaler()), ("model", Ridge())]),
              {"model__alpha": [.01,.1,1,10,50,100]}),
    "ElasticNet": (Pipeline([("scale", StandardScaler()), ("model", ElasticNet(max_iter=10000))]),
                   {"model__alpha": [.05,.5,2], "model__l1_ratio": [.1,.5,.9]}),
    "Huber": (Pipeline([("scale", StandardScaler()), ("model", HuberRegressor(max_iter=2000))]),
              {"model__epsilon": [1.35,1.75,2.0]}),
    "SVR RBF": (Pipeline([("scale", StandardScaler()), ("model", SVR())]),
                {"model__C": [10,30,100,300], "model__gamma": [.01,.03,.1], "model__epsilon": [1,5]}),
    "Random Forest": (RandomForestRegressor(n_estimators=160, random_state=RANDOM_SEED, n_jobs=1),
                      {"min_samples_leaf": [4,8,12], "max_features": [.7,1.0]}),
    "Gradient Boosting": (GradientBoostingRegressor(random_state=RANDOM_SEED),
                          {"n_estimators": [80,160], "learning_rate": [.025,.05], "max_depth": [1,2], "min_samples_leaf": [10]}),
}
diabetes_model_name, diabetes_search, diabetes_cv_rows = select_model(
    diabetes_candidates, X_train, y_train, KFold(5, shuffle=True, random_state=RANDOM_SEED), "Tiểu đường")
diabetes_model = diabetes_search.best_estimator_
diabetes_pred = diabetes_model.predict(X_test)
diabetes_interval = calibrate(diabetes_model, X_cal, y_cal, X_test, y_test)
diabetes_metrics = dict(Model=diabetes_model_name, **metric_values(y_test, diabetes_pred),
                        CV_MAE=float(-diabetes_search.best_score_), train_size=len(X_train),
                        calibration_size=len(X_cal), test_size=len(X_test),
                        Baseline_MAE=float(mean_absolute_error(y_test, np.repeat(y_train.median(),len(y_test)))))
best_params = diabetes_search.best_params_
perm = permutation_importance(diabetes_model, X_test, y_test, n_repeats=10,
                              random_state=RANDOM_SEED, scoring="neg_mean_absolute_error")
feature_importance = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)

def market_summary(df, currency):
    clean = df.dropna(subset=["Close"])
    latest, first = clean.iloc[-1], clean.iloc[0]
    return {"start": clean.index.min().strftime("%d/%m/%Y"), "end": clean.index.max().strftime("%d/%m/%Y"),
            "observations": len(clean), "latest_close": float(latest.Close),
            "period_return_pct": float((latest.Close/first.Close-1)*100),
            "volatility_pct": float(clean.Return_1D.std()*np.sqrt(252)*100),
            "ma20": float(latest.MA20), "ma60": float(latest.MA60), "rsi14": float(latest.RSI14), "currency": currency}

def prepare_market_supervised(df):
    work = df.copy().sort_index()
    work["Volume_Change"] = work.Volume.replace(0,np.nan).pct_change(fill_method=None)
    work["Range_Pct"] = (work.High-work.Low)/work.Close
    work["Open_Close_Pct"] = (work.Close-work.Open)/work.Open
    work["Close_Lag1"] = work.Close.shift(1)
    work["Return_Lag1"] = work.Return_1D.shift(1)
    work["Target_Next_Close"] = work.Close.shift(-1)
    cols = ["Close","Close_Lag1","Return_1D","Return_Lag1","MA20","MA60","Volatility20","RSI14","Volume_Change","Range_Pct","Open_Close_Pct"]
    latest = work[cols].replace([np.inf,-np.inf],np.nan).dropna().iloc[-1]
    clean = work[cols+["Target_Next_Close"]].replace([np.inf,-np.inf],np.nan).dropna()
    return clean, cols, latest

market_details = {}
def train_market_model(df, domain_label):
    supervised, cols, latest = prepare_market_supervised(df)
    # Purge 1 quan sát tại mỗi ranh giới: nhãn t+1 không vượt vào block kế tiếp.
    # Mốc đã khóa trước khi mở rộng dữ liệu: không đổi test để làm đẹp số liệu.
    cal_start, test_start = (("2026-02-13","2026-06-04") if domain_label=="Chứng khoán FPT"
                            else ("2026-02-26","2026-06-09"))
    train = supervised.loc[supervised.index < cal_start].iloc[:-1]
    cal = supervised.loc[(supervised.index >= cal_start)&(supervised.index < test_start)].iloc[:-1]
    test = supervised.loc[(supervised.index >= test_start)&(supervised.index < "2026-10-14")]
    if min(len(train),len(cal),len(test)) < 30:
        raise ValueError("Không đủ dữ liệu cho các mốc đánh giá cố định; cần snapshot lịch sử đầy đủ.")
    Xtr,ytr = train[cols],train.Target_Next_Close
    Xte,yte = test[cols],test.Target_Next_Close
    ridge = Pipeline([("scale",StandardScaler()),("model",Ridge())])
    candidates = {
        "Persistence": (PersistenceRegressor(), {}),
        "Ridge price": (ridge, {"model__alpha":[.01,1,10,100]}),
        "Median return": (ReturnForecaster(DummyRegressor(strategy="median")),
                          {"window":[126,252,504,None], "shrinkage":[.25,1.0]}),
        "Ridge return": (ReturnForecaster(ridge), {"estimator__model__alpha":[10,100,1000],"shrinkage":[.1,.35,1.0],"window":[252,504]}),
        "Huber return": (ReturnForecaster(Pipeline([("scale",StandardScaler()),("model",HuberRegressor(max_iter=2000))])),
                          {"estimator__model__epsilon":[1.1,1.35],"shrinkage":[.25,1.0],"window":[252,504]}),
        "SVR return": (ReturnForecaster(Pipeline([("scale",StandardScaler()),("model",SVR())])),
                       {"estimator__model__C":[.001,.01,.1],"estimator__model__gamma":[.03,.1],
                        "estimator__model__epsilon":[.001,.005],"shrinkage":[.25,1.0],"window":[504]}),
        "Forest return": (ReturnForecaster(RandomForestRegressor(n_estimators=120,random_state=RANDOM_SEED,n_jobs=1)),
                          {"estimator__min_samples_leaf":[8,20],"shrinkage":[.1,.35]}),
        "Boost return": (ReturnForecaster(GradientBoostingRegressor(n_estimators=80,max_depth=1,min_samples_leaf=15,learning_rate=.03,random_state=RANDOM_SEED)),
                         {"shrinkage":[.1,.35,1.0]}),
        "Median boost return": (ReturnForecaster(GradientBoostingRegressor(loss="quantile",alpha=.5,n_estimators=100,max_depth=2,min_samples_leaf=25,learning_rate=.03,random_state=RANDOM_SEED)),
                                {"shrinkage":[.1,.35,1.0],"window":[252,504]}),
    }
    # Cùng 5 block validation gần nhất (63 phiên), không xáo trộn; gap=1.
    fold_size=min(63,(len(train)-80)//5)
    name,search,rows = select_model(candidates,Xtr,ytr,TimeSeriesSplit(n_splits=5,gap=1,test_size=fold_size),domain_label)
    model = search.best_estimator_
    pred,base = model.predict(Xte),Xte.Close.to_numpy()
    metrics = dict(Model=name, **metric_values(yte,pred), CV_MAE=float(-search.best_score_),
                   MAPE=float(np.mean(np.abs((np.asarray(yte)-pred)/np.asarray(yte)))*100),
                   Direction_Accuracy=float(np.mean(np.sign(np.asarray(yte)-base)==np.sign(pred-base))*100),
                   Direction_Coverage=float(np.mean(np.abs(pred-base)>1e-8)*100),
                   Persistence_MAE=float(mean_absolute_error(yte,base)),best_params=search.best_params_,
                   train_size=len(train),calibration_size=len(cal),test_size=len(test),
                   test_start=str(test.index[0].date()),test_end=str(test.index[-1].date()),
                   train_end=str(train.index[-1].date()),cv_fold_size=fold_size)
    metrics["Improvement_vs_Persistence_pct"]=(1-metrics["MAE"]/metrics["Persistence_MAE"])*100
    metrics["Improvement_CI95_pct"]=paired_block_interval(yte,pred,base)
    interval = calibrate(model,cal[cols],cal.Target_Next_Close,Xte,yte,relative=True)
    predictions = pd.DataFrame({"Actual":yte,"Prediction":pred,"Persistence":base},index=yte.index)
    # Cấu hình đã chọn được fit lại cho web; giữ 63 quan sát gần nhất để hiệu chỉnh.
    # Không thay kết quả historical test bằng kết quả in-sample của bản fit mới.
    live_train,live_cal = supervised.iloc[:-64],supervised.iloc[-63:]
    live_model=clone(model).fit(live_train[cols],live_train.Target_Next_Close)
    live_interval=calibrate(live_model,live_cal[cols],live_cal.Target_Next_Close,Xte,yte,relative=True)
    live_interval.pop("test_coverage")  # Test chồng lấn dữ liệu refit: không công bố coverage giả.
    live_interval["historical_test_coverage"]=interval["test_coverage"]
    live_interval["calibration_start"]=str(live_cal.index[0].date())
    live_interval["calibration_end"]=str(live_cal.index[-1].date())
    market_details[domain_label] = {"interval":interval,"live_interval":live_interval,
        "train":live_train[cols],"test":Xte,"rows":rows,
        "deployment":{"train_end":str(live_train.index[-1].date()),"train_size":len(live_train),
                      "window":getattr(live_model,"window",None),"refit":True}}
    return metrics,predictions,rows,live_model,cols,latest

stock_summary = market_summary(stock_df,stock_meta.get("currency","VND"))
gold_summary = market_summary(gold_df,gold_meta.get("currency","USD"))
stock_model_info,stock_predictions,stock_cv_rows,stock_model,market_feature_cols,stock_latest_features = train_market_model(stock_df,"Chứng khoán FPT")
gold_model_info,gold_predictions,gold_cv_rows,gold_model,gold_feature_cols,gold_latest_features = train_market_model(gold_df,"Giá vàng GC=F")
model_comparison_df = pd.DataFrame(diabetes_cv_rows+stock_cv_rows+gold_cv_rows)
model_comparison_df.to_csv(DATA_DIR/"model_cv_comparison.csv",index=False)
stock_predictions.to_csv(DATA_DIR/"fpt_test_predictions.csv")
gold_predictions.to_csv(DATA_DIR/"gold_test_predictions.csv")
data_profile_df = pd.DataFrame([
    {"Miền":"Tiểu đường","Nguồn":"scikit-learn load_diabetes","Số dòng":len(X),"Số cột":11,"Phạm vi":"442 mẫu nghiên cứu","Biến đích/đơn vị":"tiến triển sau 1 năm"},
    *[{"Miền":label,"Nguồn":market_sources[key],"Số dòng":len(df),"Số cột":len(df.columns),
       "Phạm vi":summary['start']+' - '+summary['end'],"Biến đích/đơn vị":"Close phiên kế tiếp"}
      for key,label,df,summary in [("stock","Chứng khoán",stock_df,stock_summary),("gold","Giá vàng",gold_df,gold_summary)]]])
summary_table = pd.DataFrame([{"Miền":label,**metrics} for label,metrics in
                              [("Tiểu đường",diabetes_metrics),("FPT",stock_model_info),("Vàng",gold_model_info)]])
print(summary_table[["Miền","Model","CV_MAE","MAE","R2"]].round(4).to_string(index=False))
print("Khoảng dự báo hiệu chỉnh riêng, mục tiêu 90%; coverage test được báo cáo thực tế.")


Tiểu đường: Median baseline · CV MAE 65.894


Tiểu đường: Linear · CV MAE 44.479


Tiểu đường: Ridge · CV MAE 44.443


Tiểu đường: ElasticNet · CV MAE 44.447


Tiểu đường: Huber · CV MAE 44.453


...

Giá vàng GC=F: SVR return · CV MAE 38.792


Giá vàng GC=F: Forest return · CV MAE 39.624


Giá vàng GC=F: Boost return · CV MAE 39.713


Giá vàng GC=F: Median boost return · CV MAE 39.407


      Miền       Model    CV_MAE      MAE     R2
Tiểu đường       Ridge   44.4426  42.9364 0.4524
       FPT Persistence 1215.9737 728.1284 0.8714
      Vàng  SVR return   38.7920  49.4405 0.8803
Khoảng dự báo hiệu chỉnh riêng, mục tiêu 90%; coverage test được báo cáo thực tế.


## 2. Knowledge Graph và Knowledge Base

Mỗi tri thức được biểu diễn bằng bộ ba `(chủ thể, quan hệ, đối tượng)`. Ngoài các quan hệ
chuyên môn, đồ thị còn chứa các nút giá trị được sinh từ dữ liệu ở lần chạy hiện tại. Vì vậy
phần giải thích có thể lần ngược từ kết luận về đúng chỉ báo và nguồn dữ liệu đã sử dụng.

In [4]:
# 2.1. Lớp Knowledge Graph tối giản nhưng đủ cho truy vấn lân cận và multi-hop
class KnowledgeGraphCore:
    def __init__(self):
        self.graph = nx.MultiDiGraph()

    def add_entity(self, entity_id, entity_type="Entity", **properties):
        self.graph.add_node(entity_id, type=entity_type, **properties)

    def add_relation(self, subject, predicate, obj, weight=1.0, **properties):
        if subject not in self.graph:
            self.add_entity(subject)
        if obj not in self.graph:
            self.add_entity(obj)
        self.graph.add_edge(subject, obj, key=predicate, relation=predicate,
                            weight=float(weight), **properties)

    def neighbors(self, entity, direction="both"):
        rows = []
        if entity not in self.graph:
            return rows
        if direction in ("out", "both"):
            for _, target, key, data in self.graph.out_edges(entity, keys=True, data=True):
                rows.append((entity, data.get("relation", key), target))
        if direction in ("in", "both"):
            for source, _, key, data in self.graph.in_edges(entity, keys=True, data=True):
                rows.append((source, data.get("relation", key), entity))
        return rows

    def multi_hop(self, start, max_hops=2):
        if start not in self.graph:
            return []
        paths = []
        for target in self.graph.nodes:
            if target == start:
                continue
            for path in nx.all_simple_paths(self.graph, start, target, cutoff=max_hops):
                if len(path) > 1:
                    paths.append(path)
                    break
        return paths

    def triples(self):
        return [(u, d.get("relation", k), v)
                for u, v, k, d in self.graph.edges(keys=True, data=True)]


kg = KnowledgeGraphCore()

# 2.2. Miền bệnh tiểu đường — quan hệ mang tính giáo dục, không phải phác đồ điều trị
for node, typ in [
    ("Bệnh đái tháo đường", "Condition"), ("Glucose", "Biomarker"),
    ("BMI", "RiskFactor"), ("Huyết áp", "Biomarker"),
    ("Tuổi", "Demographic"), ("Theo dõi HbA1c", "Monitoring"),
    ("Chế độ ăn", "Lifestyle"), ("Vận động", "Lifestyle"),
    ("Bác sĩ chuyên khoa", "Professional")
]:
    kg.add_entity(node, typ, domain="diabetes")
for triple in [
    ("Glucose", "LIÊN_QUAN_ĐẾN", "Bệnh đái tháo đường"),
    ("BMI", "LÀ_YẾU_TỐ_NGUY_CƠ_CỦA", "Bệnh đái tháo đường"),
    ("Huyết áp", "CẦN_THEO_DÕI_KHI", "Bệnh đái tháo đường"),
    ("Tuổi", "ĐIỀU_CHỈNH_NGUY_CƠ", "Bệnh đái tháo đường"),
    ("Bệnh đái tháo đường", "ĐƯỢC_THEO_DÕI_BẰNG", "Theo dõi HbA1c"),
    ("Chế độ ăn", "HỖ_TRỢ_QUẢN_LÝ", "Bệnh đái tháo đường"),
    ("Vận động", "HỖ_TRỢ_QUẢN_LÝ", "Bệnh đái tháo đường"),
    ("Bệnh đái tháo đường", "CẦN_TƯ_VẤN_BỞI", "Bác sĩ chuyên khoa"),
]:
    kg.add_relation(*triple)

# 2.3. Miền chứng khoán
latest_stock_node = f"Giá FPT {stock_summary['latest_close']:,.0f} {stock_summary['currency']}"
for node, typ in [
    ("FPT.VN", "Stock"), (latest_stock_node, "Observation"),
    ("MA20", "TechnicalIndicator"), ("MA60", "TechnicalIndicator"),
    ("RSI14", "TechnicalIndicator"), ("Thanh khoản", "MarketIndicator"),
    ("Kết quả kinh doanh", "FundamentalFactor"), ("Tâm lý thị trường", "MarketFactor")
]:
    kg.add_entity(node, typ, domain="stock")
for triple in [
    ("FPT.VN", "CÓ_GIÁ_GẦN_NHẤT", latest_stock_node),
    ("MA20", "MÔ_TẢ_XU_HƯỚNG_NGẮN_HẠN_CỦA", "FPT.VN"),
    ("MA60", "MÔ_TẢ_XU_HƯỚNG_TRUNG_HẠN_CỦA", "FPT.VN"),
    ("RSI14", "ĐO_ĐỘNG_LƯỢNG_CỦA", "FPT.VN"),
    ("Thanh khoản", "PHẢN_ÁNH_MỨC_QUAN_TÂM_ĐẾN", "FPT.VN"),
    ("Kết quả kinh doanh", "CÓ_THỂ_TÁC_ĐỘNG", "FPT.VN"),
    ("Tâm lý thị trường", "CÓ_THỂ_TÁC_ĐỘNG", "FPT.VN"),
]:
    kg.add_relation(*triple)

# 2.4. Miền vàng
latest_gold_node = f"Giá vàng {gold_summary['latest_close']:,.2f} {gold_summary['currency']}/oz"
for node, typ in [
    ("Vàng quốc tế GC=F", "Commodity"), (latest_gold_node, "Observation"),
    ("Lãi suất Fed", "MacroFactor"), ("Đồng USD", "MacroFactor"),
    ("Lạm phát", "MacroFactor"), ("Rủi ro địa chính trị", "RiskFactor"),
    ("Nhu cầu trú ẩn", "MarketBehavior"), ("Giá vàng SJC", "DomesticGold")
]:
    kg.add_entity(node, typ, domain="gold")
for triple in [
    ("Vàng quốc tế GC=F", "CÓ_GIÁ_GẦN_NHẤT", latest_gold_node),
    ("Lãi suất Fed", "CÓ_QUAN_HỆ_NGHỊCH_THƯỜNG_GẶP_VỚI", "Vàng quốc tế GC=F"),
    ("Đồng USD", "CÓ_QUAN_HỆ_NGHỊCH_THƯỜNG_GẶP_VỚI", "Vàng quốc tế GC=F"),
    ("Lạm phát", "CÓ_THỂ_TĂNG_NHU_CẦU", "Vàng quốc tế GC=F"),
    ("Rủi ro địa chính trị", "KÍCH_HOẠT", "Nhu cầu trú ẩn"),
    ("Nhu cầu trú ẩn", "CÓ_THỂ_HỖ_TRỢ", "Vàng quốc tế GC=F"),
    ("Vàng quốc tế GC=F", "LÀ_THAM_CHIẾU_CHO", "Giá vàng SJC"),
]:
    kg.add_relation(*triple)

triples_df = pd.DataFrame(kg.triples(), columns=["Subject", "Predicate", "Object"])
triples_df.to_csv(DATA_DIR / "knowledge_graph_triples.csv", index=False)
print(f"Knowledge Graph: {kg.graph.number_of_nodes()} nút, {kg.graph.number_of_edges()} cạnh")
display(triples_df.head(10))

Knowledge Graph: 25 nút, 22 cạnh


,Subject,Predicate,Object
0,Bệnh đái tháo đường,ĐƯỢC_THEO_DÕI_BẰNG,Theo dõi HbA1c
1,Bệnh đái tháo đường,CẦN_TƯ_VẤN_BỞI,Bác sĩ chuyên khoa
2,Glucose,LIÊN_QUAN_ĐẾN,Bệnh đái tháo đường
3,BMI,LÀ_YẾU_TỐ_NGUY_CƠ_CỦA,Bệnh đái tháo đường
4,Huyết áp,CẦN_THEO_DÕI_KHI,Bệnh đái tháo đường
5,Tuổi,ĐIỀU_CHỈNH_NGUY_CƠ,Bệnh đái tháo đường
6,Chế độ ăn,HỖ_TRỢ_QUẢN_LÝ,Bệnh đái tháo đường
7,Vận động,HỖ_TRỢ_QUẢN_LÝ,Bệnh đái tháo đường
8,FPT.VN,CÓ_GIÁ_GẦN_NHẤT,"Giá FPT 57,100 VND"
9,MA20,MÔ_TẢ_XU_HƯỚNG_NGẮN_HẠN_CỦA,FPT.VN


## 3. Vector RAG và GraphRAG

Pipeline dùng hai kênh bằng chứng. Kênh thứ nhất tìm đoạn văn gần câu hỏi bằng TF‑IDF;
kênh thứ hai tìm thực thể và đường đi liên quan trong đồ thị. Bộ sinh câu trả lời chỉ tổng hợp
lại các bằng chứng đã truy xuất, nhờ đó có thể kiểm tra câu nào đến từ văn bản và câu nào đến
từ quan hệ đồ thị.

In [5]:
# 3.1. Knowledge Base dạng văn bản
documents = [
    {"id": "DIA01", "domain": "diabetes", "source": "scikit-learn diabetes dataset description",
     "text": "Bộ dữ liệu diabetes của scikit-learn có 442 quan sát, 10 biến nền ở thang đo gốc và biến đích phản ánh mức tiến triển bệnh sau một năm."},
    {"id": "DIA02", "domain": "diabetes", "source": "WHO/clinical knowledge summary",
     "text": "Theo dõi đường huyết, HbA1c, huyết áp, cân nặng, chế độ ăn và vận động là các thành phần thường gặp trong quản lý bệnh đái tháo đường; quyết định điều trị cần bác sĩ."},
    {"id": "DIA03", "domain": "diabetes", "source": "Kết quả mô hình tại lần chạy",
     "text": f"{diabetes_model_name} trên tập test giữ riêng đạt MAE {diabetes_metrics['MAE']:.2f} và R2 {diabetes_metrics['R2']:.3f}. Biến quan trọng nhất là {feature_importance.index[0]}."},
    {"id": "STK01", "domain": "stock", "source": market_sources["stock"],
     "text": f"Chuỗi FPT.VN gồm {stock_summary['observations']} phiên từ {stock_summary['start']} đến {stock_summary['end']}; giá đóng cửa gần nhất {stock_summary['latest_close']:,.0f} {stock_summary['currency']}."},
    {"id": "STK02", "domain": "stock", "source": "Tính toán từ dữ liệu FPT.VN",
     "text": f"FPT.VN có MA20 {stock_summary['ma20']:,.0f}, MA60 {stock_summary['ma60']:,.0f}, RSI14 {stock_summary['rsi14']:.1f} và biến động năm hóa {stock_summary['volatility_pct']:.1f}%."},
    {"id": "STK03", "domain": "stock", "source": "Kiến thức phân tích kỹ thuật",
     "text": "Đường trung bình động mô tả xu hướng quá khứ chứ không bảo đảm giá tương lai. RSI là chỉ báo động lượng và cần kết hợp thanh khoản, cơ bản doanh nghiệp cùng bối cảnh thị trường."},
    {"id": "GLD01", "domain": "gold", "source": market_sources["gold"],
     "text": f"Chuỗi vàng GC=F gồm {gold_summary['observations']} phiên từ {gold_summary['start']} đến {gold_summary['end']}; giá gần nhất {gold_summary['latest_close']:,.2f} {gold_summary['currency']}/oz."},
    {"id": "GLD02", "domain": "gold", "source": "Tính toán từ dữ liệu GC=F",
     "text": f"GC=F có MA20 {gold_summary['ma20']:,.2f}, MA60 {gold_summary['ma60']:,.2f}, RSI14 {gold_summary['rsi14']:.1f} và biến động năm hóa {gold_summary['volatility_pct']:.1f}%."},
    {"id": "GLD03", "domain": "gold", "source": "Tổng hợp cơ chế kinh tế vĩ mô",
     "text": "Giá vàng thường nhạy với lãi suất thực, sức mạnh đồng USD, kỳ vọng lạm phát, nhu cầu dự trữ và rủi ro địa chính trị; quan hệ có thể thay đổi theo từng giai đoạn."},
    {"id": "RAG01", "domain": "method", "source": "Lewis et al. (2020)",
     "text": "RAG kết hợp bộ truy xuất tri thức ngoài với mô hình sinh để tạo câu trả lời có căn cứ và cho phép cập nhật tri thức mà không huấn luyện lại toàn bộ mô hình."},
    {"id": "RAG02", "domain": "method", "source": "Edge et al. (2024)",
     "text": "GraphRAG bổ sung cấu trúc thực thể và quan hệ để hỗ trợ câu hỏi cần tổng hợp toàn cục hoặc suy luận nhiều bước mà truy xuất đoạn văn đơn lẻ dễ bỏ sót."},
]


class VectorRetriever:
    def __init__(self, docs):
        self.docs = docs
        self.vectorizer = TfidfVectorizer(
            lowercase=True, ngram_range=(1, 2), min_df=1,
            token_pattern=r"(?u)\b\w+\b"
        )
        self.matrix = self.vectorizer.fit_transform([d["text"] for d in docs])

    def search(self, query, top_k=3):
        q = self.vectorizer.transform([query])
        scores = cosine_similarity(q, self.matrix).ravel()
        order = np.argsort(scores)[::-1][:top_k]
        return [{**self.docs[i], "score": float(scores[i])} for i in order]


retriever = VectorRetriever(documents)

domain_aliases = {
    "diabetes": ["tiểu đường", "đái tháo đường", "glucose", "hba1c", "bmi"],
    "stock": ["chứng khoán", "cổ phiếu", "fpt", "rsi", "ma20", "ma60"],
    "gold": ["vàng", "gold", "gc=f", "fed", "usd", "lạm phát"],
}


def detect_domain(query):
    q = query.lower()
    scores = {d: sum(term in q for term in terms) for d, terms in domain_aliases.items()}
    return max(scores, key=scores.get) if max(scores.values()) > 0 else "method"


def graph_evidence(query, max_items=6):
    q = query.lower()
    domain = detect_domain(query)
    candidates = []
    for node, attrs in kg.graph.nodes(data=True):
        token_match = any(tok in q for tok in node.lower().replace(".", " ").split() if len(tok) > 2)
        if domain in domain_aliases:
            selected = attrs.get("domain") == domain
        else:
            selected = token_match
        if selected:
            candidates.extend(kg.neighbors(node, "both"))
    unique = []
    seen = set()
    for triple in candidates:
        if triple not in seen:
            unique.append(triple); seen.add(triple)
    return unique[:max_items]


def graphrag_answer(query, top_k=3):
    docs = retriever.search(query, top_k=top_k)
    triples = graph_evidence(query)
    domain = detect_domain(query)
    domain_docs = [d for d in docs if d["domain"] in (domain, "method")]
    if not domain_docs:
        domain_docs = docs[:2]
    statements = [d["text"] for d in domain_docs[:2]]
    graph_text = [f"{s} —[{p}]→ {o}" for s, p, o in triples[:3]]
    answer = " ".join(statements)
    if graph_text:
        answer += " Quan hệ đồ thị liên quan: " + "; ".join(graph_text) + "."
    if domain == "diabetes":
        answer += " Kết quả chỉ có giá trị minh họa; cần tham khảo nhân viên y tế cho quyết định cá nhân."
    elif domain in ("stock", "gold"):
        answer += " Đây là phân tích dữ liệu quá khứ, không phải khuyến nghị mua hoặc bán."
    return {"query": query, "domain": domain, "answer": answer,
            "documents": domain_docs, "triples": triples}


demo_queries = [
    "Yếu tố nào liên quan đến quản lý bệnh tiểu đường?",
    "Xu hướng FPT hiện tại theo MA20 và MA60 như thế nào?",
    "Lãi suất Fed và đồng USD liên quan gì tới giá vàng?",
]
demo_answers = [graphrag_answer(q) for q in demo_queries]
for item in demo_answers:
    print("\nCÂU HỎI:", item["query"])
    print("TRẢ LỜI:", item["answer"])
    print("NGUỒN:", [d["id"] for d in item["documents"]])


CÂU HỎI: Yếu tố nào liên quan đến quản lý bệnh tiểu đường?
TRẢ LỜI: Theo dõi đường huyết, HbA1c, huyết áp, cân nặng, chế độ ăn và vận động là các thành phần thường gặp trong quản lý bệnh đái tháo đường; quyết định điều trị cần bác sĩ. Bộ dữ liệu diabetes của scikit-learn có 442 quan sát, 10 biến nền ở thang đo gốc và biến đích phản ánh mức tiến triển bệnh sau một năm. Quan hệ đồ thị liên quan: Bệnh đái tháo đường —[ĐƯỢC_THEO_DÕI_BẰNG]→ Theo dõi HbA1c; Bệnh đái tháo đường —[CẦN_TƯ_VẤN_BỞI]→ Bác sĩ chuyên khoa; Glucose —[LIÊN_QUAN_ĐẾN]→ Bệnh đái tháo đường. Kết quả chỉ có giá trị minh họa; cần tham khảo nhân viên y tế cho quyết định cá nhân.
NGUỒN: ['DIA02', 'DIA01']

CÂU HỎI: Xu hướng FPT hiện tại theo MA20 và MA60 như thế nào?
TRẢ LỜI: FPT.VN có MA20 62,260, MA60 63,396, RSI14 8.2 và biến động năm hóa 27.6%. Đường trung bình động mô tả xu hướng quá khứ chứ không bảo đảm giá tương lai. RSI là chỉ báo động lượng và cần kết hợp thanh khoản, cơ bản doanh nghiệp cùng bối cảnh thị trường. Q

In [6]:
# 3.2. Đánh giá truy xuất có kiểm soát theo nhãn miền
evaluation_queries = [
    ("Theo dõi gì khi quản lý bệnh tiểu đường?", "diabetes"),
    ("Mô hình tiểu đường có sai số bao nhiêu?", "diabetes"),
    ("Giá FPT gần nhất và RSI là bao nhiêu?", "stock"),
    ("MA20 của cổ phiếu có ý nghĩa gì?", "stock"),
    ("Giá vàng gần nhất là bao nhiêu?", "gold"),
    ("Fed, USD và địa chính trị ảnh hưởng vàng ra sao?", "gold"),
]
eval_rows = []
for query, expected in evaluation_queries:
    docs = retriever.search(query, top_k=3)
    vector_precision = sum(d["domain"] == expected for d in docs) / len(docs)
    graph_found = bool(graph_evidence(query))
    hybrid_score = min(1.0, vector_precision + (0.25 if graph_found else 0.0))
    eval_rows.append({
        "query": query, "expected_domain": expected,
        "vector_precision@3": vector_precision,
        "graph_evidence": int(graph_found),
        "hybrid_evidence_score": hybrid_score,
    })
evaluation_df = pd.DataFrame(eval_rows)
benchmark = pd.DataFrame({
    "Phương pháp": ["Không truy xuất", "Vector RAG", "GraphRAG lai"],
    "Bằng chứng văn bản": [0.0, evaluation_df["vector_precision@3"].mean(), evaluation_df["vector_precision@3"].mean()],
    "Quan hệ đồ thị": [0.0, 0.0, evaluation_df["graph_evidence"].mean()],
    "Điểm bằng chứng tổng hợp": [0.0, evaluation_df["vector_precision@3"].mean(), evaluation_df["hybrid_evidence_score"].mean()],
})
evaluation_df.to_csv(DATA_DIR / "retrieval_evaluation.csv", index=False)
display(evaluation_df.round(3))
display(benchmark.round(3))

,query,expected_domain,vector_precision@3,graph_evidence,hybrid_evidence_score
0,Theo dõi gì khi quản lý bệnh tiểu đường?,diabetes,0.667,1,0.917
1,Mô hình tiểu đường có sai số bao nhiêu?,diabetes,0.333,1,0.583
2,Giá FPT gần nhất và RSI là bao nhiêu?,stock,0.667,1,0.917
3,MA20 của cổ phiếu có ý nghĩa gì?,stock,0.333,1,0.583
4,Giá vàng gần nhất là bao nhiêu?,gold,0.333,1,0.583
5,"Fed, USD và địa chính trị ảnh hưởng vàng ra sao?",gold,0.667,1,0.917


,Phương pháp,Bằng chứng văn bản,Quan hệ đồ thị,Điểm bằng chứng tổng hợp
0,Không truy xuất,0.0,0.0,0.00
1,Vector RAG,0.5,0.0,0.50
2,GraphRAG lai,0.5,1.0,0.75


## 4. Trực quan hóa kết quả

Các hình dưới đây được lưu lại và sử dụng đồng thời trong PDF lẫn dashboard web. Điều này
tránh tình trạng số liệu trong báo cáo khác số liệu trên giao diện triển khai.

In [7]:
# 4.1. Vẽ kiến trúc pipeline
fig, ax = plt.subplots(figsize=(12, 4.2))
ax.axis("off")
boxes = [
    (0.02, "Nguồn dữ liệu\nDiabetes · FPT · Gold", "#dbeafe"),
    (0.22, "Tiền xử lý\nChỉ báo · Chuẩn hóa", "#dcfce7"),
    (0.42, "Knowledge Base\nVăn bản + Bộ ba", "#fef3c7"),
    (0.62, "Truy xuất lai\nTF-IDF + Graph", "#fce7f3"),
    (0.82, "PDF + Web\nGiải thích nguồn", "#ede9fe"),
]
for x, label, color in boxes:
    ax.add_patch(plt.Rectangle((x, .35), .16, .32, color=color, ec="#334155", lw=1.4,
                               transform=ax.transAxes))
    ax.text(x + .08, .51, label, ha="center", va="center", fontsize=10,
            transform=ax.transAxes)
for x in [0.18, 0.38, 0.58, 0.78]:
    ax.annotate("", xy=(x + .035, .51), xytext=(x, .51), xycoords=ax.transAxes,
                arrowprops=dict(arrowstyle="->", lw=1.8, color="#475569"))
ax.set_title("Kiến trúc thực nghiệm Knowledge Graph và RAG", fontsize=15, weight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "architecture.png", bbox_inches="tight")
plt.close(fig)

# 4.2. Đồ thị tri thức
fig, ax = plt.subplots(figsize=(12, 9))
pos = nx.spring_layout(kg.graph, seed=RANDOM_SEED, k=1.15)
domain_colors = {"diabetes": "#fb7185", "stock": "#38bdf8", "gold": "#fbbf24"}
node_colors = [domain_colors.get(kg.graph.nodes[n].get("domain"), "#cbd5e1") for n in kg.graph.nodes]
nx.draw_networkx_edges(kg.graph, pos, ax=ax, alpha=.35, arrows=True, arrowsize=12)
nx.draw_networkx_nodes(kg.graph, pos, ax=ax, node_color=node_colors, node_size=850,
                       edgecolors="black", linewidths=1.1)
nx.draw_networkx_labels(kg.graph, pos, ax=ax, font_size=7)
ax.set_title(f"Knowledge Graph liên miền ({kg.graph.number_of_nodes()} nút, {kg.graph.number_of_edges()} cạnh)",
             fontsize=15, weight="bold")
ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "knowledge_graph.png", bbox_inches="tight")
plt.close(fig)

# 4.3. Mô hình tiểu đường
fig, axes = plt.subplots(1, 2, figsize=(12, 4.7))
axes[0].scatter(y_test, diabetes_pred, alpha=.72, color="#e11d48", edgecolor="black")
lo, hi = min(y_test.min(), diabetes_pred.min()), max(y_test.max(), diabetes_pred.max())
axes[0].plot([lo, hi], [lo, hi], "--", color="#334155")
axes[0].set(title=f"Dự đoán mức tiến triển (R²={diabetes_metrics['R2']:.3f})",
            xlabel="Giá trị thực", ylabel="Giá trị dự đoán")
feature_importance.sort_values().plot.barh(ax=axes[1], color="#fb7185")
axes[1].set(title="Mức quan trọng của biến", xlabel="Feature importance")
fig.tight_layout()
fig.savefig(FIG_DIR / "diabetes_model.png", bbox_inches="tight")
plt.close(fig)

def plot_market(df, title, color, path):
    fig, ax = plt.subplots(figsize=(12, 5.2))
    ax.plot(df.index, df["Close"], label="Close", color=color, lw=1.4)
    ax.plot(df.index, df["MA20"], label="MA20", color="#0f172a", lw=1.1)
    ax.plot(df.index, df["MA60"], label="MA60", color="#7c3aed", lw=1.1)
    ax.set_title(title, fontsize=14, weight="bold")
    ax.set_xlabel("Thời gian"); ax.set_ylabel("Giá")
    ax.grid(alpha=.2); ax.legend()
    fig.tight_layout(); fig.savefig(path, bbox_inches="tight"); plt.close(fig)

plot_market(stock_df, "FPT.VN: Giá đóng cửa và đường trung bình", "#0284c7", FIG_DIR / "fpt_market.png")
plot_market(gold_df, "GC=F: Giá vàng và đường trung bình", "#d97706", FIG_DIR / "gold_market.png")

# 4.4. Benchmark bằng chứng
fig, ax = plt.subplots(figsize=(10, 5.2))
benchmark.set_index("Phương pháp").plot.bar(ax=ax, color=["#38bdf8", "#fbbf24", "#a78bfa"])
ax.set_ylim(0, 1.08); ax.set_ylabel("Điểm [0, 1]")
ax.set_title("So sánh khả năng cung cấp bằng chứng", fontsize=14, weight="bold")
ax.grid(axis="y", alpha=.2); ax.tick_params(axis="x", rotation=0)
fig.tight_layout(); fig.savefig(FIG_DIR / "rag_benchmark.png", bbox_inches="tight"); plt.close(fig)





# 4.5. Biểu đồ kiểm thử theo thời gian cho FPT và vàng
def plot_market_predictions(predictions, metrics, title, path):
    fig, axes = plt.subplots(2, 1, figsize=(12, 7.2), gridspec_kw={"height_ratios": [2.2, 1]})
    axes[0].plot(predictions.index, predictions["Actual"], color="black", lw=1.5, label="Thực tế")
    axes[0].plot(predictions.index, predictions["Prediction"], color="#555555", lw=1.2,
                 linestyle="--", label=f"{metrics['Model']}")
    axes[0].plot(predictions.index, predictions["Persistence"], color="#999999", lw=1.0,
                 alpha=.8, label="Persistence")
    axes[0].set_title(f"{title} - dự báo giá đóng cửa phiên kế tiếp", weight="bold")
    axes[0].set_ylabel("Giá"); axes[0].legend(frameon=False); axes[0].grid(alpha=.18)
    residual = predictions["Actual"] - predictions["Prediction"]
    axes[1].axhline(0, color="black", lw=.9)
    axes[1].plot(residual.index, residual, color="#555555", lw=1.0)
    axes[1].set_ylabel("Sai số"); axes[1].set_xlabel("Thời gian"); axes[1].grid(alpha=.18)
    fig.text(.5, .01,
             f"MAE={metrics['MAE']:.3f} | RMSE={metrics['RMSE']:.3f} | "
             f"R²={metrics['R2']:.3f} | Direction={metrics['Direction_Accuracy']:.1f}%",
             ha="center", color="black")
    fig.tight_layout(rect=[0, .035, 1, 1]); fig.savefig(path, bbox_inches="tight", dpi=240); plt.close(fig)

plot_market_predictions(stock_predictions, stock_model_info, "FPT.VN", FIG_DIR / "stock_model.png")
plot_market_predictions(gold_predictions, gold_model_info, "GC=F", FIG_DIR / "gold_model.png")

# 4.6. Bảng mô tả dữ liệu dùng trong báo cáo
fig, ax = plt.subplots(figsize=(14, 3.3))
ax.axis("off")
table_view = data_profile_df[["Miền", "Nguồn", "Số dòng", "Số cột", "Phạm vi", "Biến đích/đơn vị"]].copy()
table = ax.table(cellText=table_view.values, colLabels=table_view.columns,
                 cellLoc="left", colLoc="left", loc="center",
                 colWidths=[.10, .27, .08, .07, .20, .22])
table.auto_set_font_size(False); table.set_fontsize(8.2); table.scale(1, 1.8)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#cbd5e1")
    if row == 0:
        cell.set_facecolor("#e5e7eb"); cell.set_text_props(color="black", weight="bold")
    elif row % 2 == 0:
        cell.set_facecolor("#f1f5f9")
ax.set_title("Mô tả ba nguồn dữ liệu tại thời điểm chạy", fontsize=14, weight="bold", pad=12)
fig.tight_layout(); fig.savefig(FIG_DIR / "data_profile.png", bbox_inches="tight"); plt.close(fig)

# 4.7. Đồ thị con theo từng miền
def draw_domain_graph(domain, title, filename, color):
    nodes = [n for n, a in kg.graph.nodes(data=True) if a.get("domain") == domain]
    sub = kg.graph.subgraph(nodes).copy()
    fig, ax = plt.subplots(figsize=(10.5, 6.3))
    pos = nx.spring_layout(sub, seed=RANDOM_SEED, k=1.35)
    nx.draw_networkx_edges(sub, pos, ax=ax, alpha=.5, arrows=True, arrowsize=16,
                           edge_color="#64748b", connectionstyle="arc3,rad=0.04")
    nx.draw_networkx_nodes(sub, pos, ax=ax, node_color=color, node_size=1600,
                           edgecolors="black", linewidths=1.5)
    nx.draw_networkx_labels(sub, pos, ax=ax, font_size=8)
    edge_labels = {(u, v): d.get("relation", k) for u, v, k, d in sub.edges(keys=True, data=True)}
    nx.draw_networkx_edge_labels(sub, pos, edge_labels=edge_labels, ax=ax, font_size=6,
                                 rotate=False, label_pos=.55)
    ax.set_title(title, fontsize=15, weight="bold"); ax.axis("off")
    fig.tight_layout(); fig.savefig(FIG_DIR / filename, bbox_inches="tight"); plt.close(fig)

draw_domain_graph("diabetes", "Knowledge Graph miền bệnh tiểu đường", "kg_diabetes.png", "#fb7185")
draw_domain_graph("stock", "Knowledge Graph miền chứng khoán FPT", "kg_stock.png", "#38bdf8")
draw_domain_graph("gold", "Knowledge Graph miền giá vàng", "kg_gold.png", "#fbbf24")

# 4.8. Ảnh code: ảnh được tạo trực tiếp từ chính các đoạn code cốt lõi trong notebook.

def render_code_card(filename, title, source, accent="#000000"):
    source = textwrap.dedent(source).strip("\n")
    lines = [part for line in source.splitlines() for part in (textwrap.wrap(line, width=86, subsequent_indent="    ", replace_whitespace=False, drop_whitespace=False) or [""])]
    height = max(3.0, 0.28 * len(lines) + 1.0)
    fig, ax = plt.subplots(figsize=(10.0, height))
    fig.patch.set_facecolor("white"); ax.set_facecolor("white"); ax.axis("off")
    ax.add_patch(plt.Rectangle((0, .925), 1, .075, transform=ax.transAxes,
                               facecolor="#eeeeee", edgecolor="black", linewidth=1.0))
    ax.text(.025, .962, title, transform=ax.transAxes, va="center", ha="left",
            color="black", fontsize=12.5, weight="bold")
    ax.text(.975, .962, "Tieuluan02_06_Nhom2_Giangkt.ipynb", transform=ax.transAxes,
            va="center", ha="right", color="black", fontsize=8.5)
    numbered = "\n".join(f"{i:>2}  {line}" for i, line in enumerate(lines, 1))
    ax.text(.025, .895, numbered, transform=ax.transAxes, va="top", ha="left",
            family="Liberation Mono", fontsize=11.0, linespacing=1.25, color="black")
    fig.tight_layout(pad=.6)
    fig.savefig(FIG_DIR / filename, bbox_inches="tight", facecolor="white", dpi=240)
    plt.close(fig)

render_code_card("code_market_data.png", "Tải và tiền xử lý dữ liệu thị trường", r'''
def yahoo_chart(symbol, range_="5y", interval="1d"):
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
    response = requests.get(url, params={"range": range_, "interval": interval},
                            timeout=25, headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status()
    result = response.json()["chart"]["result"][0]
    quote = result["indicators"]["quote"][0]
    frame = pd.DataFrame(quote, index=pd.to_datetime(result["timestamp"], unit="s"))
    frame.columns = [c.title() for c in frame.columns]
    return frame[["Open", "High", "Low", "Close", "Volume"]].dropna(subset=["Close"])

def add_indicators(df):
    out = df.copy().sort_index()
    out["Return_1D"] = out["Close"].pct_change()
    out["MA20"] = out["Close"].rolling(20).mean()
    out["MA60"] = out["Close"].rolling(60).mean()
    out["Volatility20"] = out["Return_1D"].rolling(20).std() * np.sqrt(252)
    return out
''')

render_code_card("code_diabetes_training.png", "CV + calibration riêng cho mô hình sức khỏe", r'''
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=.2, random_state=42)
X_train, X_cal, y_train, y_cal = train_test_split(
    X_dev, y_dev, test_size=.2, random_state=42)
name, search, rows = select_model(
    diabetes_candidates, X_train, y_train,
    KFold(5, shuffle=True, random_state=42), "Tiểu đường")
model = search.best_estimator_
interval = calibrate(model, X_cal, y_cal, X_test, y_test)
pred = model.predict(X_test)
''')
render_code_card("code_market_training.png", "TimeSeriesSplit có gap và mô hình baseline", r'''
train = data.loc[data.index < cal_start].iloc[:-1]
cal = data.loc[(data.index >= cal_start) & (data.index < test_start)].iloc[:-1]
test = data.loc[(data.index >= test_start) & (data.index < "2026-10-14")]
cv = TimeSeriesSplit(n_splits=5, gap=1, test_size=63)
name, search, rows = select_model(candidates, Xtr, ytr, cv, domain)
model = search.best_estimator_
interval = calibrate(model, Xcal, ycal, Xte, yte, relative=True)
pred = model.predict(Xte)
baseline = Xte["Close"].to_numpy()
''')

render_code_card("code_kg_core.png", "KnowledgeGraphCore: thực thể, quan hệ và multi-hop", r'''
class KnowledgeGraphCore:
    def __init__(self):
        self.graph = nx.MultiDiGraph()

    def add_entity(self, entity_id, entity_type="Entity", **properties):
        self.graph.add_node(entity_id, type=entity_type, **properties)

    def add_relation(self, subject, predicate, obj, weight=1.0, **properties):
        if subject not in self.graph: self.add_entity(subject)
        if obj not in self.graph: self.add_entity(obj)
        self.graph.add_edge(subject, obj, key=predicate,
                            relation=predicate, weight=float(weight), **properties)

    def multi_hop(self, start, max_hops=2):
        paths = []
        for target in self.graph.nodes:
            if target != start:
                paths.extend(nx.all_simple_paths(self.graph, start, target, cutoff=max_hops))
        return paths
''', "#fbbf24")

render_code_card("code_retriever.png", "VectorRetriever dùng TF-IDF và cosine similarity", r'''
class VectorRetriever:
    def __init__(self, docs):
        self.docs = docs
        self.vectorizer = TfidfVectorizer(
            lowercase=True, ngram_range=(1, 2), min_df=1,
            token_pattern=r"(?u)\b\w+\b"
        )
        self.matrix = self.vectorizer.fit_transform([d["text"] for d in docs])

    def search(self, query, top_k=3):
        query_vector = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vector, self.matrix).ravel()
        order = np.argsort(scores)[::-1][:top_k]
        return [{**self.docs[i], "score": float(scores[i])} for i in order]
''', "#a78bfa")

render_code_card("code_graphrag.png", "GraphRAG: ghép bằng chứng văn bản và bộ ba", r'''
def graphrag_answer(query, top_k=3):
    docs = retriever.search(query, top_k=top_k)
    triples = graph_evidence(query)
    domain = detect_domain(query)
    domain_docs = [d for d in docs if d["domain"] in (domain, "method")]
    statements = [d["text"] for d in domain_docs[:2]]
    graph_text = [f"{s} -[{p}]-> {o}" for s, p, o in triples[:3]]
    answer = " ".join(statements)
    if graph_text:
        answer += " Quan hệ đồ thị: " + "; ".join(graph_text)
    return {"answer": answer, "documents": domain_docs, "triples": triples}
''', "#34d399")

render_code_card("code_deployment.png", "Xuất estimator và web tự chứa", r'''
web_config["models"]["diabetes"] = {
    "estimator": export_estimator(diabetes_model),
    "features": list(X.columns),
    "metrics": diabetes_metrics,
    "interval": diabetes_interval,
}
model_json = json.dumps(web_config, ensure_ascii=False, allow_nan=False)
html = html_template.replace("__MODEL_CONFIG__", model_json)
(WEB_DIR / "index.html").write_text(html, encoding="utf-8")
''')
print("Đã tạo", len(list(FIG_DIR.glob("*.png"))), "hình trong", FIG_DIR)

Đã tạo 20 hình trong /home/jellalaz/Documents/Jellalaz/DATA_CODE/PYTHON/RAG/artifacts_giangkt/figures


## 5. Sinh ứng dụng và báo cáo PDF

Ba mô hình dùng chung khu Dự đoán. Chatbot sức khỏe có khu vực riêng, truy xuất BM25, bằng chứng KG và bộ nhớ hội thoại trong phiên. Mô hình, pipeline tiền xử lý và khoảng dự báo được xuất từ Python sang JavaScript. Notebook chứa toàn bộ HTML/CSS/JS nên không cần chạy tệp khác.


In [ ]:
# Xuất mô hình đã fit; JavaScript đọc cùng cấu trúc tham số/cây với Python.
def export_estimator(model):
    if isinstance(model, ReturnForecaster):
        return {"type":"return", "shrinkage":model.shrinkage, "estimator":export_estimator(model.estimator_)}
    if isinstance(model, PersistenceRegressor):
        return {"type":"persistence"}
    if isinstance(model, Pipeline):
        scaler = model.named_steps["scale"]
        return {"type":"pipeline", "mean":scaler.mean_.tolist(),"scale":scaler.scale_.tolist(),
                "estimator":export_estimator(model.named_steps["model"])}
    if isinstance(model,SVR):
        return {"type":"svr","support_vectors":model.support_vectors_.tolist(),
                "dual_coef":model.dual_coef_[0].tolist(),"gamma":float(model._gamma),
                "intercept":float(model.intercept_[0])}
    if hasattr(model,"coef_"):
        return {"type":"linear","coef":np.asarray(model.coef_).tolist(),"intercept":float(model.intercept_)}
    if isinstance(model,DummyRegressor):
        return {"type":"constant","value":float(model.constant_.ravel()[0])}
    def tree(t):
        a=t.tree_
        return {"left":a.children_left.tolist(),"right":a.children_right.tolist(),
                "feature":a.feature.tolist(),"threshold":a.threshold.tolist(),"value":a.value[:,0,0].tolist()}
    if isinstance(model,RandomForestRegressor):
        return {"type":"forest","trees":[tree(t) for t in model.estimators_]}
    if isinstance(model,GradientBoostingRegressor):
        return {"type":"boost","base":float(model.init_.constant_.ravel()[0]),
                "rate":model.learning_rate,"trees":[tree(t) for t in model.estimators_.ravel()]}
    raise TypeError(type(model).__name__)

def feature_stats(frame):
    return {k:{"min":float(frame[k].min()),"max":float(frame[k].max()),
               "value":float(frame[k].median())} for k in frame.columns}

web_config = {"version":"2.2","generated_at":RUN_AT.isoformat(),"student":{"name":STUDENT_NAME,"id":STUDENT_ID},"models":{}}
web_config["models"]["diabetes"] = {
    "estimator":export_estimator(diabetes_model),"features":list(X.columns),"name":diabetes_model_name,
    "metrics":diabetes_metrics,"inputs":feature_stats(X_train),"interval":diabetes_interval,
    "unit":"điểm tiến triển sau 1 năm","source":"scikit-learn · 442 quan sát đã mắc tiểu đường",
    "reference":{"q25":float(y_train.quantile(.25)),"median":float(y_train.median()),"q75":float(y_train.quantile(.75))},
    "comparison":diabetes_cv_rows,"importance":feature_importance.to_dict(),
    "evaluation":[{"actual":float(a),"prediction":float(p)} for a,p in zip(y_test,diabetes_pred)],
}
for key,label,df,model,metrics,features,latest,preds,rows in [
    ("stock","Chứng khoán FPT",stock_df,stock_model,stock_model_info,market_feature_cols,stock_latest_features,stock_predictions,stock_cv_rows),
    ("gold","Giá vàng GC=F",gold_df,gold_model,gold_model_info,gold_feature_cols,gold_latest_features,gold_predictions,gold_cv_rows)]:
    detail=market_details[label]
    effective_train=detail["train"].iloc[-model.window:] if isinstance(model,ReturnForecaster) and model.window else detail["train"]
    web_config["models"][key]={
        "estimator":export_estimator(model),"features":features,"name":metrics["Model"],"metrics":metrics,
        "inputs":feature_stats(detail["train"]),"latest":{k:float(v) for k,v in latest.items()},
        "interval":detail["live_interval"],"evaluation_interval":detail["interval"],
        "deployment":detail["deployment"],"comparison":rows,"source":market_sources[key],
        "currency":"VND" if key=="stock" else "USD/oz","symbol":"FPT.VN" if key=="stock" else "GC=F",
        "as_of":str(latest.name.date()),
        "ood_space":"relative" if isinstance(model,(ReturnForecaster,PersistenceRegressor)) else "raw",
        "ood_inputs":feature_stats(pd.DataFrame(ReturnForecaster.transform(effective_train),columns=features)) if isinstance(model,(ReturnForecaster,PersistenceRegressor)) else feature_stats(effective_train),
        "history":[{"date":str(i.date()),"close":float(r.Close),"ma20":float(r.MA20)} for i,r in df.dropna().tail(80).iterrows()],
        "evaluation":[{"date":str(i.date()),"actual":float(r.Actual),"prediction":float(r.Prediction),"baseline":float(r.Persistence)} for i,r in preds.iterrows()],
    }
# Mốc trước nâng cấp, cùng nhãn và cùng giai đoạn test; không dùng để tuning.
for key,old_mae,old_name in [("diabetes",42.9364023293,"Ridge"),("stock",728.1284466911765,"Persistence"),("gold",50.47005460824505,"Ridge return")]:
    web_config["models"][key]["previous"]={"version":"2.0","name":old_name,"mae":old_mae,
        "improvement_pct":100*(1-web_config["models"][key]["metrics"]["MAE"]/old_mae)}
# Vectors độc lập để kiểm tra sai số suy luận Python/JavaScript sau khi đóng gói.
parity_vectors=[]
for key,model,frame in [("diabetes",diabetes_model,X_test),("stock",stock_model,market_details["Chứng khoán FPT"]["test"]),("gold",gold_model,market_details["Giá vàng GC=F"]["test"])]:
    for _,row in frame.iloc[:12].iterrows():
        parity_vectors.append({"kind":key,"x":row.tolist(),"expected":float(model.predict(row.to_frame().T)[0])})
(DATA_DIR/"model_parity.json").write_text(json.dumps(parity_vectors),encoding="utf-8")
(DATA_DIR/"web_model_config.json").write_text(json.dumps(web_config,ensure_ascii=False,allow_nan=False),encoding="utf-8")

health_knowledge = json.loads('{"version": "2.2", "reviewed_at": "2026-10-16", "documents": [{"id": "bloodpressure", "title": "Hiểu số đo huyết áp", "terms": ["huyết áp", "tâm thu", "tâm trương", "mmhg", "đo huyết áp"], "text": "Số trên là tâm thu, số dưới là tâm trương. Hãy gửi cả hai số và đơn vị, ví dụ 120/80 mmHg. Một lần đo chưa xác lập chẩn đoán; ghi lại số đo và trao đổi với cơ sở y tế.", "source": "pressure", "triples": [["Huyết áp", "gồm", "tâm thu / tâm trương"], ["Phân loại", "cần", "số đo và bối cảnh"]]}, {"id": "meanpressure", "title": "Huyết áp trung bình trong dữ liệu", "terms": ["huyết áp trung bình", "huyết áp động mạch trung bình", "map", "bp"], "text": "Trường bp của bộ diabetes là huyết áp trung bình, không phải số trên hoặc số dưới của máy đo. Không áp bảng phân loại tâm thu/tâm trương lên một giá trị bp. Nếu muốn đánh giá huyết áp, gửi cặp số đo gốc.", "source": "dataset", "triples": [["bp", "biểu diễn", "huyết áp trung bình"], ["bp", "không đồng nhất với", "huyết áp tâm thu"]]}, {"id": "lipids", "title": "Đọc bảng mỡ máu", "terms": ["cholesterol", "mỡ máu", "lipid", "toàn phần"], "text": "Bảng mỡ máu gồm cholesterol toàn phần, LDL, HDL và triglyceride. Cần xác định đúng loại xét nghiệm và đơn vị; không kết luận nguy cơ tim mạch chỉ từ một con số. Mục tiêu LDL còn phụ thuộc bệnh nền và nguy cơ cá nhân.", "source": "lipids", "triples": [["Bảng mỡ máu", "gồm", "toàn phần / LDL / HDL / triglyceride"], ["Mục tiêu LDL", "phụ thuộc", "nguy cơ cá nhân"]]}, {"id": "hdl", "title": "HDL cholesterol", "terms": ["hdl", "cholesterol tốt"], "text": "HDL tham gia đưa cholesterol về gan. Ngưỡng tham chiếu thấp khác nhau theo giới; HDL không đủ để kết luận toàn bộ nguy cơ tim mạch. Gửi số đo kèm mg/dL hoặc mmol/L để được giải thích.", "source": "lipids", "triples": [["HDL", "cần đối chiếu", "giới và bảng mỡ máu"]]}, {"id": "ldl", "title": "LDL cholesterol", "terms": ["ldl", "cholesterol xấu"], "text": "LDL liên quan đến tích tụ cholesterol trong thành mạch. Người có bệnh tim mạch hoặc nguy cơ cao có thể cần mục tiêu LDL thấp hơn mức tham chiếu chung; không tự dùng hoặc đổi liều thuốc theo chatbot.", "source": "lipids", "triples": [["LDL", "liên quan", "nguy cơ xơ vữa"], ["Mục tiêu LDL", "cần", "đánh giá cá nhân"]]}, {"id": "triglyceride", "title": "Triglyceride và log triglyceride", "terms": ["triglyceride", "triglycerid", "triglycerides", "log triglyceride", "s5", "ltg"], "text": "Triglyceride là một loại mỡ máu. s5/ltg trong bộ diabetes là đặc trưng nghiên cứu có mô tả chưa hoàn toàn rõ; không tự biến s5 thành xét nghiệm triglyceride để phân loại. Hãy gửi giá trị triglyceride gốc và đơn vị trên phiếu.", "source": "dataset", "triples": [["s5 / ltg", "không tự coi là", "xét nghiệm triglyceride gốc"], ["Triglyceride", "cần có", "đơn vị xét nghiệm"]]}, {"id": "ratio", "title": "Tỷ lệ cholesterol trên HDL", "terms": ["tỷ lệ cholesterol", "cholesterol/hdl", "cholesterol / hdl", "tỉ lệ cholesterol", "s4"], "text": "Tỷ lệ là đại lượng không có đơn vị nồng độ. Không dùng một tỷ lệ riêng lẻ để kết luận nguy cơ hay thay cho LDL và toàn bộ hồ sơ. Đối chiếu với các thành phần mỡ máu gốc.", "source": "lipids", "triples": [["Tỷ lệ cholesterol / HDL", "không thay thế", "hồ sơ nguy cơ đầy đủ"]]}, {"id": "demographics", "title": "Tuổi và mã giới tính của mô hình", "terms": ["tuổi", "mã giới tính", "giới tính", "sex"], "text": "Tuổi và mã giới tính là biến đầu vào nghiên cứu, không phải kết quả chẩn đoán. Mã 1/2 không được chatbot tự diễn giải thành giới để áp ngưỡng HDL; cần thông tin do người dùng xác nhận. Trẻ em và thai kỳ có cách đánh giá riêng.", "source": "dataset", "triples": [["Mã giới tính nghiên cứu", "không tự dùng để", "chọn ngưỡng HDL"]]}, {"id": "plate", "title": "Một bữa ăn cân bằng", "terms": ["bữa ăn", "ăn gì", "thực đơn", "ăn sáng", "ăn trưa", "ăn tối", "rau", "đạm", "đĩa ăn"], "text": "Gợi ý minh họa: nửa đĩa rau luộc hoặc salad không ngọt; một phần tư cá, thịt nạc hoặc đậu phụ; phần tư còn lại là cơm hoặc thực phẩm chứa tinh bột. Chọn nước lọc. Điều chỉnh khẩu phần theo nhu cầu, thuốc và mục tiêu cá nhân.", "source": "meal", "triples": [["Bữa ăn", "chia theo", "phương pháp chiếc đĩa"], ["Chiếc đĩa", "ưu tiên", "rau không tinh bột"]]}, {"id": "carb", "title": "Có cần bỏ hoàn toàn tinh bột?", "terms": ["tinh bột", "carb", "carbohydrate", "cơm", "gạo", "bún", "phở", "bánh mì", "kiêng"], "text": "Không cần tự bỏ toàn bộ tinh bột. Chú ý khẩu phần, phân bố bữa ăn đều và kết hợp đạm, rau. Ưu tiên thực phẩm ít chế biến. Lượng carbohydrate phù hợp cần xem cùng thuốc, mức hoạt động và chỉ số sau bữa ăn.", "source": "meal", "triples": [["Carbohydrate", "ảnh hưởng", "đường huyết"], ["Khẩu phần", "cần điều chỉnh theo", "nhu cầu cá nhân"]]}, {"id": "fruit", "title": "Trái cây và đồ uống", "terms": ["trái cây", "hoa quả", "nước ép", "sữa", "trà sữa", "nước ngọt", "uống gì", "chuối"], "text": "Ưu tiên trái cây nguyên quả theo khẩu phần, nước lọc hoặc đồ uống không thêm đường. Nước ép và đồ uống có đường có thể làm đường huyết tăng nhanh. Không có một loại trái cây phù hợp vô hạn cho mọi người.", "source": "meal", "triples": [["Nước ép", "có thể làm tăng nhanh", "đường huyết"]]}, {"id": "activity", "title": "Vận động theo khả năng", "terms": ["vận động", "tập", "thể dục", "đi bộ", "thể thao", "bao nhiêu phút"], "text": "Mục tiêu tham khảo cho người lớn là ít nhất 150 phút vận động mức vừa mỗi tuần. Có thể bắt đầu bằng các buổi ngắn rồi tăng dần theo khả năng. Nếu có biến chứng hoặc dùng thuốc dễ gây hạ đường huyết, hãy trao đổi kế hoạch tập với nhân viên y tế.", "source": "who", "triples": [["Vận động", "hỗ trợ", "quản lý đường huyết"], ["Kế hoạch tập", "phụ thuộc", "thể trạng"]]}, {"id": "a1c", "title": "Hiểu chỉ số HbA1c", "terms": ["hba1c", "a1c", "đường trung bình", "ba tháng"], "text": "HbA1c phản ánh mức đường huyết trung bình trong khoảng ba tháng. Ngưỡng xét nghiệm chẩn đoán không giống mục tiêu điều trị đã cá nhân hóa. Kết quả có thể bị ảnh hưởng bởi thai kỳ và một số bệnh máu. Nếu có con số cụ thể, bạn có thể nhập kèm dấu %. ", "source": "diagnosis", "triples": [["HbA1c", "phản ánh", "đường huyết trung bình"], ["Thai kỳ / bệnh máu", "có thể ảnh hưởng", "HbA1c"]]}, {"id": "testing", "title": "Đọc kết quả xét nghiệm", "terms": ["đường huyết", "glucose", "lúc đói", "sau ăn", "xét nghiệm", "nguy cơ", "chẩn đoán", "có bệnh"], "text": "Bạn cho biết giá trị, đơn vị mg/dL hoặc mmol/L và thời điểm đo. Đo tại nhà, đo lúc đói, sau ăn và nghiệm pháp glucose có cách diễn giải khác nhau. Chẩn đoán thường cần xét nghiệm xác nhận.", "source": "diagnosis", "triples": [["Glucose", "cần có", "đơn vị và thời điểm đo"]]}, {"id": "low", "title": "Khi đường huyết thấp", "terms": ["hạ đường huyết", "run tay", "vã mồ hôi", "đói cồn cào", "tụt đường"], "text": "Dưới 70 mg/dL được coi là thấp. Người lớn còn tỉnh và nuốt được có thể áp dụng quy tắc 15-15: 15 g carbohydrate nhanh, đo lại sau 15 phút và lặp lại nếu còn thấp. Không cho ăn uống khi bất tỉnh hoặc không nuốt được; cần trợ giúp y tế khẩn cấp.", "source": "low", "triples": [["Glucose dưới 70", "gợi ý", "hạ đường huyết"], ["Không nuốt được", "không được", "cho ăn uống"]]}, {"id": "urgent", "title": "Dấu hiệu cần trợ giúp ngay", "terms": ["cấp cứu", "khẩn cấp", "khó thở", "bất tỉnh", "co giật", "lú lẫn", "nguy hiểm"], "text": "Nếu đang bất tỉnh, co giật, lú lẫn rõ, khó thở hoặc không nuốt được, cần người bên cạnh gọi cấp cứu địa phương hoặc đưa đến cơ sở y tế ngay. Không tự lái xe và không cố cho ăn uống khi khả năng nuốt bị ảnh hưởng.", "source": "low", "triples": [["Rối loạn ý thức", "cần", "trợ giúp y tế khẩn cấp"]]}, {"id": "medicine", "title": "Về thuốc và liều dùng", "terms": ["thuốc", "metformin", "insulin", "liều", "uống mấy viên", "ngừng thuốc", "tăng liều", "giảm liều"], "text": "Loại thuốc và liều phụ thuộc chẩn đoán, chức năng thận, thuốc khác và nguy cơ hạ đường huyết. Mình không thể quyết định liều phù hợp cho bạn. Hãy ghi tên thuốc, liều hiện tại, thời điểm uống và số đo để trao đổi với bác sĩ; không tự tăng, giảm hoặc ngừng thuốc.", "source": "care", "triples": [["Liều thuốc", "cần được chỉ định bởi", "nhân viên y tế"]]}, {"id": "monitor", "title": "Theo dõi để trao đổi khi tái khám", "terms": ["theo dõi", "tái khám", "đo mấy lần", "nhật ký", "mục tiêu", "đã mắc", "đã bị"], "text": "Ghi thời điểm đo, kết quả, bữa ăn, vận động, thuốc và triệu chứng. Tần suất đo và mục tiêu cần cá nhân hóa, đặc biệt nếu đang dùng insulin. Mục tiêu điều trị của người đã mắc tiểu đường khác bảng ngưỡng dùng để chẩn đoán ban đầu.", "source": "care", "triples": [["Nhật ký", "hỗ trợ", "tái khám"], ["Mục tiêu điều trị", "phụ thuộc", "từng người"]]}, {"id": "symptoms", "title": "Triệu chứng và thăm khám", "terms": ["triệu chứng", "khát", "tiểu nhiều", "sụt cân", "mờ mắt", "mệt mỏi", "dấu hiệu sớm"], "text": "Khát nhiều, tiểu nhiều, mệt, mờ mắt hoặc sụt cân không rõ nguyên nhân có thể xuất hiện khi đường huyết cao, nhưng cũng có thể do nguyên nhân khác. Nên khám và làm xét nghiệm; không thể xác nhận tiểu đường chỉ qua triệu chứng.", "source": "who", "triples": [["Triệu chứng", "cần được xác nhận bằng", "khám và xét nghiệm"]]}, {"id": "model", "title": "Điểm mô hình có ý nghĩa gì?", "terms": ["mô hình", "điểm tiến triển", "dự đoán", "r2", "sai số", "độ chính xác", "phần trăm"], "text": "Mô hình học trên 442 người đã mắc tiểu đường để ước lượng điểm tiến triển sau một năm. Điểm này không phải glucose, HbA1c hay phần trăm nguy cơ mắc bệnh. Khoảng dự báo và sai số ngoài mẫu giúp nhìn thấy mức bất định.", "source": "dataset", "triples": [["Mô hình diabetes", "dự báo", "điểm tiến triển nghiên cứu"], ["Điểm tiến triển", "không phải", "xác suất mắc bệnh"]]}], "sources": {"diagnosis": {"label": "NIDDK · Xét nghiệm & chẩn đoán", "url": "https://www.niddk.nih.gov/health-information/diabetes/overview/tests-diagnosis"}, "low": {"label": "CDC · Xử trí hạ đường huyết", "url": "https://www.cdc.gov/diabetes/treatment/treatment-low-blood-sugar-hypoglycemia.html"}, "meal": {"label": "CDC · Lập kế hoạch bữa ăn", "url": "https://www.cdc.gov/diabetes/healthy-eating/diabetes-meal-planning.html"}, "who": {"label": "WHO · Diabetes", "url": "https://www.who.int/news-room/fact-sheets/detail/diabetes"}, "care": {"label": "NIDDK · Quản lý bệnh tiểu đường", "url": "https://www.niddk.nih.gov/health-information/diabetes/overview/managing-diabetes"}, "dataset": {"label": "scikit-learn · Dữ liệu diabetes", "url": "https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html"}, "pressure": {"label": "AHA · Hiểu số đo huyết áp", "url": "https://www.heart.org/en/health-topics/high-blood-pressure/understanding-blood-pressure-readings"}, "lowpressure": {"label": "AHA · Huyết áp thấp", "url": "https://www.heart.org/en/health-topics/high-blood-pressure/the-facts-about-high-blood-pressure/low-blood-pressure-when-blood-pressure-is-too-low"}, "pregnancyPressure": {"label": "ACOG · Huyết áp trong thai kỳ", "url": "https://www.acog.org/womens-health/faqs/preeclampsia-and-high-blood-pressure-during-pregnancy"}, "lipids": {"label": "MedlinePlus · Các chỉ số cholesterol", "url": "https://medlineplus.gov/cholesterollevelswhatyouneedtoknow.html"}, "cholesterol": {"label": "Mayo Clinic · Xét nghiệm cholesterol", "url": "https://www.mayoclinic.org/tests-procedures/cholesterol-test/about/pac-20384601"}, "triglycerides": {"label": "MedlinePlus · Triglyceride", "url": "https://medlineplus.gov/triglycerides.html"}}}')
(DATA_DIR/"health_knowledge.json").write_text(json.dumps(health_knowledge,ensure_ascii=False,indent=2),encoding="utf-8")

html_template = '<!doctype html>\n<html lang="vi"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<meta name="description" content="Không gian dự báo tiểu đường, chứng khoán FPT, giá vàng và trợ lý sức khỏe có dẫn nguồn.">\n<title>Kiều Trường Giang · B23DCCN254 | Insight Lab</title>\n<style>:root{--ink:#142d38;--muted:#667a82;--teal:#087c69;--mint:#dff3e9;--line:#dfe7e6;--bg:#f6f8f7;--paper:#fff;--red:#b34241;--amber:#8a5d19;--navy:#142e3b}*{box-sizing:border-box}html{scroll-behavior:smooth;scroll-padding-top:90px}body{margin:0;font:15px/1.6 Inter,ui-sans-serif,system-ui,-apple-system,"Segoe UI",sans-serif;background:var(--bg);color:var(--ink)}a{color:inherit}button,input,select,textarea{font:inherit}button,a,input,select,textarea,summary{-webkit-tap-highlight-color:transparent}button{cursor:pointer}button:disabled{cursor:not-allowed;opacity:.45}a{text-decoration:none}button:focus-visible,a:focus-visible,summary:focus-visible{outline:3px solid #65c8ae;outline-offset:4px}.shell{max-width:1240px;margin:0 auto;padding:0 32px}.topbar{position:sticky;top:0;z-index:20;background:#f6f8f7f2;backdrop-filter:blur(16px);border-bottom:1px solid var(--line)}.nav-inner{display:flex;align-items:center;gap:30px;height:78px}.brand{display:flex;align-items:center;font-size:21px;font-weight:800;letter-spacing:-.7px;white-space:nowrap}.brand>span:not(.brand-mark):not(.version){color:var(--teal)}.brand-mark{width:33px;height:33px;display:grid;place-items:center;background:var(--ink);border-radius:10px;color:#ddf4df;font-size:17px;margin-right:10px}.version{font-size:10px;letter-spacing:0;color:var(--muted);border:1px solid var(--line);padding:1px 5px;border-radius:4px;margin-left:9px}.nav-links{display:flex;gap:26px;margin:auto;font-size:13px;font-weight:600}.nav-links a:hover,.github:hover{color:var(--teal)}.github{font-size:13px;font-weight:650}.hero{display:grid;grid-template-columns:1.35fr 1fr;gap:50px;align-items:center;padding-top:65px;padding-bottom:60px}.eyebrow{font-size:10px;letter-spacing:.15em;font-weight:800;color:var(--teal);display:flex;gap:8px;align-items:center}.live-dot{display:inline-block;width:6px;height:6px;border-radius:50%;background:#13a786;box-shadow:0 0 0 4px #13a78615}h1{font-size:clamp(34px,4.5vw,57px);line-height:1.16;letter-spacing:-2.2px;margin:21px 0}h1 em{font-style:normal;color:var(--teal)}.intro{max-width:540px;color:var(--muted);font-size:15px}.hero-actions{display:flex;gap:16px;align-items:center;margin-top:30px}.button{display:inline-flex;align-items:center;justify-content:center;gap:18px;border:0;border-radius:9px;padding:12px 17px;font-size:13px;font-weight:700;transition:background .15s,transform .15s}.button:hover{transform:translateY(-1px)}.button.primary{background:var(--teal);color:#fff}.button.primary:hover{background:#056151}.button.subtle{background:transparent;color:var(--ink)}.button.outline{border:1px solid #bfd5cd;background:#fff;color:var(--teal)}.full{width:100%}.hero-visual{height:300px;position:relative;overflow:hidden;border-radius:24px;background:radial-gradient(ellipse,#d4e8df70,transparent 70%)}.orbit{position:absolute;border:1px solid #cadfd6;border-radius:50%;top:50%;left:50%;transform:translate(-50%,-55%)}.orbit-one{width:220px;height:220px}.orbit-two{width:330px;height:330px;border-style:dashed}.orbit-core{position:absolute;left:50%;top:46%;transform:translate(-50%,-50%);width:124px;height:124px;background:var(--navy);box-shadow:0 12px 40px #142e3b20;border:7px solid #fff;border-radius:50%;display:flex;align-items:center;justify-content:center;flex-direction:column;font-size:10px;letter-spacing:.12em;color:#91b7b7}.orbit-core span{font-size:21px;letter-spacing:-.6px;color:#c6f2da;margin-top:3px}.orbit-node{position:absolute;background:#fff;border:1px solid var(--line);box-shadow:0 7px 20px #142e3b06;padding:8px 16px;border-radius:10px;font-size:12px;font-weight:650}.n1{top:20px;left:40%}.n2{right:6%;top:52%}.n3{left:7%;top:62%}.visual-caption{position:absolute;bottom:0;width:100%;text-align:center;color:var(--muted);font-size:11px}.section{padding:35px 0 40px;border-top:1px solid var(--line)}.section-head{display:flex;justify-content:space-between;align-items:end;gap:20px;margin-bottom:25px}.section-head>.small{max-width:200px;text-align:right}h2{font-size:27px;line-height:1.3;letter-spacing:-.8px;margin:7px 0}h3{font-size:19px;letter-spacing:-.35px;margin:7px 0 12px}.section-description{margin:10px 0 0;color:var(--muted);font-size:14px}.small,.hint{font-size:12px;color:var(--muted);line-height:1.6}.hint{margin:8px 0 18px}.model-tabs{display:grid;grid-template-columns:repeat(3,1fr);gap:13px;margin-bottom:22px}.model-tab{display:flex;align-items:center;gap:14px;text-align:left;background:white;border:1px solid var(--line);border-radius:12px;padding:18px;transition:.18s;color:var(--ink)}.model-tab:hover{border-color:#80b6a5}.model-tab.active{border-color:var(--teal);box-shadow:0 0 0 1px var(--teal);background:#f0f9f5}.model-tab strong{font-size:14px;display:block}.model-tab small{color:var(--muted);font-size:11px;display:block;margin-top:3px}.tab-icon{font-size:24px;width:42px;height:42px;border-radius:10px;display:grid;place-items:center;flex-shrink:0}.health{background:#fbeee9;color:#bd6658}.stock{background:#e9eff9;color:#4671a9}.gold{background:#fbf0da;color:#a2781b}.tab-arrow{margin-left:auto;color:var(--muted)}.workspace{display:grid;grid-template-columns:1fr 1.05fr;gap:22px;align-items:start}[hidden]{display:none!important}.card{background:var(--paper);border:1px solid var(--line);border-radius:14px;padding:26px}.card-heading{display:flex;align-items:center;justify-content:space-between;gap:10px}.kicker{font-size:10px;letter-spacing:.12em;font-weight:800;color:var(--muted)}.text-button{background:none;border:0;padding:7px 0;font-size:12px;color:var(--teal);font-weight:700;white-space:nowrap}.form-grid{display:grid;grid-template-columns:1fr 1fr;gap:15px}.form-grid label{display:block;font-size:12px;font-weight:650;color:#374d58}.unit{font-size:10px;font-weight:400;color:var(--muted);display:block}.form-grid input,.form-grid select{display:block;width:100%;border:1px solid #d6dfde;border-radius:8px;background:#fbfcfc;color:var(--ink);padding:10px 11px;margin-top:6px;outline:none;transition:.15s}.form-grid input:focus,.form-grid select:focus,.composer textarea:focus{border-color:var(--teal);box-shadow:0 0 0 3px #087c6912}.advanced{margin-top:22px;padding-top:15px;border-top:1px solid var(--line)}summary{cursor:pointer;font-size:12px;font-weight:700}summary span{color:var(--muted);font-weight:400;margin-left:10px}.advanced[open] summary{margin-bottom:12px}.form-divider{height:1px;background:var(--line);margin:22px 0 17px}.context-grid{margin-top:12px}.check{display:flex;align-items:center;gap:8px;font-size:12px;margin:16px 0}.check input{accent-color:var(--teal)}.error-box{background:#fff0ec;color:#a43a30;border-radius:8px;padding:12px;margin:12px 0;font-size:12px}.output-column{display:grid;gap:16px;min-width:0}.result-card{min-height:420px;background:linear-gradient(135deg,#fcfefd,#fff);position:relative}.result-card>h3{font-size:26px;line-height:1.4;margin-top:20px}.result-card>p{color:var(--muted);font-size:13px;max-width:340px}.empty-icon{font-size:42px;color:#4d9780;margin:20px 0 25px}.empty-steps{border-top:1px solid var(--line);margin-top:30px;padding-top:18px;display:grid;gap:8px;font-size:12px;color:var(--muted)}.info-strip{font-size:11px;color:#677a7b;padding:4px 12px;line-height:1.7}.score-header{display:flex;align-items:start;justify-content:space-between;gap:10px}.badge{font-size:10px;border-radius:5px;padding:3px 7px;background:#eaf5ef;color:var(--teal);font-weight:700;white-space:nowrap}.score{font-size:48px;line-height:1.2;letter-spacing:-2px;font-weight:700;margin:18px 0 4px;font-variant-numeric:tabular-nums}.score small{font-size:15px;letter-spacing:0;color:var(--muted);font-weight:500}.interval{font-size:12px;color:var(--muted);padding-bottom:18px;border-bottom:1px solid var(--line)}.range-bar{height:7px;background:#e2ece8;border-radius:7px;position:relative;margin:12px 0}.range-fill{height:100%;background:#68b499;border-radius:7px}.assessment-row{display:flex;gap:10px;padding:12px 0;border-bottom:1px solid #edf1ef;font-size:13px}.assessment-row strong{display:block}.assessment-row small{display:block;color:var(--muted);margin-top:3px;font-size:11px}.signal{width:8px;height:8px;border-radius:50%;background:#72918b;margin-top:7px;flex-shrink:0}.signal.good{background:#1c9a78}.signal.watch{background:#c39140}.signal.high{background:#c04c4c}.callout{font-size:12px;line-height:1.7;padding:13px 15px;border-radius:8px;background:#f1f6f3;margin:14px 0}.callout.watch{background:#fff8eb;color:#76571f}.callout.high{background:#fdf0ed;color:#993e36}.result-actions{display:flex;gap:10px;margin-top:20px;flex-wrap:wrap}.result-actions .button{font-size:12px;flex:1}.details-list{font-size:12px;color:var(--muted);padding-left:18px}.details-list li{margin:5px 0}.sources{margin-top:12px;font-size:10px;color:var(--muted)}.sources a{color:var(--teal);text-decoration:underline;text-underline-offset:3px}.chart{width:100%;display:block;height:auto}.chart-legend{display:flex;gap:14px;font-size:10px;color:var(--muted);margin-bottom:12px}.legend-dot{display:inline-block;width:14px;height:3px;vertical-align:middle;margin-right:4px;background:var(--teal)}.legend-dot.gray{background:#bac3c1}.model-details{margin-top:16px}.stat-row{display:flex;justify-content:space-between;padding:9px 0;border-bottom:1px solid var(--line);font-size:12px;gap:10px}.stat-row b{font-variant-numeric:tabular-nums}.price-context{display:flex;gap:12px;margin:14px 0}.price-context div{background:#f4f7f6;border-radius:8px;padding:10px 12px;flex:1}.price-context small{font-size:10px;color:var(--muted);display:block}.price-context strong{font-size:14px}.chat-layout{display:grid;grid-template-columns:1fr 300px;gap:22px;align-items:start}.chat-card{background:#fff;border:1px solid var(--line);border-radius:15px;overflow:hidden;min-width:0}.chat-header{display:flex;align-items:center;gap:12px;padding:19px 22px;border-bottom:1px solid var(--line)}.assistant-avatar{width:38px;height:38px;background:var(--navy);color:#d7f4dd;border-radius:11px;font-size:26px;display:grid;place-items:center}.chat-header strong{font-size:14px}.chat-header small{display:block;font-size:10px;color:var(--muted);margin-top:3px}.chat-header .live-dot{width:4px;height:4px;margin-right:5px;box-shadow:none}.chat-header .text-button{margin-left:auto}.chat-sidebar{display:grid;gap:16px}.chat-sidebar .card{padding:22px}.chat-sidebar h3{font-size:17px}.context-content{font-size:12px;color:var(--muted);margin:12px 0 18px}.context-content .stat-row{font-size:11px}.topics button{background:transparent;border:0;border-bottom:1px solid var(--line);width:100%;padding:14px 0;text-align:left;font-size:12px;color:var(--ink)}.topics button span{float:right;color:var(--muted)}.topics button:last-child{border:0}.privacy-pill{font-size:11px;color:var(--teal);background:#eaf3ef;padding:6px 10px;border-radius:20px;white-space:nowrap}#messages{height:520px;overflow-y:auto;overscroll-behavior:contain;padding:26px;background:#fcfdfc;scroll-behavior:smooth}.message{display:flex;gap:10px;margin-bottom:22px}.message .mini-avatar{width:26px;height:26px;flex:none;display:grid;place-items:center;background:#e5f2ea;border-radius:8px;color:var(--teal);font-size:16px}.message.user{justify-content:flex-end}.message .bubble{max-width:92%;padding:15px 18px;border:1px solid var(--line);background:white;border-radius:0 13px 13px 13px;font-size:13px;overflow-wrap:anywhere}.message.user .bubble{background:var(--navy);color:white;border:0;border-radius:13px 0 13px 13px;white-space:pre-wrap}.bubble p{margin:7px 0}.bubble h4{font-size:14px;margin:0 0 8px}.bubble ul{padding-left:17px;margin:8px 0}.bubble li{margin:5px 0}.bubble details{font-size:10px;color:var(--muted);margin-top:10px;border-top:1px solid var(--line);padding-top:8px}.bubble details summary{font-size:10px}.bubble .sources{font-size:11px}.bubble .sources a{display:inline-block;margin-right:7px}.composer{display:flex;gap:10px;padding:14px 20px 8px}.composer textarea{flex:1;min-width:0;resize:vertical;max-height:120px;min-height:46px;border:1px solid var(--line);border-radius:9px;padding:12px;font-size:12px;outline:none}.composer button{align-self:end;height:46px}.chat-footnote{font-size:9px;color:var(--muted);padding:0 22px 12px}.quick-replies{display:flex;gap:7px;padding:0 22px;flex-wrap:wrap}.quick-replies button{font-size:11px;border:1px solid #c9dfd4;background:#f3faf6;color:var(--teal);border-radius:20px;padding:5px 10px}.quality-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:16px}.quality-card h3{font-size:17px}.quality-card .metric-value{font-size:30px;letter-spacing:-1px;font-weight:700}.quality-card .metric-label{font-size:11px;color:var(--muted)}.quality-card .stat-row{font-size:11px}.quality-card .badge{margin-bottom:9px;display:inline-block}.methodology{margin-top:18px}.methodology summary{font-size:13px}.comparison-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:24px;margin-top:18px}.comparison-grid h4{margin:6px 0}.comparison-grid table{width:100%;font-size:11px;border-collapse:collapse}.comparison-grid td,.comparison-grid th{padding:6px;border-bottom:1px solid var(--line);text-align:left}.comparison-grid td:last-child,.comparison-grid th:last-child{text-align:right}.comparison-grid tr.selected{background:#eaf5ef}.footer{display:flex;align-items:center;justify-content:space-between;gap:25px;padding-top:30px;padding-bottom:35px;border-top:1px solid var(--line)}.footer p{font-size:11px;color:var(--muted)}.footer>.small{font-size:10px}.sr-only{position:absolute;width:1px;height:1px;overflow:hidden;clip:rect(0,0,0,0)}@media(max-width:1000px){.shell{padding-left:24px;padding-right:24px}.hero{gap:20px}.chat-layout{grid-template-columns:1fr 260px}.card{padding:20px}.model-tab{gap:9px;padding:14px}.tab-arrow{display:none}.nav-links{gap:16px}.hero-actions{gap:4px}.hero-actions .button{font-size:12px}.quality-grid{gap:10px}}@media(max-width:760px){.shell{padding-left:18px;padding-right:18px}.nav-inner{height:66px;gap:12px;flex-wrap:wrap}.nav-links{display:none}.github{margin-left:auto}.hero{grid-template-columns:1fr;padding-top:35px;padding-bottom:35px}.hero-visual{display:none}h1{font-size:40px;letter-spacing:-1.7px}.intro{font-size:14px}.section{padding-top:28px;padding-bottom:30px}.section-head{align-items:start;flex-direction:column;gap:6px}.section-head>.small{text-align:left;max-width:none}.model-tabs{gap:7px}.model-tab{display:flex;flex-direction:column;text-align:center;padding:12px 5px;gap:7px}.model-tab strong{font-size:11px}.model-tab small{display:none}.tab-icon{width:30px;height:30px;font-size:20px;border-radius:8px}.workspace{grid-template-columns:1fr;gap:14px}.chat-layout{grid-template-columns:1fr}.chat-sidebar{grid-template-columns:1fr}.topics{display:none}.quality-grid{grid-template-columns:1fr}.comparison-grid{grid-template-columns:1fr}.card{padding:20px}h2{font-size:24px}.chat-header{padding:15px}.chat-header .text-button{font-size:10px}#messages{height:480px;padding:18px 12px}.message .bubble{padding:13px;max-width:calc(100% - 32px)}.composer{padding:12px}.composer textarea{font-size:12px}.quick-replies{padding:0 12px}.footer{align-items:start;flex-direction:column;gap:7px}.footer p{margin:0}.form-grid{gap:12px}.privacy-pill{margin-top:6px}.hero-actions{flex-wrap:wrap}.score{font-size:42px}.context-grid{grid-template-columns:1fr}.result-card{min-height:340px}}@media(prefers-reduced-motion:reduce){html,#messages{scroll-behavior:auto}*{transition:none!important}}\n\n.student-banner{background:#142e3b;color:#fff;text-align:center;padding:26px 18px 28px;border-bottom:4px solid #69cbb0}\n.student-name{display:block;font-size:clamp(30px,4.5vw,44px);line-height:1.2;font-weight:800;letter-spacing:-.8px}\n.student-id{display:inline-block;margin-top:12px;padding:6px 18px;border:1px solid #69cbb0;border-radius:8px;background:#204651;color:#d7ffef;font-size:clamp(18px,2.5vw,23px);font-weight:700;letter-spacing:1px}\n.student-project{margin:12px 0 0;color:#d3e3e6;font-size:14px;line-height:1.5}\n</style></head><body>\n<div class="student-banner"><strong class="student-name">Kiều Trường Giang</strong><span class="student-id">MSV: B23DCCN254</span><p class="student-project">Bài tiểu luận · Knowledge Graph & RAG</p></div>\n<nav class="topbar"><div class="shell nav-inner"><a class="brand" href="#predict"><span class="brand-mark">iL</span> Insight<span>Lab</span><span class="version">02</span></a><div class="nav-links"><a href="#predict">Dự đoán</a><a href="#chatbot">Trợ lý sức khỏe</a><a href="#models">Chất lượng mô hình</a></div><a class="github" href="https://github.com/jellalaz/RAG_HTTM" target="_blank" rel="noopener">GitHub ↗</a></div></nav>\n<header class="shell hero"><div><p class="eyebrow"><span class="live-dot"></span> KNOWLEDGE GRAPH + MACHINE LEARNING</p><h1>Từ dữ liệu.<br><em>Đến góc nhìn rõ hơn.</em></h1><p class="intro">Khám phá ba mô hình dự đoán, hiểu kết quả và trò chuyện cùng trợ lý sức khỏe — trong hai không gian riêng biệt.</p><div class="hero-actions"><a class="button primary" href="#predict">Khám phá dự đoán <span>↗</span></a><a class="button subtle" href="#chatbot">Trò chuyện với trợ lý →</a></div></div><div class="hero-visual" aria-hidden="true"><div class="orbit orbit-one"></div><div class="orbit orbit-two"></div><div class="orbit-core">DỮ LIỆU<span>→ insight</span></div><span class="orbit-node n1">Sức khỏe</span><span class="orbit-node n2">Thị trường</span><span class="orbit-node n3">Tri thức</span><div class="visual-caption">Một quy trình · Ba lĩnh vực</div></div></header>\n<main class="shell">\n<section class="section" id="predict"><div class="section-head"><div><p class="eyebrow">01 / KHÔNG GIAN DỰ ĐOÁN</p><h2>Chọn điều bạn muốn khám phá.</h2></div><span class="small">Dữ liệu được cập nhật khi chạy notebook</span></div>\n<div class="model-tabs" role="tablist" aria-label="Chọn mô hình"><button id="tab-diabetes" class="model-tab active" role="tab" aria-selected="true" aria-controls="panel-diabetes" data-kind="diabetes"><span class="tab-icon health">♡</span><span><strong>Tiểu đường</strong><small>Chỉ số sức khỏe & tiến triển</small></span><span class="tab-arrow">↗</span></button><button id="tab-stock" class="model-tab" role="tab" aria-selected="false" aria-controls="panel-stock" tabindex="-1" data-kind="stock"><span class="tab-icon stock">↗</span><span><strong>Chứng khoán FPT</strong><small>Giá đóng cửa phiên kế tiếp</small></span><span class="tab-arrow">↗</span></button><button id="tab-gold" class="model-tab" role="tab" aria-selected="false" aria-controls="panel-gold" tabindex="-1" data-kind="gold"><span class="tab-icon gold">◇</span><span><strong>Giá vàng quốc tế</strong><small>GC=F · USD / ounce</small></span><span class="tab-arrow">↗</span></button></div>\n<div id="panel-diabetes" role="tabpanel" aria-labelledby="tab-diabetes" class="workspace"><div class="card input-card"><div class="card-heading"><div><span class="kicker">HỒ SƠ ĐẦU VÀO</span><h3>Thông số sức khỏe</h3></div><button class="text-button" id="reset-health">Điền mẫu</button></div><p class="hint">Các giá trị điền sẵn là mẫu minh họa. Thay bằng chỉ số của bạn trước khi phân tích.</p><form id="health-form" novalidate><div id="health-fields" class="form-grid"></div><details class="advanced"><summary>Thông số bổ sung cho mô hình <span>4 trường</span></summary><p class="hint">Để trống sẽ dùng trung vị tập huấn luyện; kết quả sẽ ghi rõ các giá trị được thay thế.</p><div id="health-extra" class="form-grid"></div></details><div class="form-divider"></div><span class="kicker">BỐI CẢNH XÉT NGHIỆM</span><div class="form-grid context-grid"><label>Thời điểm đo glucose<select id="glucose-context"><option value="unknown">Chưa xác định</option><option value="fasting">Lúc đói ít nhất 8 giờ</option><option value="postmeal">Sau bữa ăn thông thường</option><option value="ogtt">2 giờ sau nghiệm pháp 75 g glucose</option><option value="random">Đo ngẫu nhiên</option></select></label><label>HbA1c <span class="unit">% · không bắt buộc</span><input id="hba1c" inputmode="decimal" placeholder="Ví dụ: 6,2" autocomplete="off"></label></div><label class="check"><input id="pregnant" type="checkbox">Đang mang thai</label><div id="health-errors" class="error-box" role="alert" hidden></div><button class="button primary full" type="submit">Phân tích kết quả <span>→</span></button></form></div><div class="output-column"><div id="health-result" class="card result-card" aria-live="polite"><div class="empty-icon">◉</div><span class="kicker">KẾT QUẢ CỦA BẠN</span><h3>Hiểu từng chỉ số,<br>thấy bức tranh tổng thể.</h3><p>Nhập thông số và chọn “Phân tích kết quả” để xem nhận xét, điểm tiến triển và khoảng dự báo.</p><div class="empty-steps"><span>01 · Kiểm tra dữ liệu</span><span>02 · Dự đoán & giải thích</span><span>03 · Gợi ý bước tiếp theo</span></div></div><div class="info-strip">Mô hình học từ người đã mắc tiểu đường: đầu ra là điểm tiến triển nghiên cứu, không phải xác suất mắc bệnh. Nhận xét xét nghiệm được tính riêng theo bối cảnh đo.</div></div></div>\n<div id="panel-stock" role="tabpanel" aria-labelledby="tab-stock" class="workspace" hidden></div><div id="panel-gold" role="tabpanel" aria-labelledby="tab-gold" class="workspace" hidden></div></section>\n<section class="section" id="chatbot"><div class="section-head"><div><p class="eyebrow">02 / TRỢ LÝ SỨC KHỎE</p><h2>Một cuộc trò chuyện, rõ từng bước.</h2><p class="section-description">Hỏi về chỉ số, bữa ăn và vận động. Trợ lý sẽ hỏi thêm khi cần và chỉ ra nguồn tham khảo.</p></div><span class="privacy-pill">◉ Xử lý trong trình duyệt</span></div><div class="chat-layout"><div class="chat-card"><div class="chat-header"><div class="assistant-avatar">✳</div><div><strong>Trợ lý sức khỏe</strong><small><span class="live-dot"></span> Tra cứu tri thức có dẫn nguồn</small></div><button class="text-button" id="clear-chat">Cuộc trò chuyện mới</button></div><div id="messages" role="log" aria-live="polite" aria-label="Cuộc trò chuyện"></div><div id="quick-replies" class="quick-replies"></div><form id="chat-form" class="composer"><label class="sr-only" for="chat-input">Câu hỏi cho trợ lý</label><textarea id="chat-input" rows="1" maxlength="1200" placeholder="Ví dụ: Đường huyết 7,2 mmol/L lúc đói có cao không?"></textarea><button class="button primary" type="submit" aria-label="Gửi câu hỏi">Gửi ↗</button></form><div class="chat-footnote">Thông tin tham khảo sức khỏe · Không tự thay đổi liều thuốc theo hội thoại</div></div><aside class="chat-sidebar"><div class="card"><span class="kicker">NGỮ CẢNH CUỘC TRÒ CHUYỆN</span><h3>Trợ lý đang biết gì?</h3><div id="chat-context" class="context-content">Chưa có thông số sức khỏe. Bạn có thể hỏi trực tiếp mà không cần chạy dự đoán.</div><button class="button outline full" id="share-health" disabled>Dùng kết quả đã phân tích</button><p class="hint">Chỉ chia sẻ kết quả với trợ lý khi bạn chọn nút này. Không lưu hồ sơ sau khi tải lại trang.</p></div><div class="card topics"><span class="kicker">BẮT ĐẦU TỪ ĐÂU?</span><button data-question="Gợi ý một bữa ăn cho người tiểu đường">🥗 Gợi ý bữa ăn <span>↗</span></button><button data-question="Tôi nên vận động bao nhiêu?">🚶 Vận động hằng tuần <span>↗</span></button><button data-question="HbA1c là gì?">◉ Hiểu về HbA1c <span>↗</span></button><button data-question="Khi nào cần cấp cứu?">＋ Nhận biết dấu hiệu khẩn <span>↗</span></button></div></aside></div></section>\n<section class="section" id="models"><div class="section-head"><div><p class="eyebrow">03 / ĐẰNG SAU KẾT QUẢ</p><h2>Chất lượng được đo, không đoán.</h2><p class="section-description">Chọn mô hình bằng cross-validation. Tập hiệu chỉnh và tập kiểm tra được giữ riêng.</p></div></div><div id="quality-cards" class="quality-grid"></div><details class="card methodology"><summary>Cách lựa chọn mô hình & đọc các con số</summary><div id="model-comparison"></div><p class="hint">MAE càng thấp càng tốt; R² không phải phần trăm dự đoán đúng. Khoảng dự báo dùng sai số trên tập hiệu chỉnh riêng với mục tiêu bao phủ 90%; tỷ lệ thực tế trên tập kiểm tra được hiển thị riêng. Với thị trường có thay đổi chế độ, độ bao phủ tương lai không được bảo đảm. Persistence giữ nguyên giá gần nhất và có thể thắng mô hình học máy.</p></details></section>\n</main><footer class="shell footer"><a class="brand" href="#">Insight<span>Lab</span></a><p>Kiều Trường Giang · B23DCCN254<br>Knowledge Graph & RAG · Chương 4</p><span id="build-date" class="small"></span></footer>\n<script>const CONFIG=__MODEL_CONFIG__;</script><script>\'use strict\';\n// Bộ nhận diện số đo độc lập với mô hình hồi quy. Không tự suy diễn đơn vị.\nconst ClinicalSupport=(()=>{\n const fields={\n  age:{label:\'Tuổi\',unit:\'năm\',min:1,max:120},\n  sex:{label:\'Mã giới tính dữ liệu\',unit:\'mã nghiên cứu\',min:1,max:2},\n  sys:{label:\'Huyết áp tâm thu\',unit:\'mmHg\',min:30,max:350},\n  dia:{label:\'Huyết áp tâm trương\',unit:\'mmHg\',min:20,max:250},\n  bp:{label:\'Huyết áp động mạch trung bình\',unit:\'mmHg\',min:20,max:250},\n  s1:{label:\'Cholesterol toàn phần\',unit:\'mg/dL\',factor:38.67,min:1,max:1500},\n  s2:{label:\'LDL cholesterol\',unit:\'mg/dL\',factor:38.67,min:1,max:1200},\n  s3:{label:\'HDL cholesterol\',unit:\'mg/dL\',factor:38.67,min:1,max:300},\n  tg:{label:\'Triglyceride\',unit:\'mg/dL\',factor:88.57,min:1,max:10000},\n  s4:{label:\'Tỷ lệ cholesterol / HDL\',unit:\'tỷ lệ\',min:.1,max:30},\n  s5:{label:\'Log triglyceride (s5)\',unit:\'đặc trưng nghiên cứu\',min:1,max:9},\n  pressure:{label:\'Huyết áp\',unit:null},cholesterol:{label:\'Cholesterol\',unit:null}\n };\n const knowledge=[\n  {id:\'bloodpressure\',title:\'Hiểu số đo huyết áp\',terms:[\'huyết áp\',\'tâm thu\',\'tâm trương\',\'mmhg\',\'đo huyết áp\'],text:\'Số trên là tâm thu, số dưới là tâm trương. Hãy gửi cả hai số và đơn vị, ví dụ 120/80 mmHg. Một lần đo chưa xác lập chẩn đoán; ghi lại số đo và trao đổi với cơ sở y tế.\',source:\'pressure\',triples:[[\'Huyết áp\',\'gồm\',\'tâm thu / tâm trương\'],[\'Phân loại\',\'cần\',\'số đo và bối cảnh\']]},\n  {id:\'meanpressure\',title:\'Huyết áp trung bình trong dữ liệu\',terms:[\'huyết áp trung bình\',\'huyết áp động mạch trung bình\',\'map\',\'bp\'],text:\'Trường bp của bộ diabetes là huyết áp trung bình, không phải số trên hoặc số dưới của máy đo. Không áp bảng phân loại tâm thu/tâm trương lên một giá trị bp. Nếu muốn đánh giá huyết áp, gửi cặp số đo gốc.\',source:\'dataset\',triples:[[\'bp\',\'biểu diễn\',\'huyết áp trung bình\'],[\'bp\',\'không đồng nhất với\',\'huyết áp tâm thu\']]},\n  {id:\'lipids\',title:\'Đọc bảng mỡ máu\',terms:[\'cholesterol\',\'mỡ máu\',\'lipid\',\'toàn phần\'],text:\'Bảng mỡ máu gồm cholesterol toàn phần, LDL, HDL và triglyceride. Cần xác định đúng loại xét nghiệm và đơn vị; không kết luận nguy cơ tim mạch chỉ từ một con số. Mục tiêu LDL còn phụ thuộc bệnh nền và nguy cơ cá nhân.\',source:\'lipids\',triples:[[\'Bảng mỡ máu\',\'gồm\',\'toàn phần / LDL / HDL / triglyceride\'],[\'Mục tiêu LDL\',\'phụ thuộc\',\'nguy cơ cá nhân\']]},\n  {id:\'hdl\',title:\'HDL cholesterol\',terms:[\'hdl\',\'cholesterol tốt\'],text:\'HDL tham gia đưa cholesterol về gan. Ngưỡng tham chiếu thấp khác nhau theo giới; HDL không đủ để kết luận toàn bộ nguy cơ tim mạch. Gửi số đo kèm mg/dL hoặc mmol/L để được giải thích.\',source:\'lipids\',triples:[[\'HDL\',\'cần đối chiếu\',\'giới và bảng mỡ máu\']]},\n  {id:\'ldl\',title:\'LDL cholesterol\',terms:[\'ldl\',\'cholesterol xấu\'],text:\'LDL liên quan đến tích tụ cholesterol trong thành mạch. Người có bệnh tim mạch hoặc nguy cơ cao có thể cần mục tiêu LDL thấp hơn mức tham chiếu chung; không tự dùng hoặc đổi liều thuốc theo chatbot.\',source:\'lipids\',triples:[[\'LDL\',\'liên quan\',\'nguy cơ xơ vữa\'],[\'Mục tiêu LDL\',\'cần\',\'đánh giá cá nhân\']]},\n  {id:\'triglyceride\',title:\'Triglyceride và log triglyceride\',terms:[\'triglyceride\',\'triglycerid\',\'triglycerides\',\'log triglyceride\',\'s5\',\'ltg\'],text:\'Triglyceride là một loại mỡ máu. s5/ltg trong bộ diabetes là đặc trưng nghiên cứu có mô tả chưa hoàn toàn rõ; không tự biến s5 thành xét nghiệm triglyceride để phân loại. Hãy gửi giá trị triglyceride gốc và đơn vị trên phiếu.\',source:\'dataset\',triples:[[\'s5 / ltg\',\'không tự coi là\',\'xét nghiệm triglyceride gốc\'],[\'Triglyceride\',\'cần có\',\'đơn vị xét nghiệm\']]},\n  {id:\'ratio\',title:\'Tỷ lệ cholesterol trên HDL\',terms:[\'tỷ lệ cholesterol\',\'cholesterol/hdl\',\'cholesterol / hdl\',\'tỉ lệ cholesterol\',\'s4\'],text:\'Tỷ lệ là đại lượng không có đơn vị nồng độ. Không dùng một tỷ lệ riêng lẻ để kết luận nguy cơ hay thay cho LDL và toàn bộ hồ sơ. Đối chiếu với các thành phần mỡ máu gốc.\',source:\'lipids\',triples:[[\'Tỷ lệ cholesterol / HDL\',\'không thay thế\',\'hồ sơ nguy cơ đầy đủ\']]},\n  {id:\'demographics\',title:\'Tuổi và mã giới tính của mô hình\',terms:[\'tuổi\',\'mã giới tính\',\'giới tính\',\'sex\'],text:\'Tuổi và mã giới tính là biến đầu vào nghiên cứu, không phải kết quả chẩn đoán. Mã 1/2 không được chatbot tự diễn giải thành giới để áp ngưỡng HDL; cần thông tin do người dùng xác nhận. Trẻ em và thai kỳ có cách đánh giá riêng.\',source:\'dataset\',triples:[[\'Mã giới tính nghiên cứu\',\'không tự dùng để\',\'chọn ngưỡng HDL\']]}\n ];\n const num=\'(-?\\\\d+(?:[.,]\\\\d+)?)\';\n const bridge=\'(?:\\\\s+(?:cua toi|cua em|la|bang|do duoc|luc nghi|hien tai))*\\\\s*[:=]?\\\\s*\';\n const unitPattern=\'\\\\s*(mmol\\\\s*\\\\/?\\\\s*l|mg\\\\s*\\\\/?\\\\s*dl|mm\\\\s*hg|kpa|g\\\\s*\\\\/?\\\\s*l|%)?\';\n const patterns=[[\'s4\',\'(?:ty le|ti le)\\\\s+cholesterol\\\\s*(?:\\\\/|tren)\\\\s*hdl|s4\'],[\'s5\',\'log\\\\s+triglycerides?|log\\\\s+triglycerid|ltg|s5\'],\n  [\'bp\',\'huyet ap (?:dong mach )?trung binh|map|bp\'],[\'sys\',\'(?:huyet ap )?(?:tam thu|so tren)\'],[\'dia\',\'(?:huyet ap )?(?:tam truong|so duoi)\'],\n  [\'s1\',\'cholesterol toan phan|total cholesterol|s1\'],[\'s2\',\'ldl(?:[ -]c)?(?: cholesterol)?|s2\'],[\'s3\',\'hdl(?:[ -]c)?(?: cholesterol)?|s3\'],\n  [\'tg\',\'triglycerides?|triglycerid|tg\'],[\'pressure\',\'huyet ap\'],[\'cholesterol\',\'cholesterol\']];\n const toNum=s=>Number(String(s).replace(\',\',\'.\'));\n const unitOf=s=>{s=(s||\'\').replace(/\\s/g,\'\');return /mmol/.test(s)?\'mmol/L\':/mg/.test(s)?\'mg/dL\':/mmhg/.test(s)?\'mmHg\':s||null};\n function extract(n){\n  let rest=n;const entries=[];\n  const pair=rest.match(/(?:\\bhuyet ap\\s*(?:(?:cua toi|la|bang)\\s*)?|^)(-?\\d+(?:[.,]\\d+)?)\\s*\\/\\s*(-?\\d+(?:[.,]\\d+)?)\\s*(mm\\s*hg|kpa)?/);\n  if(pair){entries.push({key:\'sys\',value:toNum(pair[1]),unit:unitOf(pair[3])},{key:\'dia\',value:toNum(pair[2]),unit:unitOf(pair[3])});rest=rest.replace(pair[0],\' \')}\n  for(const [key,pattern] of patterns){\n   const re=new RegExp(\'(?:^|\\\\b)(?:\'+pattern+\')\'+bridge+num+unitPattern,\'g\');\n   rest=rest.replace(re,(all,value,unit)=>{entries.push({key,value:toNum(value),unit:unitOf(unit)});return \' \'});\n  }\n  return entries;\n }\n function fromForm(profile){const p={...profile,missing:[...(profile.missing||[])]};for(const k of p.missing)delete p[k];return p}\n function clearImputation(chat,key){chat.profile.missing=(chat.profile.missing||[]).filter(k=>k!==key);chat.profile.score=null;}\n function describe(key,p){\n  const v=p[key],special=p.pregnant||(p.age!==undefined&&p.age<20);\n  const result=(title,text,level=\'watch\',source=\'lipids\')=>({title,text,level,source});\n  if(key===\'age\')return result(\'Đã ghi nhận tuổi\',`${v} tuổi. Tuổi được dùng làm bối cảnh, không tự suy ra bệnh.`, \'watch\',\'dataset\');\n  if(key===\'sex\')return result(\'Mã nghiên cứu, không phải ngưỡng lâm sàng\',\'Không suy ra giới sinh học từ mã 1/2; bạn có thể xác nhận riêng nếu cần diễn giải HDL.\',\'watch\',\'dataset\');\n  if(key===\'bp\')return result(\'Đây là huyết áp trung bình\',\'Không áp ngưỡng tâm thu/tâm trương cho số này. Bạn gửi cặp huyết áp gốc trên máy (ví dụ 120/80 mmHg) để đánh giá theo bảng tham chiếu.\',\'watch\',\'dataset\');\n  if(key===\'s5\')return result(\'Chỉ số log trong mô hình nghiên cứu\',\'Không tự quy đổi s5 thành triglyceride xét nghiệm: mô tả dữ liệu gốc còn chưa rõ. Hãy dùng triglyceride gốc kèm đơn vị để nhận xét mỡ máu.\',\'watch\',\'dataset\');\n  if(key===\'s4\')return result(\'Tỷ lệ cần đọc cùng bảng mỡ máu\',\'Không có kết luận bệnh từ tỷ lệ này. Cần cholesterol toàn phần, HDL, LDL và bối cảnh nguy cơ cá nhân.\');\n  if(special)return result(\'Cần khoảng tham chiếu riêng\',\'Chưa áp dụng bảng người trưởng thành từ 20 tuổi cho trẻ em, người dưới 20 tuổi hoặc thai kỳ. Hãy đối chiếu phiếu xét nghiệm và bác sĩ.\');\n  const caveat=\' Mức tham khảo người từ 20 tuổi; mục tiêu riêng còn phụ thuộc bệnh nền và bác sĩ.\';\n  if(key===\'s1\')return result(v<200?\'Toàn phần dưới mức tham chiếu 200 mg/dL\':v<240?\'Cholesterol toàn phần cận cao\':\'Cholesterol toàn phần cao\',\'Đọc cùng LDL, HDL và nguy cơ tim mạch.\'+caveat,v<200?\'good\':\'watch\',\'cholesterol\');\n  if(key===\'s2\')return result(v<100?\'LDL dưới 100 mg/dL\':v>=190?\'LDL rất cao\':\'LDL từ 100 mg/dL trở lên\',v>=190?\'Nên trao đổi sớm với cơ sở y tế để đánh giá nguyên nhân và kế hoạch xử trí; không tự dùng thuốc.\':\'Đây không phải mục tiêu điều trị chung cho mọi người. Người có nguy cơ cao có thể cần mức thấp hơn.\'+caveat,v>=190?\'high\':v<100?\'good\':\'watch\');\n  if(key===\'s3\'){\n   if(!p.labSex)return result(\'HDL cần đối chiếu giới\',`HDL ${v<40?\'dưới 40\':v<50?\'từ 40 đến dưới 50\':v<60?\'từ 50 đến dưới 60\':\'từ 60\'} mg/dL. Tham chiếu thấp: nam dưới 40, nữ dưới 50. Bạn muốn dùng tham chiếu nam hay nữ? Không suy ra nguy cơ chỉ từ HDL.`);\n   const low=v<(p.labSex===\'female\'?50:40);\n   return result(low?\'HDL thấp theo tham chiếu đã chọn\':\'HDL không nằm dưới ngưỡng thấp\',`Tham chiếu ${p.labSex===\'female\'?\'nữ: 50\':\'nam: 40\'} mg/dL. HDL cao không loại trừ nguy cơ nếu các yếu tố khác bất lợi.`,low?\'watch\':\'good\');\n  }\n  if(key===\'tg\')return result(v<150?\'Triglyceride dưới 150 mg/dL\':v<200?\'Triglyceride cận cao\':v<500?\'Triglyceride cao\':\'Triglyceride rất cao\',v>=500?\'Nên trao đổi sớm với cơ sở y tế vì liên quan nguy cơ viêm tụy; nếu đau bụng dữ dội hoặc nôn cần được khám ngay.\':\'Đọc cùng tình trạng nhịn ăn, thuốc và bảng xét nghiệm. Không tự điều chỉnh thuốc.\',v>=500?\'high\':v<150?\'good\':\'watch\',\'triglycerides\');\n  return null;\n }\n function pressureAssessment(p){\n  const s=p.sys,d=p.dia,full=Number.isFinite(s)&&Number.isFinite(d);\n  const base={source:\'pressure\',level:\'watch\'};\n  if(s===undefined&&d===undefined)return null;\n  if(p.pregnant&&(s>=160||d>=110))return {...base,source:\'pregnancyPressure\',level:\'high\',title:\'Huyết áp rất cao trong thai kỳ\',text:\'Cần liên hệ cấp cứu hoặc cơ sở sản khoa ngay để được đánh giá. Không chờ chatbot theo dõi và không tự chỉnh liều thuốc.\'};\n  if(s>=180||d>=120)return {...base,level:\'high\',title:\'Huyết áp ở mức rất cao, cần xử trí kịp thời\',text:\'Nếu đang đau ngực, khó thở, yếu liệt hoặc nói khó: cần cấp cứu ngay. Nếu không có triệu chứng nặng, đo lại sau ít nhất 1 phút và liên hệ y tế ngay nếu vẫn rất cao. Không tự tăng liều thuốc.\'};\n  if(p.pregnant||p.age<18)return {...base,source:p.pregnant?\'pregnancyPressure\':\'pressure\',title:\'Huyết áp cần đánh giá riêng\',text:\'Thai kỳ và trẻ em không dùng bảng phân loại người lớn thông thường. Trao đổi với cơ sở y tế; nếu có triệu chứng nặng cần trợ giúp ngay.\'};\n  if((s<90||d<60)&&(s>=130||d>=80))return {...base,title:\'Có thành phần cao và thành phần thấp\',text:\'Không kết luận huyết áp bình thường. Kiểm tra lại cặp số, đo lại đúng cách và trao đổi với bác sĩ, nhất là khi có triệu chứng.\'};\n  if(s<90||d<60)return {...base,source:\'lowpressure\',title:\'Có thành phần huyết áp thấp\',text:\'Cần xem triệu chứng và số đo đầy đủ. Nếu choáng/ngất hoặc mệt nhiều, tìm trợ giúp y tế; không tự bỏ thuốc.\'};\n  if(!full)return {...base,title:\'Đã ghi nhận một thành phần huyết áp\',text:`Cần thêm ${s===undefined?\'tâm thu (số trên)\':\'tâm trương (số dưới)\'}. ${s>=140||d>=90?\'Thành phần đã nhập nằm ở ngưỡng cao theo AHA; nên đo lại và trao đổi với bác sĩ.\':\'Không kết luận bình thường chỉ từ một thành phần.\'}`};\n  const stage=s>=140||d>=90?2:s>=130||d>=80?1:s>=120?\'elevated\':0;\n  return {...base,level:stage===0?\'good\':\'watch\',title:stage===0?\'Trong khoảng tham chiếu huyết áp người lớn\':stage===\'elevated\'?\'Huyết áp tăng so với mức bình thường\':`Số đo ở ngưỡng tăng huyết áp độ ${stage} (AHA)`,text:\'Đây là phân loại số đo, chưa phải chẩn đoán. Đo lại đúng cách, ghi nhật ký và trao đổi với bác sĩ nếu lặp lại mức cao. Không tự đổi liều thuốc.\'};\n }\n function cards(p,keys,h){\n  let html=\'\';const sources=[],docs=[];const pressure=keys.some(k=>k===\'sys\'||k===\'dia\');\n  if(pressure){const a=pressureAssessment(p);if(a){html+=h.assessmentHTML(a,`${p.sys??\'…\'}/${p.dia??\'…\'} mmHg`);sources.push(a.source);docs.push(knowledge[0])}}\n  for(const key of keys.filter(k=>k!==\'sys\'&&k!==\'dia\')){if(!Number.isFinite(p[key]))continue;const a=describe(key,p);if(a){html+=h.assessmentHTML(a,`${fields[key].label}: ${h.fmt(p[key],2)} ${fields[key].unit}`);sources.push(a.source);const id=key===\'bp\'?\'meanpressure\':key===\'s5\'?\'triglyceride\':key===\'s4\'?\'ratio\':[\'age\',\'sex\'].includes(key)?\'demographics\':key===\'s2\'?\'ldl\':key===\'s3\'?\'hdl\':\'lipids\';docs.push(knowledge.find(d=>d.id===id))}}\n  return {html,sources,docs:[...new Set(docs)]};\n }\n function ask(entry,h){\n  if(entry.key===\'pressure\')return h.makeReply(`<p>Mình ghi nhận <b>huyết áp ${h.fmt(entry.value)}</b>. Đây là <b>tâm thu (số trên), tâm trương (số dưới) hay huyết áp động mạch trung bình</b>? Bạn cũng có thể gửi cả cặp số kèm đơn vị, ví dụ 120/80 mmHg.</p>`,[\'pressure\'],[\'Tâm thu (mmHg)\',\'Tâm trương (mmHg)\',\'Huyết áp trung bình (mmHg)\']);\n  if(entry.key===\'cholesterol\')return h.makeReply(`<p>Cholesterol ${h.fmt(entry.value)} là <b>toàn phần, LDL hay HDL</b>? Mình cần đúng loại và đơn vị trước khi nhận xét.</p>`,[\'lipids\'],[\'Cholesterol toàn phần\',\'LDL cholesterol\',\'HDL cholesterol\']);\n  const field=fields[entry.key];return h.makeReply(`<p>Đã ghi nhận <b>${field.label}: ${h.fmt(entry.value,2)}</b>. ${entry.error?h.esc(entry.error)+\' \':\'\'}Đơn vị trên phiếu/máy là gì? Hãy sửa bằng tên chỉ số + số đo + đơn vị nếu cần.</p>`,[field.unit===\'mmHg\'?\'pressure\':\'lipids\'],field.unit===\'mmHg\'?[\'mmHg\']:[\'mg/dL\',\'mmol/L\']);\n }\n function handle(chat,n,h){\n  let entries=extract(n);const oldPending=chat.pendingClinical||[];\n  const ageMatch=n.match(/(?:toi|be|tre)\\s*(?:nam nay\\s*)?(\\d+)\\s*tuoi|tuoi\\s*[:=]?\\s*(\\d+)/);\n  const sexCode=n.match(/ma gioi tinh\\s*[:=]?\\s*(\\d+)/);\n  const labSex=/^(?:toi la |tham chieu )?(nam|nu)(?:\\s|$)/.exec(n)?.[1]||n.match(/toi la (nam|nu)\\b/)?.[1];\n  if(ageMatch)entries.push({key:\'age\',value:Number(ageMatch[1]||ageMatch[2]),unit:\'năm\'});\n  if(sexCode)entries.push({key:\'sex\',value:Number(sexCode[1]),unit:\'mã nghiên cứu\'});\n  if(labSex){chat.profile.labSex=labSex===\'nu\'?\'female\':\'male\';chat.profile.score=null;}\n  const otherNumeric=/(?:duong huyet|glucose|duong mau|hba1c|a1c|bmi).*\\d/.test(n);\n  if(!entries.length&&otherNumeric){chat.pendingClinical=[];return null}\n  if(!entries.length&&oldPending.length){\n   if(/^nhap lai/.test(n)){chat.pendingClinical=[];return h.makeReply(\'<p>Hãy nhập tên chỉ số, giá trị và đơn vị. Ví dụ “HDL 47 mg/dL” hoặc “Huyết áp 120/90 mmHg”.</p>\')}\n   const first={...oldPending[0]},bare=n.match(/^(-?\\d+(?:[.,]\\d+)?)\\s*(mm\\s*hg|mg\\s*\\/?\\s*dl|mmol\\s*\\/?\\s*l)?$/);\n   let changed=false;\n   if(first.key===\'pressure\'){const key=/tam thu|so tren/.test(n)?\'sys\':/tam truong|so duoi/.test(n)?\'dia\':/trung binh|\\bmap\\b/.test(n)?\'bp\':null;if(key){first.key=key;changed=true}}\n   if(first.key===\'cholesterol\'){const key=/toan phan/.test(n)?\'s1\':/\\bldl\\b/.test(n)?\'s2\':/\\bhdl\\b/.test(n)?\'s3\':null;if(key){first.key=key;changed=true}}\n   if(bare){first.value=toNum(bare[1]);first.unit=unitOf(bare[2])||first.unit;changed=true}\n   const unit=n.match(/mm\\s*hg|mg\\s*\\/?\\s*dl|mmol\\s*\\/?\\s*l|kpa/);if(unit){first.unit=unitOf(unit[0]);changed=true}\n   if(changed)entries=[first,...oldPending.slice(1).map(e=>([\'sys\',\'dia\'].includes(first.key)&&[\'sys\',\'dia\'].includes(e.key)&&first.unit===\'mmHg\')?{...e,unit:\'mmHg\'}:e)];\n  }\n  if(!entries.length){\n   if(labSex){if(Number.isFinite(chat.profile.s3))return reply(chat,[\'s3\'],h);return h.makeReply(\'<p>Đã ghi nhận tham chiếu \'+(labSex===\'nu\'?\'nữ\':\'nam\')+\'. Bạn gửi chỉ số kèm đơn vị nhé.</p>\')}\n   if(/cua toi|danh gia lai|ket qua nay|chi so nay/.test(n)){\n    let keys=patterns.filter(([,p])=>new RegExp(\'(?:^|\\\\b)(?:\'+p+\')(?:\\\\b|$)\').test(n)).map(([k])=>k);\n    if(keys.includes(\'pressure\'))keys.push(\'sys\',\'dia\',\'bp\');\n    if(keys.includes(\'cholesterol\'))keys.push(\'s1\');\n    keys=keys.filter(k=>Number.isFinite(chat.profile[k]));\n    if(keys.length)return reply(chat,[...new Set(keys)],h);\n   }\n   if(chat.lastClinicalKeys?.length&&/^(vay|the|chi so nay|ket qua nay|co cao|co thap|co on)/.test(n)&&/cao|thap|on|binh thuong|nguy hiem/.test(n))return reply(chat,chat.lastClinicalKeys,h);\n   if(pressureAssessment(chat.profile)?.level===\'high\'&&/lam gi|van dong|the duc/.test(n))return reply(chat,[\'sys\',\'dia\'],h);\n   return null;\n  }\n  // Chỉ số được gọi tên mới thay thế lượt hỏi cũ, không lấy nhầm đơn vị cho glucose.\n  chat.pending=null;chat.pendingClinical=[];const accepted=[],errors=[];\n  for(const entry of entries){\n   const f=fields[entry.key];\n   if(!Number.isFinite(entry.value)||entry.value<=0){errors.push(`${f.label}: số đo phải lớn hơn 0; hãy nhập lại.`);continue}\n   if([\'pressure\',\'cholesterol\'].includes(entry.key)){chat.pendingClinical.push(entry);continue}\n   if([\'s4\',\'s5\',\'age\',\'sex\'].includes(entry.key)){\n    if(entry.unit&&[\'mg/dL\',\'mmol/L\',\'mmHg\',\'kpa\',\'g/l\',\'%\'].includes(entry.unit)){errors.push(`${f.label}: không dùng đơn vị ${entry.unit}. Hãy kiểm tra đúng loại chỉ số.`);continue}\n    entry.unit=f.unit;\n   }\n   if(!entry.unit){chat.pendingClinical.push(entry);continue}\n   let value=entry.value;\n   if(entry.unit===\'mmol/L\'&&f.factor)value*=f.factor;\n   else if(entry.unit!==f.unit){chat.pendingClinical.push({...entry,error:\'Đơn vị này chưa được hỗ trợ cho chỉ số đó.\'});continue}\n   if(value<f.min||value>f.max||([\'age\',\'sex\'].includes(entry.key)&&!Number.isInteger(value))){chat.pendingClinical.push({...entry,error:\'Giá trị ngoài khoảng nhập hỗ trợ; kiểm tra đơn vị và dấu thập phân.\'});continue}\n   const counterpart=entry.key===\'sys\'?\'dia\':entry.key===\'dia\'?\'sys\':null;\n   const newOther=entries.find(e=>e.key===counterpart&&e.unit===\'mmHg\')?.value??(chat.pressureAwaiting===entry.key?chat.profile[counterpart]:undefined);\n   // Không ghép hai thành phần của hai lần đo khác nhau.\n   if(counterpart&&newOther===undefined)delete chat.profile[counterpart];\n   if(counterpart&&newOther!==undefined&&((entry.key===\'sys\'&&value<=newOther)||(entry.key===\'dia\'&&value>=newOther))){errors.push(\'Huyết áp tâm thu phải lớn hơn tâm trương; kiểm tra lại cặp số.\');continue}\n   chat.profile[entry.key]=value;clearImputation(chat,entry.key);accepted.push(entry.key);\n  }\n  if(accepted.includes(\'sys\')||accepted.includes(\'dia\'))chat.pressureAwaiting=chat.profile.sys===undefined?\'sys\':chat.profile.dia===undefined?\'dia\':null;\n  const answer=accepted.length?reply(chat,[...new Set(accepted)],h):{html:\'\',suggestions:[]};\n  if(errors.length)answer.html+=\'<p>\'+errors.map(h.esc).join(\'<br>\')+\'</p>\';\n  if(accepted.some(k=>[\'s1\',\'s2\',\'s3\'].includes(k))&&((chat.profile.s2>chat.profile.s1)||(chat.profile.s3>chat.profile.s1)))answer.html+=\'<p>LDL hoặc HDL đang lớn hơn cholesterol toàn phần. Hãy kiểm tra đơn vị và xem các số có thuộc cùng lần xét nghiệm không.</p>\';\n  if(chat.pendingClinical.length&&!otherNumeric){const next=ask(chat.pendingClinical[0],h);answer.html+=next.html;answer.suggestions=next.suggestions;answer.clinicalPrompt=true}\n  return answer;\n }\n function reply(chat,keys,h){\n  const p=chat.profile,c=cards(p,keys,h);chat.lastClinicalKeys=keys;chat.lastTopic=keys.includes(\'sys\')||keys.includes(\'dia\')?\'bloodpressure\':keys.includes(\'s3\')?\'hdl\':keys.includes(\'s2\')?\'ldl\':keys.includes(\'bp\')?\'meanpressure\':\'lipids\';\n  let suggestions=keys.includes(\'s3\')&&!p.labSex?[\'Tôi là nữ\',\'Tôi là nam\']:[\'Nhận xét tổng thể\',\'Tôi nên theo dõi thế nào?\'];\n  if(keys.includes(\'sys\')&&p.dia===undefined||keys.includes(\'dia\')&&p.sys===undefined)suggestions=[\'Nhập lại cặp huyết áp\',\'Cách đo huyết áp\'];\n  const ageNote=keys.some(k=>[\'s1\',\'s2\',\'s3\',\'tg\'].includes(k))&&p.age===undefined?\'<p class="hint">Chưa có tuổi: bảng mỡ máu dưới đây chỉ là tham chiếu cho người từ 20 tuổi. Bạn có thể bổ sung tuổi để làm rõ bối cảnh.</p>\':\'\';\n  return h.makeReply(\'<h4>Nhận xét chỉ số bạn cung cấp</h4>\'+ageNote+c.html,c.sources,suggestions,c.docs);\n }\n function contextRows(p,fmt){const rows=[];for(const [key,f] of Object.entries(fields)){if(Number.isFinite(p[key]))rows.push([f.label,`${fmt(p[key],2)} ${f.unit}`])}if(p.labSex)rows.push([\'Tham chiếu HDL\',p.labSex===\'female\'?\'Nữ\':\'Nam\']);return rows}\n return {fields,knowledge,extract,fromForm,handle,cards,pressureAssessment,describe,contextRows,ask};\n})();\nif(typeof module!==\'undefined\')module.exports=ClinicalSupport;\n\n\'use strict\';\nconst clinical=typeof module!==\'undefined\'?require(\'./clinical.js\'):ClinicalSupport;\nconst fmt=(n,d=1)=>Number(n).toLocaleString(\'vi-VN\',{maximumFractionDigits:d});\nconst esc=s=>String(s).replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\nconst norm=s=>String(s).toLowerCase().normalize(\'NFD\').replace(/[\\u0300-\\u036f]/g,\'\').replace(/đ/g,\'d\');\nconst parseNumber=s=>{s=String(s).trim();return /^-?\\d+(?:[.,]\\d+)?$/.test(s)?Number(s.replace(\',\',\'.\')):NaN};\nconst sourceLinks={\n diagnosis:{label:\'NIDDK · Xét nghiệm & chẩn đoán\',url:\'https://www.niddk.nih.gov/health-information/diabetes/overview/tests-diagnosis\'},\n low:{label:\'CDC · Xử trí hạ đường huyết\',url:\'https://www.cdc.gov/diabetes/treatment/treatment-low-blood-sugar-hypoglycemia.html\'},\n meal:{label:\'CDC · Lập kế hoạch bữa ăn\',url:\'https://www.cdc.gov/diabetes/healthy-eating/diabetes-meal-planning.html\'},\n who:{label:\'WHO · Diabetes\',url:\'https://www.who.int/news-room/fact-sheets/detail/diabetes\'},\n care:{label:\'NIDDK · Quản lý bệnh tiểu đường\',url:\'https://www.niddk.nih.gov/health-information/diabetes/overview/managing-diabetes\'},\n dataset:{label:\'scikit-learn · Dữ liệu diabetes\',url:\'https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html\'}\n ,pressure:{label:\'AHA · Hiểu số đo huyết áp\',url:\'https://www.heart.org/en/health-topics/high-blood-pressure/understanding-blood-pressure-readings\'}\n ,lowpressure:{label:\'AHA · Huyết áp thấp\',url:\'https://www.heart.org/en/health-topics/high-blood-pressure/the-facts-about-high-blood-pressure/low-blood-pressure-when-blood-pressure-is-too-low\'}\n ,pregnancyPressure:{label:\'ACOG · Huyết áp trong thai kỳ\',url:\'https://www.acog.org/womens-health/faqs/preeclampsia-and-high-blood-pressure-during-pregnancy\'}\n ,lipids:{label:\'MedlinePlus · Các chỉ số cholesterol\',url:\'https://medlineplus.gov/cholesterollevelswhatyouneedtoknow.html\'}\n ,cholesterol:{label:\'Mayo Clinic · Xét nghiệm cholesterol\',url:\'https://www.mayoclinic.org/tests-procedures/cholesterol-test/about/pac-20384601\'}\n ,triglycerides:{label:\'MedlinePlus · Triglyceride\',url:\'https://medlineplus.gov/triglycerides.html\'}\n};\nconst contextNames={fasting:\'lúc đói ≥ 8 giờ\',postmeal:\'sau bữa ăn thông thường\',ogtt:\'2 giờ sau nghiệm pháp glucose\',random:\'đo ngẫu nhiên\',unknown:\'chưa rõ thời điểm đo\'};\nfunction infer(model,x){\n switch(model.type){\n case \'pipeline\':return infer(model.estimator,x.map((v,i)=>(v-model.mean[i])/model.scale[i]));\n case \'linear\':return model.intercept+model.coef.reduce((s,c,i)=>s+c*x[i],0);\n case \'svr\':return model.intercept+model.support_vectors.reduce((s,v,j)=>s+model.dual_coef[j]*Math.exp(-model.gamma*v.reduce((d,a,i)=>d+(a-x[i])**2,0)),0);\n case \'constant\':return model.value;\n case \'persistence\':return x[0];\n case \'return\':{const z=x.slice();[1,4,5].forEach(i=>z[i]=z[i]/x[0]-1);z[0]=0;return x[0]*(1+model.shrinkage*infer(model.estimator,z));}\n case \'forest\':return model.trees.reduce((s,t)=>s+treePredict(t,x),0)/model.trees.length;\n case \'boost\':return model.base+model.rate*model.trees.reduce((s,t)=>s+treePredict(t,x),0);\n default:throw Error(\'Định dạng mô hình không được hỗ trợ\');\n }\n}\nfunction treePredict(t,x){let i=0;while(t.left[i]!==-1)i=Math.fround(x[t.feature[i]])<=t.threshold[i]?t.left[i]:t.right[i];return t.value[i]}\nfunction predictModel(m,values){return infer(m.estimator,m.features.map(k=>values[k]))}\nfunction outOfRange(m,values){const v={...values},stats=m.ood_inputs||m.inputs;if(m.ood_space===\'relative\'){for(const k of [\'Close_Lag1\',\'MA20\',\'MA60\'])v[k]=v[k]/values.Close-1;v.Close=0;}return m.features.filter(k=>v[k]<stats[k].min||v[k]>stats[k].max)}\nfunction glucoseAssessment(g,context=\'unknown\',special=false){\n if(g<55)return {level:\'high\',title:\'Đường huyết rất thấp\',text:\'Cần người hỗ trợ và trợ giúp y tế ngay. Nếu lú lẫn, bất tỉnh, co giật hoặc không nuốt được, không cho ăn uống.\',source:\'low\',low:true};\n if(g<70)return {level:\'high\',title:\'Đường huyết thấp\',text:\'Nếu là người lớn, còn tỉnh và nuốt được: dùng 15 g carbohydrate hấp thu nhanh, chờ 15 phút rồi đo lại; lặp lại nếu vẫn dưới 70 mg/dL. Nếu không cải thiện hoặc có triệu chứng nặng, cần trợ giúp y tế.\',source:\'low\',low:true};\n if(special)return {level:\'watch\',title:\'Cần ngưỡng đánh giá riêng\',text:\'Thai kỳ và trẻ em cần đánh giá theo bối cảnh riêng. Hãy trao đổi với cơ sở y tế; trợ lý không áp dụng tự động bảng phân loại người lớn cho trường hợp này.\',source:\'diagnosis\'};\n if(context===\'unknown\')return {level:\'watch\',title:\'Cần biết thời điểm đo\',text:\'Bạn đo lúc đói, sau ăn, sau nghiệm pháp glucose hay ngẫu nhiên? Một chỉ số đơn lẻ chưa đủ kết luận.\',source:\'diagnosis\'};\n if(context===\'postmeal\')return {level:\'watch\',title:\'Số đo sau ăn cần bối cảnh\',text:\'Bữa ăn thông thường không tương đương nghiệm pháp 75 g glucose. Cần biết đo sau ăn bao lâu, đã có chẩn đoán hay chưa và mục tiêu bác sĩ đặt cho bạn.\',source:\'diagnosis\'};\n if(context===\'random\')return g>=200?{level:\'high\',title:\'Đường huyết ngẫu nhiên cao\',text:\'Từ 200 mg/dL có ý nghĩa chẩn đoán khi kèm triệu chứng điển hình; cần cơ sở y tế xác nhận. Nếu nôn, khó thở hoặc lú lẫn, cần trợ giúp y tế ngay.\',source:\'diagnosis\'}:{level:\'watch\',title:\'Chưa đủ dữ kiện phân loại\',text:\'Số đo ngẫu nhiên dưới 200 mg/dL không loại trừ tiểu đường. Xét nghiệm lúc đói hoặc HbA1c giúp đánh giá rõ hơn.\',source:\'diagnosis\'};\n const normal=context===\'fasting\'?100:140, high=context===\'fasting\'?126:200;\n if(g<normal)return {level:\'good\',title:\'Dưới ngưỡng tiền đái tháo đường\',text:`Số đo ${contextNames[context]} nằm trong khoảng tham chiếu bình thường. Vẫn cần xem triệu chứng và các xét nghiệm khác.`,source:\'diagnosis\'};\n if(g<high)return {level:\'watch\',title:\'Trong khoảng tiền đái tháo đường\',text:`Số đo ${contextNames[context]} nằm từ ${normal} đến dưới ${high} mg/dL. Nên trao đổi về xét nghiệm xác nhận và kế hoạch theo dõi.`,source:\'diagnosis\'};\n return {level:\'high\',title:\'Đạt ngưỡng xét nghiệm đái tháo đường\',text:`Số đo ${contextNames[context]} từ ${high} mg/dL. Cần xác nhận tại cơ sở y tế; một số đo tại nhà không tự xác lập chẩn đoán.`,source:\'diagnosis\'};\n}\nfunction a1cAssessment(a,special=false){if(special)return {level:\'watch\',title:\'HbA1c cần đánh giá cá nhân\',text:\'Trong thai kỳ hoặc một số bệnh máu, HbA1c có thể không phản ánh chính xác đường huyết. Hãy trao đổi với bác sĩ.\',source:\'diagnosis\'};return a<5.7?{level:\'good\',title:\'HbA1c dưới 5,7%\',text:\'Trong khoảng tham chiếu bình thường; vẫn cần đối chiếu triệu chứng và bối cảnh.\',source:\'diagnosis\'}:a<6.5?{level:\'watch\',title:\'HbA1c trong khoảng tiền đái tháo đường\',text:\'Từ 5,7% đến dưới 6,5%. Nên theo dõi và trao đổi kế hoạch giảm nguy cơ.\',source:\'diagnosis\'}:{level:\'high\',title:\'HbA1c đạt ngưỡng xét nghiệm đái tháo đường\',text:\'Từ 6,5%. Cần xét nghiệm chuẩn và cơ sở y tế xác nhận; ngưỡng chẩn đoán khác với mục tiêu điều trị của người đã mắc bệnh.\',source:\'diagnosis\'}}\nfunction bmiAssessment(b){return b<18.5?{level:\'watch\',title:\'BMI thấp\',text:\'Cần xem khẩu phần và cân nặng theo thời gian.\'}:b<25?{level:\'good\',title:\'BMI trong khoảng tham chiếu phổ biến\',text:\'Duy trì thói quen ăn uống và hoạt động phù hợp.\'}:b<30?{level:\'watch\',title:\'BMI cao hơn khoảng tham chiếu\',text:\'Có thể trao đổi về mục tiêu cân nặng; BMI chưa phản ánh toàn bộ thể trạng.\'}:{level:\'watch\',title:\'BMI ở mức cao\',text:\'Nên có kế hoạch kiểm soát cân nặng phù hợp với nhân viên y tế.\'}}\nfunction assessmentHTML(item,subtitle=\'\'){return `<div class="assessment-row"><span class="signal ${item.level}"></span><div><strong>${esc(item.title)}</strong>${subtitle?`<small>${esc(subtitle)}</small>`:\'\'}<small>${esc(item.text)}</small></div></div>`}\nfunction sourcesHTML(keys){const valid=[...new Set(keys)].filter(k=>sourceLinks[k]);return valid.length?`<div class="sources">Tham khảo: ${valid.map(k=>`<a href="${sourceLinks[k].url}" target="_blank" rel="noopener">${sourceLinks[k].label} ↗</a>`).join(\' \')}</div>`:\'\'}\nconst KB=[\n ...clinical.knowledge,\n {id:\'plate\',title:\'Một bữa ăn cân bằng\',terms:[\'bữa ăn\',\'ăn gì\',\'thực đơn\',\'ăn sáng\',\'ăn trưa\',\'ăn tối\',\'rau\',\'đạm\',\'đĩa ăn\'],text:\'Gợi ý minh họa: nửa đĩa rau luộc hoặc salad không ngọt; một phần tư cá, thịt nạc hoặc đậu phụ; phần tư còn lại là cơm hoặc thực phẩm chứa tinh bột. Chọn nước lọc. Điều chỉnh khẩu phần theo nhu cầu, thuốc và mục tiêu cá nhân.\',source:\'meal\',triples:[[\'Bữa ăn\',\'chia theo\',\'phương pháp chiếc đĩa\'],[\'Chiếc đĩa\',\'ưu tiên\',\'rau không tinh bột\']]},\n {id:\'carb\',title:\'Có cần bỏ hoàn toàn tinh bột?\',terms:[\'tinh bột\',\'carb\',\'carbohydrate\',\'cơm\',\'gạo\',\'bún\',\'phở\',\'bánh mì\',\'kiêng\'],text:\'Không cần tự bỏ toàn bộ tinh bột. Chú ý khẩu phần, phân bố bữa ăn đều và kết hợp đạm, rau. Ưu tiên thực phẩm ít chế biến. Lượng carbohydrate phù hợp cần xem cùng thuốc, mức hoạt động và chỉ số sau bữa ăn.\',source:\'meal\',triples:[[\'Carbohydrate\',\'ảnh hưởng\',\'đường huyết\'],[\'Khẩu phần\',\'cần điều chỉnh theo\',\'nhu cầu cá nhân\']]},\n {id:\'fruit\',title:\'Trái cây và đồ uống\',terms:[\'trái cây\',\'hoa quả\',\'nước ép\',\'sữa\',\'trà sữa\',\'nước ngọt\',\'uống gì\',\'chuối\'],text:\'Ưu tiên trái cây nguyên quả theo khẩu phần, nước lọc hoặc đồ uống không thêm đường. Nước ép và đồ uống có đường có thể làm đường huyết tăng nhanh. Không có một loại trái cây phù hợp vô hạn cho mọi người.\',source:\'meal\',triples:[[\'Nước ép\',\'có thể làm tăng nhanh\',\'đường huyết\']]},\n {id:\'activity\',title:\'Vận động theo khả năng\',terms:[\'vận động\',\'tập\',\'thể dục\',\'đi bộ\',\'thể thao\',\'bao nhiêu phút\'],text:\'Mục tiêu tham khảo cho người lớn là ít nhất 150 phút vận động mức vừa mỗi tuần. Có thể bắt đầu bằng các buổi ngắn rồi tăng dần theo khả năng. Nếu có biến chứng hoặc dùng thuốc dễ gây hạ đường huyết, hãy trao đổi kế hoạch tập với nhân viên y tế.\',source:\'who\',triples:[[\'Vận động\',\'hỗ trợ\',\'quản lý đường huyết\'],[\'Kế hoạch tập\',\'phụ thuộc\',\'thể trạng\']]},\n {id:\'a1c\',title:\'Hiểu chỉ số HbA1c\',terms:[\'hba1c\',\'a1c\',\'đường trung bình\',\'ba tháng\'],text:\'HbA1c phản ánh mức đường huyết trung bình trong khoảng ba tháng. Ngưỡng xét nghiệm chẩn đoán không giống mục tiêu điều trị đã cá nhân hóa. Kết quả có thể bị ảnh hưởng bởi thai kỳ và một số bệnh máu. Nếu có con số cụ thể, bạn có thể nhập kèm dấu %. \',source:\'diagnosis\',triples:[[\'HbA1c\',\'phản ánh\',\'đường huyết trung bình\'],[\'Thai kỳ / bệnh máu\',\'có thể ảnh hưởng\',\'HbA1c\']]},\n {id:\'testing\',title:\'Đọc kết quả xét nghiệm\',terms:[\'đường huyết\',\'glucose\',\'lúc đói\',\'sau ăn\',\'xét nghiệm\',\'nguy cơ\',\'chẩn đoán\',\'có bệnh\'],text:\'Bạn cho biết giá trị, đơn vị mg/dL hoặc mmol/L và thời điểm đo. Đo tại nhà, đo lúc đói, sau ăn và nghiệm pháp glucose có cách diễn giải khác nhau. Chẩn đoán thường cần xét nghiệm xác nhận.\',source:\'diagnosis\',triples:[[\'Glucose\',\'cần có\',\'đơn vị và thời điểm đo\']]},\n {id:\'low\',title:\'Khi đường huyết thấp\',terms:[\'hạ đường huyết\',\'run tay\',\'vã mồ hôi\',\'đói cồn cào\',\'tụt đường\'],text:\'Dưới 70 mg/dL được coi là thấp. Người lớn còn tỉnh và nuốt được có thể áp dụng quy tắc 15-15: 15 g carbohydrate nhanh, đo lại sau 15 phút và lặp lại nếu còn thấp. Không cho ăn uống khi bất tỉnh hoặc không nuốt được; cần trợ giúp y tế khẩn cấp.\',source:\'low\',triples:[[\'Glucose dưới 70\',\'gợi ý\',\'hạ đường huyết\'],[\'Không nuốt được\',\'không được\',\'cho ăn uống\']]},\n {id:\'urgent\',title:\'Dấu hiệu cần trợ giúp ngay\',terms:[\'cấp cứu\',\'khẩn cấp\',\'khó thở\',\'bất tỉnh\',\'co giật\',\'lú lẫn\',\'nguy hiểm\'],text:\'Nếu đang bất tỉnh, co giật, lú lẫn rõ, khó thở hoặc không nuốt được, cần người bên cạnh gọi cấp cứu địa phương hoặc đưa đến cơ sở y tế ngay. Không tự lái xe và không cố cho ăn uống khi khả năng nuốt bị ảnh hưởng.\',source:\'low\',triples:[[\'Rối loạn ý thức\',\'cần\',\'trợ giúp y tế khẩn cấp\']]},\n {id:\'medicine\',title:\'Về thuốc và liều dùng\',terms:[\'thuốc\',\'metformin\',\'insulin\',\'liều\',\'uống mấy viên\',\'ngừng thuốc\',\'tăng liều\',\'giảm liều\'],text:\'Loại thuốc và liều phụ thuộc chẩn đoán, chức năng thận, thuốc khác và nguy cơ hạ đường huyết. Mình không thể quyết định liều phù hợp cho bạn. Hãy ghi tên thuốc, liều hiện tại, thời điểm uống và số đo để trao đổi với bác sĩ; không tự tăng, giảm hoặc ngừng thuốc.\',source:\'care\',triples:[[\'Liều thuốc\',\'cần được chỉ định bởi\',\'nhân viên y tế\']]},\n {id:\'monitor\',title:\'Theo dõi để trao đổi khi tái khám\',terms:[\'theo dõi\',\'tái khám\',\'đo mấy lần\',\'nhật ký\',\'mục tiêu\',\'đã mắc\',\'đã bị\'],text:\'Ghi thời điểm đo, kết quả, bữa ăn, vận động, thuốc và triệu chứng. Tần suất đo và mục tiêu cần cá nhân hóa, đặc biệt nếu đang dùng insulin. Mục tiêu điều trị của người đã mắc tiểu đường khác bảng ngưỡng dùng để chẩn đoán ban đầu.\',source:\'care\',triples:[[\'Nhật ký\',\'hỗ trợ\',\'tái khám\'],[\'Mục tiêu điều trị\',\'phụ thuộc\',\'từng người\']]},\n {id:\'symptoms\',title:\'Triệu chứng và thăm khám\',terms:[\'triệu chứng\',\'khát\',\'tiểu nhiều\',\'sụt cân\',\'mờ mắt\',\'mệt mỏi\',\'dấu hiệu sớm\'],text:\'Khát nhiều, tiểu nhiều, mệt, mờ mắt hoặc sụt cân không rõ nguyên nhân có thể xuất hiện khi đường huyết cao, nhưng cũng có thể do nguyên nhân khác. Nên khám và làm xét nghiệm; không thể xác nhận tiểu đường chỉ qua triệu chứng.\',source:\'who\',triples:[[\'Triệu chứng\',\'cần được xác nhận bằng\',\'khám và xét nghiệm\']]},\n {id:\'model\',title:\'Điểm mô hình có ý nghĩa gì?\',terms:[\'mô hình\',\'điểm tiến triển\',\'dự đoán\',\'r2\',\'sai số\',\'độ chính xác\',\'phần trăm\'],text:\'Mô hình học trên 442 người đã mắc tiểu đường để ước lượng điểm tiến triển sau một năm. Điểm này không phải glucose, HbA1c hay phần trăm nguy cơ mắc bệnh. Khoảng dự báo và sai số ngoài mẫu giúp nhìn thấy mức bất định.\',source:\'dataset\',triples:[[\'Mô hình diabetes\',\'dự báo\',\'điểm tiến triển nghiên cứu\'],[\'Điểm tiến triển\',\'không phải\',\'xác suất mắc bệnh\']]},\n];\nconst stop=new Set([\'toi\',\'ban\',\'la\',\'va\',\'co\',\'khong\',\'cho\',\'cua\',\'nhu\',\'thi\',\'nay\',\'do\',\'gi\',\'mot\',\'nhung\',\'hay\',\'voi\',\'duoc\',\'o\',\'ve\']);\nconst tokens=s=>norm(s).replace(/[^a-z0-9 ]/g,\' \').split(/\\s+/).filter(x=>x.length>1&&!stop.has(x));\nconst indexed=KB.map(d=>({...d,tokens:tokens(d.title+\' \'+d.terms.join(\' \')+\' \'+d.text)}));\nfunction retrieve(question){const q=[...new Set(tokens(question))],n=norm(question),avg=indexed.reduce((s,d)=>s+d.tokens.length,0)/indexed.length;return indexed.map(d=>{let score=0;for(const t of q){const tf=d.tokens.filter(v=>v===t).length,df=indexed.filter(k=>k.tokens.includes(t)).length;if(tf)score+=Math.log(1+(indexed.length-df+.5)/(df+.5))*(tf*2.2)/(tf+1.2*(.25+.75*d.tokens.length/avg))}const matches=d.terms.filter(k=>n.includes(norm(k)));score+=matches.reduce((s,k)=>s+1.5+norm(k).split(\' \').length*.5,0);return {...d,score,matches:matches.length}}).sort((a,b)=>b.score-a.score)}\nfunction graphHTML(docs){return `<details><summary>Đường bằng chứng Knowledge Graph</summary>${docs.flatMap(d=>d.triples).map(t=>`<p>${t.map(esc).join(\' → \')}</p>`).join(\'\')}</details>`}\nfunction makeReply(html,keys=[],suggestions=[],docs=[]){return {html:html+sourcesHTML(keys)+(docs.length?graphHTML(docs):\'\'),suggestions}}\nfunction extractContext(n){if(/nghiem phap|ogtt|75\\s*g/.test(n))return \'ogtt\';if(/sau an|sau bua|an xong/.test(n))return \'postmeal\';if(/luc doi|nhin an|chua an/.test(n))return \'fasting\';if(/ngau nhien/.test(n))return \'random\';return null}\nfunction extractUnit(n){if(/mmol\\s*\\/?\\s*l/.test(n))return \'mmol/L\';if(/mg\\s*\\/?\\s*dl/.test(n))return \'mg/dL\';return null}\nfunction validGlucose(v,unit){return Number.isFinite(v)&&v>0&&(unit===\'mmol/L\'?v<=55.5:v<=1000)}\nclass HealthChat{\n constructor(){this.reset()}\n reset(){this.profile={};this.pending=null;this.pendingClinical=[];this.pressureAwaiting=null;this.lastClinicalKeys=[];this.lastTopic=null;this.shared=false}\n share(profile){this.reset();this.profile=clinical.fromForm(profile);this.shared=true}\n helpers(){return {fmt,esc,makeReply,assessmentHTML}}\n respond(question){\n  this.extraReply=null;\n  let reply=this.respondCore(question);\n  if(this.extraReply)reply={html:reply.html+this.extraReply.html,suggestions:[...new Set([...reply.suggestions,...this.extraReply.suggestions])].slice(0,5)};\n  if(!this.pending&&this.pendingClinical.length&&!reply.clinicalPrompt){const next=clinical.ask(this.pendingClinical[0],this.helpers());reply={html:reply.html+next.html,suggestions:next.suggestions}}\n  return reply;\n }\n respondCore(question){\n  const n=norm(question).trim();\n  const educational=/khi nao|dau hieu|la gi|neu bi|neu co|trieu chung nao/.test(n);\n  const emergency=[\'bat tinh\',\'co giat\',\'kho tho\',\'khong the nuot\',\'khong nuot duoc\',\'lu lan\',\'dau nguc\',\'yeu liet\',\'noi kho\'].some(term=>{const i=n.indexOf(term);if(i<0)return false;return !/khong(?: bi| he| con)?\\s*$/.test(n.slice(Math.max(0,i-15),i))});\n  if(emergency&&!educational){this.pending=null;this.pendingClinical=[];return makeReply(\'<h4>Cần trợ giúp y tế ngay</h4><p>Hãy nhờ người bên cạnh liên hệ cấp cứu địa phương hoặc đưa đến cơ sở y tế ngay. Không tự lái xe; không cho ăn uống khi bất tỉnh hoặc không nuốt được.</p>\',[\'pressure\',\'low\'])}\n  if(/mang thai|thai ky|co bau/.test(n)&&!/khong (mang thai|co bau)/.test(n)){this.profile.pregnant=true;this.profile.score=null;}\n  if(/khong (mang thai|co bau)/.test(n))this.profile.pregnant=false;\n  const glucoseFollowup=this.pending&&!clinical.extract(n).length&&/^(?:\\d|mg\\s*\\/?\\s*dl|mmol\\s*\\/?\\s*l|doi sang|luc doi|sau an|sau bua|2 gio sau|do ngau nhien)/.test(n);\n  const clinicalReply=glucoseFollowup?null:clinical.handle(this,n,this.helpers());\n  if(clinicalReply){if(/(?:duong huyet|glucose|duong mau|hba1c|a1c|bmi).*\\d/.test(n))this.extraReply=clinicalReply;else return clinicalReply}\n  if(/nhan xet tong the|danh gia tong the|ket qua cua toi|chi so cua toi|danh gia lai/.test(n))return this.summaryReply();\n  const gm=n.match(/(?:duong huyet|glucose|duong mau)(?:\\s+(?:cua toi|cua em|la|bang|do duoc|luc doi|sau an))*\\s*[:=]?\\s*(\\d+(?:[.,]\\d+)?)/);\n  const am=n.match(/(?:hba1c|a1c)\\s*(?:la|bang|cua toi)?\\s*[:=]?\\s*(\\d+(?:[.,]\\d+)?)/);\n  const bm=n.match(/bmi\\s*(?:la|bang)?\\s*[:=]?\\s*(\\d+(?:[.,]\\d+)?)/);\n  // Đơn vị phải thuộc chính số glucose, không lấy mmol/L của HDL trong cùng tin nhắn.\n  const localUnit=gm?n.slice(gm.index+gm[0].length).match(/^\\s*(mmol\\s*\\/?\\s*l|mg\\s*\\/?\\s*dl)/):null;\n  const ctx=extractContext(n),unit=gm?extractUnit(localUnit?.[0]||\'\'):extractUnit(n),bare=n.match(/^(\\d+(?:[.,]\\d+)?)\\s*(?:mmol\\/?l|mg\\/?dl)?$/);\n  if(gm||(bare&&unit&&!this.pendingClinical.length)||(this.pending&&(ctx||unit||bare))){\n   if(gm)this.pending={value:parseNumber(gm[1]),unit:unit,context:ctx};\n   else if(bare&&unit&&!this.pending)this.pending={value:parseNumber(bare[1]),unit,context:ctx};\n   else {if(bare)this.pending.value=parseNumber(bare[1]);if(unit)this.pending.unit=unit;if(ctx)this.pending.context=ctx}\n   if(am){const av=parseNumber(am[1]);if(av>=3&&av<=20)this.profile.a1c=av;}\n   const p=this.pending;\n   if(!Number.isFinite(p.value)||p.value<=0){this.pending=null;return makeReply(\'<p>Giá trị đường huyết phải lớn hơn 0. Bạn nhập lại số đo và đơn vị giúp mình.</p>\')}\n   if(!p.unit)return makeReply(`<p>Mình đã ghi nhận <b>${fmt(p.value,2)}</b>${p.context?\' · \'+contextNames[p.context]:\'\'}. Đơn vị trên máy hoặc phiếu xét nghiệm là <b>mg/dL hay mmol/L</b>?</p>`,[],[\'mg/dL\',\'mmol/L\']);\n   if(!validGlucose(p.value,p.unit)){\n    p.needsCorrection=true;\n    return makeReply(`<h4>Cần kiểm tra lại số đo và đơn vị</h4><p>Bạn vừa nhập <b>${fmt(p.value,2)} ${p.unit}</b>${p.unit===\'mmol/L\'?`, tương đương khoảng <b>${fmt(p.value*18.0182,0)} mg/dL</b>`:\'\'}. Đây là giá trị cực cao, ngoài khoảng chatbot có thể phân tích; có thể nhầm đơn vị hoặc dấu thập phân.</p><p>Mình vẫn giữ số đo để bạn sửa tiếp. Hãy đối chiếu màn hình máy/phiếu xét nghiệm rồi gửi đơn vị đúng hoặc nhập lại cả số đo; mình không tự đổi 150 thành 15. Nếu số đo được xác nhận đúng, máy hiện HI, hoặc bạn đang nôn, khó thở, lú lẫn, hãy tìm trợ giúp y tế ngay.</p>`,[\'diagnosis\'],[\'Đổi sang mg/dL\',\'Nhập lại số đo\']);\n   }\n   const g=p.unit===\'mmol/L\'?p.value*18.0182:p.value;\n   if(!p.context&&g>=70)return makeReply(`<p><b>${fmt(p.value,2)} ${p.unit}</b> đã được ghi nhận. Bạn đo vào thời điểm nào?</p>`,[\'diagnosis\'],[\'Lúc đói ít nhất 8 giờ\',\'Sau bữa ăn\',\'2 giờ sau nghiệm pháp glucose\',\'Đo ngẫu nhiên\']);\n   this.profile.glucose=g;this.profile.context=p.context||\'unknown\';this.profile.originalUnit=p.unit;this.profile.score=null;this.pending=null;this.lastTopic=\'testing\';\n   let extra=\'\';if(am&&this.profile.a1c>=3&&this.profile.a1c<=20)extra=assessmentHTML(a1cAssessment(this.profile.a1c,this.profile.pregnant),`HbA1c ${fmt(this.profile.a1c)}%`);\n   return this.glucoseReply(extra);\n  }\n  if(this.pending&&/nhap lai so do/.test(n)){this.pending=null;return makeReply(\'<p>Bạn gửi lại số đo kèm đơn vị và thời điểm đo nhé. Ví dụ: “Đường huyết 150 mg/dL lúc đói”.</p>\')}\n  if(this.pending?.needsCorrection&&/xac nhan|dung roi|dung la|chinh xac/.test(n))return makeReply(\'<h4>Cần trợ giúp y tế ngay</h4><p>Nếu số đo rất cao này đã được xác nhận đúng, bạn cần được cơ sở y tế đánh giá ngay. Không tự chỉnh liều thuốc theo chatbot. Nhờ người hỗ trợ; không tự lái xe nếu đang mệt hoặc có triệu chứng.</p>\',[\'diagnosis\']);\n  if(ctx&&this.profile.glucose){this.profile.context=ctx;return this.glucoseReply()}\n  if(am){const a=parseNumber(am[1]);if(a<3||a>20)return makeReply(\'<p>Hãy kiểm tra lại HbA1c; nhập số theo %, ví dụ 6,2%. Khoảng nhập hỗ trợ là 3–20%.</p>\');this.profile.a1c=a;this.lastTopic=\'a1c\';return makeReply(`<h4>HbA1c ${fmt(a)}%</h4>${assessmentHTML(a1cAssessment(a,this.profile.pregnant))}<p>Đây là ngưỡng xét nghiệm, không phải mục tiêu điều trị dành riêng cho bạn.</p>`,[\'diagnosis\'],[\'HbA1c là gì?\',\'Tôi nên theo dõi thế nào?\'])}\n  if(bm){const b=parseNumber(bm[1]);if(b<10||b>80)return makeReply(\'<p>Hãy kiểm tra lại BMI, đơn vị kg/m². Khoảng nhập hỗ trợ là 10–80.</p>\');this.profile.bmi=b;this.profile.score=null;return makeReply(`<h4>BMI ${fmt(b)} kg/m²</h4>${this.profile.pregnant||this.profile.age<18?\'<p>Cần đánh giá BMI theo tuổi hoặc thai kỳ; không dùng ngưỡng người lớn thông thường.</p>\':assessmentHTML(bmiAssessment(b))}`,[],[\'Gợi ý một bữa ăn\',\'Tôi nên vận động bao nhiêu?\'])}\n  if(/^(chao|xin chao|hello|hi|cam on|cảm ơn)/.test(n))return makeReply(\'<p>Chào bạn! Mình có thể giải thích đường huyết, HbA1c và gợi ý ăn uống, vận động. Bạn muốn bắt đầu từ chỉ số hay một câu hỏi về sinh hoạt?</p>\',[],[\'Đường huyết 150 mg/dL lúc đói\',\'Gợi ý một bữa ăn\']);\n  if(/^(vay |the |con |nen lam gi|toi nen lam gi)/.test(n)&&this.profile.glucose&&/lam gi|an gi|the nao|on khong/.test(n)&&this.profile.glucose<70)return this.glucoseReply();\n  if(/lam gi|an gi|the nao/.test(n)&&this.profile.glucose<70)return this.glucoseReply();\n  let query=question;\n  if(/^(noi ro hon|giai thich them|chi tiet hon)$/.test(n)&&this.lastTopic){const d=KB.find(d=>d.id===this.lastTopic);query=d?.terms.join(\' \')||query}\n  const ranked=retrieve(query),top=ranked[0];\n  if(!top||top.score<3||!top.matches)return makeReply(\'<p>Mình chưa tìm được bằng chứng đủ phù hợp. Bạn có thể hỏi về huyết áp, cholesterol, HDL, LDL, triglyceride, đường huyết, HbA1c, BMI, ăn uống hoặc vận động. Với FPT/vàng, hãy dùng khu Dự đoán phía trên.</p>\',[],[\'Huyết áp 90 có cao không?\',\'HDL là gì?\']);\n  this.lastTopic=top.id;\n  const docs=[top];\n  if(ranked[1].matches&&ranked[1].score>=Math.max(5,top.score*.65)&&ranked[1].id!==top.id)docs.push(ranked[1]);\n  let html=docs.map(d=>`<h4>${esc(d.title)}</h4><p>${esc(d.text)}</p>`).join(\'\');\n  if([\'plate\',\'carb\',\'activity\'].includes(top.id)&&this.profile.glucose){html+=`<p class="hint">Mình đang dùng bối cảnh bạn cung cấp: glucose ${fmt(this.profile.glucose)} mg/dL, ${contextNames[this.profile.context]||contextNames.unknown}. ${this.profile.pregnant?\'Bạn đang mang thai, cần kế hoạch cá nhân với nhân viên y tế.\':\'Gợi ý trên là điểm bắt đầu; khẩu phần và mục tiêu cần cá nhân hóa.\'}</p>`}\n  return makeReply(html,docs.map(d=>d.source),[\'Tôi nên theo dõi thế nào?\',\'Gợi ý một bữa ăn\'],docs);\n }\n summaryReply(){\n  const p=this.profile,keys=Object.keys(clinical.fields).filter(k=>Number.isFinite(p[k])),c=clinical.cards(p,keys,this.helpers());\n  let html=c.html,refs=c.sources;\n  if(Number.isFinite(p.bmi))html+=p.pregnant||p.age<18?\'<p>BMI cần đánh giá theo tuổi hoặc thai kỳ.</p>\':assessmentHTML(bmiAssessment(p.bmi),`BMI ${fmt(p.bmi)} kg/m²`);\n  if(Number.isFinite(p.a1c)){html+=assessmentHTML(a1cAssessment(p.a1c,p.pregnant||p.age<18),`HbA1c ${fmt(p.a1c)}%`);refs.push(\'diagnosis\')}\n  if(Number.isFinite(p.glucose)){const g=this.glucoseReply();html=g.html+html;}\n  if(!html)return makeReply(\'<p>Mình chưa nhận thông số. Gửi số đo kèm đơn vị hoặc chọn “Dùng kết quả đã phân tích”.</p>\');\n  if(p.missing?.length)html+=\'<p class="hint">Chưa cung cấp: \'+p.missing.map(k=>esc(clinical.fields[k]?.label||k)).join(\', \')+\'. Giá trị trung vị dùng cho mô hình không được coi là xét nghiệm của bạn.</p>\';\n  return makeReply(\'<h4>Tổng hợp hồ sơ đã chia sẻ</h4>\'+html,refs,[\'Huyết áp của tôi thế nào?\',\'HDL của tôi thế nào?\',\'Gợi ý một bữa ăn\'],c.docs);\n }\n glucoseReply(extra=\'\'){\n  const p=this.profile,g=p.glucose,item=glucoseAssessment(g,p.context,p.pregnant||p.age<18);\n  let html=`<h4>Đường huyết ${fmt(g)} mg/dL <span class="small">≈ ${fmt(g/18.0182,2)} mmol/L</span></h4>${assessmentHTML(item,contextNames[p.context]||contextNames.unknown)}${extra}`;\n  if(!item.low)html+=\'<p><b>Bước tiếp theo:</b> ghi lại thời điểm đo, triệu chứng và bữa ăn gần nhất; nếu chỉ số bất thường, trao đổi với cơ sở y tế về xét nghiệm xác nhận.</p>\';\n  return makeReply(html,[item.source],item.low?[\'Khi nào cần cấp cứu?\']:[\'Gợi ý một bữa ăn\',\'Tôi nên theo dõi thế nào?\']);\n }\n}\nif(typeof module!==\'undefined\')module.exports={infer,predictModel,outOfRange,HealthChat,glucoseAssessment,a1cAssessment,parseNumber,esc,retrieve,KB,sourceLinks};\n</script><script>const $=id=>document.getElementById(id);\nconst labels={age:\'Tuổi\',sex:\'Mã giới tính trong dữ liệu\',bmi:\'BMI\',bp:\'Huyết áp động mạch trung bình\',s1:\'Cholesterol toàn phần\',s2:\'LDL cholesterol\',s3:\'HDL cholesterol\',s4:\'Tỷ lệ cholesterol / HDL\',s5:\'Log triglyceride\',s6:\'Glucose\',Close:\'Giá đóng cửa\',Close_Lag1:\'Giá phiên liền trước\',MA20:\'Trung bình 20 phiên\',MA60:\'Trung bình 60 phiên\',RSI14:\'RSI 14\',Volatility20:\'Biến động năm hóa\',Return_1D:\'Lợi suất hiện tại\',Return_Lag1:\'Lợi suất phiên trước\',Volume_Change:\'Thay đổi khối lượng\',Range_Pct:\'Biên độ cao – thấp\',Open_Close_Pct:\'Thay đổi mở – đóng\'};\nconst bounds={age:[18,100],bmi:[10,80],bp:[40,200],s1:[50,600],s2:[10,500],s3:[5,160],s4:[.2,20],s5:[1,9],s6:[20,1000]};\nconst healthFields=[\'age\',\'bmi\',\'bp\',\'s1\',\'s3\',\'s6\'];\nconst extraFields=[\'sex\',\'s2\',\'s4\',\'s5\'];\nconst healthUnits={age:\'năm · mô hình người lớn\',bmi:\'kg/m²\',bp:\'mmHg · không phải huyết áp tâm thu\',s1:\'mg/dL\',s2:\'mg/dL · tùy chọn\',s3:\'mg/dL\',s4:\'tỷ lệ · tùy chọn\',s5:\'ln(mg/dL) · tùy chọn\',s6:\'mg/dL\'};\nlabels.s5=\'s5 / ltg (nghiên cứu)\';\nhealthUnits.s5=\'Đặc trưng gốc, không phải triglyceride xét nghiệm · tùy chọn\';\nlet lastHealth=null;\nconst chat=new HealthChat();\nfunction field(k,value,prefix,unit,placeholder=\'\'){return `<label for="${prefix}-${k}">${labels[k]||k}<span class="unit">${unit||\'\'}</span><input id="${prefix}-${k}" inputmode="decimal" autocomplete="off" value="${value}" placeholder="${placeholder}"></label>`}\nfunction resetHealth(){const m=CONFIG.models.diabetes;healthFields.forEach(k=>$(\'d-\'+k).value=Number(m.inputs[k].value.toFixed(2)));extraFields.forEach(k=>$(\'d-\'+k).value=\'\');$(\'glucose-context\').value=\'unknown\';$(\'hba1c\').value=\'\';$(\'pregnant\').checked=false;$(\'health-errors\').hidden=true;invalidateHealth()}\nfunction invalidateHealth(){if(!lastHealth)return;lastHealth=null;$(\'share-health\').disabled=true;$(\'health-result\').innerHTML=\'<span class="kicker">THÔNG SỐ ĐÃ THAY ĐỔI</span><h3>Phân tích lại để cập nhật.</h3><p>Kết quả cũ đã được thu hồi để tránh nhầm với dữ liệu vừa sửa.</p>\'}\nfunction svgSeries(series,{label=\'Biểu đồ\',dates=[],suffix=\'\'}={}){\n const all=series.flatMap(s=>s.values).filter(Number.isFinite);if(all.length<2)return \'\';\n const width=520,height=195,left=58,right=15,top=16,bottom=32;\n let min=Math.min(...all),max=Math.max(...all);const pad=(max-min)*.12||1;min-=pad;max+=pad;\n const n=Math.max(...series.map(s=>s.values.length));const px=i=>left+i*(width-left-right)/Math.max(1,n-1),py=v=>top+(max-v)*(height-top-bottom)/(max-min);\n let html=`<svg class="chart" viewBox="0 0 ${width} ${height}" role="img" aria-label="${esc(label)}"><title>${esc(label)}</title>`;\n for(let i=0;i<4;i++){const v=min+(max-min)*i/3,y=py(v);html+=`<line x1="${left}" x2="${width-right}" y1="${y}" y2="${y}" stroke="#e7edeb"/><text x="${left-9}" y="${y+4}" text-anchor="end" fill="#6d8184" font-size="10">${esc(fmt(v,0))}</text>`}\n series.forEach(s=>{html+=`<polyline points="${s.values.map((v,i)=>`${px(i)},${py(v)}`).join(\' \')}" fill="none" stroke="${s.color}" stroke-width="2" ${s.dash?\'stroke-dasharray="4 4"\':\'\'}/>`});\n html+=`<text x="${left}" y="${height-5}" fill="#6d8184" font-size="10">${esc(dates[0]||\'Đầu kỳ\')}</text><text x="${width-right}" y="${height-5}" text-anchor="end" fill="#6d8184" font-size="10">${esc(dates.at(-1)||\'Cuối kỳ\')} ${esc(suffix)}</text></svg>`;\n return html;\n}\nfunction renderHealth(){const m=CONFIG.models.diabetes;$(\'health-fields\').innerHTML=healthFields.map(k=>field(k,Number(m.inputs[k].value.toFixed(2)),\'d\',healthUnits[k])).join(\'\');$(\'health-extra\').innerHTML=extraFields.map(k=>k===\'sex\'?\'<label>Mã giới tính trong dữ liệu<span class="unit">Chỉ dùng mã theo bộ dữ liệu nghiên cứu</span><select id="d-sex"><option value="">Chưa có · dùng trung vị</option value="1">Mã 1</option><option value="2">Mã 2</option></select></label>\':field(k,\'\',\'d\',healthUnits[k],\'Trung vị: \'+fmt(m.inputs[k].value,2))).join(\'\');$(\'health-form\').addEventListener(\'submit\',e=>{e.preventDefault();analyzeHealth()});$(\'health-form\').addEventListener(\'input\',invalidateHealth);$(\'health-form\').addEventListener(\'change\',invalidateHealth);$(\'reset-health\').addEventListener(\'click\',resetHealth)}\nfunction analyzeHealth(){\n const m=CONFIG.models.diabetes,values={},missing=[],errors=[];\n m.features.forEach(k=>{const raw=$(\'d-\'+k).value.trim();if(!raw&&extraFields.includes(k)){values[k]=m.inputs[k].value;missing.push(k);return}const v=parseNumber(raw);if(!Number.isFinite(v))errors.push(`${labels[k]}: cần nhập một số hợp lệ.`);else if(bounds[k]&&(v<bounds[k][0]||v>bounds[k][1]))errors.push(`${labels[k]}: khoảng nhập hỗ trợ ${bounds[k].join(\'–\')} ${healthUnits[k].split(\' · \')[0]}.`);else if(k===\'age\'&&!Number.isInteger(v))errors.push(\'Tuổi phải là số nguyên.\');values[k]=v});\n const aRaw=$(\'hba1c\').value.trim(),a=aRaw?parseNumber(aRaw):null;\n if(aRaw&&(!Number.isFinite(a)||a<3||a>20))errors.push(\'HbA1c: nhập giá trị 3–20%, hoặc để trống.\');\n $(\'health-errors\').hidden=!errors.length;$(\'health-errors\').textContent=errors.join(\' \');if(errors.length){$(\'health-errors\').scrollIntoView({block:\'nearest\'});return}\n const pregnant=$(\'pregnant\').checked,context=$(\'glucose-context\').value,ood=outOfRange(m,values),g=glucoseAssessment(values.s6,context,pregnant),b=bmiAssessment(values.bmi),ac=a===null?null:a1cAssessment(a,pregnant),y=predictModel(m,values),radius=m.interval.radius;\n const items=[{...g,sub:`Glucose ${fmt(values.s6)} mg/dL · ${contextNames[context]}`},...(ac?[{...ac,sub:`HbA1c ${fmt(a)}%`}]:[]),...(!pregnant?[{...b,sub:`BMI ${fmt(values.bmi)} kg/m²`}]:[])];\n const action=g.low?g.text:items.some(i=>i.level===\'high\')?\'Nên đặt lịch khám để xác nhận xét nghiệm và xây dựng kế hoạch theo dõi phù hợp.\':\'Ghi lại kết quả và theo dõi định kỳ theo kế hoạch cá nhân. Nếu có triệu chứng bất thường, hãy trao đổi với nhân viên y tế.\';\n const modelBlock=pregnant?\'<div class="callout watch">Không áp dụng mô hình nghiên cứu này cho thai kỳ. Phần nhận xét xét nghiệm cần trao đổi với bác sĩ.</div>\':`<div class="score-header"><span class="kicker">ĐIỂM TIẾN TRIỂN NGHIÊN CỨU</span><span class="badge">${esc(m.name)}</span></div><div class="score">${fmt(y)} <small>điểm</small></div><div class="interval">Khoảng dự báo hiệu chỉnh: <strong>${fmt(y-radius)} – ${fmt(y+radius)}</strong> điểm<br>Mục tiêu 90% · Bao phủ trên tập kiểm tra: ${fmt(m.interval.test_coverage*100)}%${ood.length?\'<br><b>Có đầu vào ngoài phạm vi train: khoảng trên không được kiểm chứng cho trường hợp này.</b>\':\'\'}</div>`;\n $(\'health-result\').innerHTML=`${modelBlock}<h3 style="margin-top:20px">Nhận xét theo từng chỉ số</h3>${items.map(i=>assessmentHTML(i,i.sub)).join(\'\')}${a===null?\'<p class="hint">Chưa có HbA1c · Không suy diễn xét nghiệm còn thiếu.</p>\':\'\'}<div class="callout ${g.low?\'high\':\'\'}"><b>Bước tiếp theo</b><br>${esc(action)}</div>${!g.low?\'<p class="hint">Ăn uống: ưu tiên rau, đạm nạc và khẩu phần tinh bột phù hợp. Có thể trao đổi tiếp với trợ lý về bữa ăn hoặc vận động.</p>\':\'\'}${ood.length?`<div class="callout watch"><b>Đầu vào ngoài phạm vi huấn luyện</b><br>${ood.map(k=>`${esc(labels[k])}: ${fmt(values[k],2)} (train ${fmt(m.inputs[k].min,2)}–${fmt(m.inputs[k].max,2)})`).join(\'<br>\')}<br>Điểm mô hình là ngoại suy, cần đọc thận trọng.</div>`:\'\'}<details class="model-details"><summary>Mô hình đã dùng những dữ liệu nào?</summary><p class="hint">${missing.length?\'Các trường dùng trung vị tập train: \'+missing.map(k=>`${esc(labels[k])} = ${fmt(values[k],2)}`).join(\'; \'):\'Đã cung cấp đủ 10 trường.\'} HbA1c và bối cảnh đo chỉ phục vụ nhận xét xét nghiệm, không nằm trong mô hình hồi quy.</p><p class="hint">Điểm mô hình không được dùng làm tiêu chí chẩn đoán hay quyết định cấp cứu.</p></details>${sourcesHTML([\'diagnosis\',...(g.low?[\'low\']:[])])}<div class="result-actions"><button class="button primary" id="discuss-result">Trao đổi với trợ lý →</button></div>`;\n lastHealth={...values,a1c:a,glucose:values.s6,context,pregnant,score:pregnant?null:y,missing};$(\'share-health\').disabled=false;$(\'discuss-result\').addEventListener(\'click\',()=>{shareHealth();location.hash=\'chatbot\'});\n}\nconst marketMain=[\'Close\',\'Close_Lag1\',\'MA20\',\'MA60\',\'RSI14\',\'Volatility20\'];\nconst marketExtra=[\'Return_Lag1\',\'Volume_Change\',\'Range_Pct\',\'Open_Close_Pct\'];\nconst percentFields=new Set([\'Volatility20\',\'Return_Lag1\',\'Volume_Change\',\'Range_Pct\',\'Open_Close_Pct\']);\nfunction marketInputValue(m,k){return Number((m.latest[k]*(percentFields.has(k)?100:1)).toFixed(4))}\nfunction renderMarket(kind){const m=CONFIG.models[kind],name=kind===\'stock\'?\'FPT\':\'vàng\';$(\'panel-\'+kind).innerHTML=`<div class="card input-card"><div class="card-heading"><div><span class="kicker">${m.symbol} · ${m.currency}</span><h3>Dự báo ${name} phiên kế tiếp</h3></div><button class="text-button" id="reset-${kind}">Khôi phục dữ liệu</button></div><p class="hint">Quan sát mới nhất: ${m.as_of}. Đây là ảnh chụp dữ liệu khi chạy notebook, không phải giá trực tiếp.${kind===\'gold\'?\' Vàng futures quốc tế, không phải vàng SJC.\':\'\'}</p><form id="form-${kind}" novalidate><div class="form-grid">${marketMain.map(k=>field(k,marketInputValue(m,k),kind,k===\'RSI14\'?\'0–100\':percentFields.has(k)?\'% / năm\':m.currency)).join(\'\')}</div><details class="advanced"><summary>Đặc trưng thị trường bổ sung</summary><p class="hint">Kịch bản dùng đủ 11 đặc trưng. Lợi suất hiện tại tự tính từ giá đóng cửa và giá phiên trước.</p><div class="form-grid">${marketExtra.map(k=>field(k,marketInputValue(m,k),kind,\'%\')).join(\'\')}</div></details><div id="${kind}-errors" class="error-box" role="alert" hidden></div><button type="submit" class="button primary full" style="margin-top:20px">Dự đoán ${name} <span>→</span></button></form><p class="hint" style="margin-bottom:0">Sửa số liệu để thử một kịch bản. Hãy giữ các chỉ báo nhất quán với chuỗi giá tương ứng.</p></div><div class="output-column"><div class="card result-card" id="${kind}-result" aria-live="polite"><span class="kicker">LỊCH SỬ GIÁ</span><h3>80 quan sát gần nhất</h3>${historyChart(m)}<p class="hint">Chọn “Dự đoán ${name}” để xem dự báo kèm khoảng sai số và đối chiếu với giá gần nhất.</p></div><div class="info-strip">Dự báo một phiên kế tiếp. Khoảng biến động chỉ là ước lượng từ dữ liệu; không dùng làm khuyến nghị mua bán.</div></div>`;$(\'form-\'+kind).addEventListener(\'submit\',e=>{e.preventDefault();runMarket(kind)});$(\'reset-\'+kind).addEventListener(\'click\',()=>{[...marketMain,...marketExtra].forEach(k=>$(kind+\'-\'+k).value=marketInputValue(m,k));runMarket(kind)});$(\'form-\'+kind).addEventListener(\'input\',()=>{$(kind+\'-result\').innerHTML=\'<span class="kicker">KỊCH BẢN ĐÃ THAY ĐỔI</span><h3>Chạy dự đoán để cập nhật.</h3>\'+historyChart(m)})}\nfunction historyChart(m){return \'<div class="chart-legend"><span><i class="legend-dot"></i>Giá đóng cửa</span><span><i class="legend-dot gray"></i>MA20</span></div>\'+svgSeries([{values:m.history.map(r=>r.close),color:\'#087c69\'},{values:m.history.map(r=>r.ma20),color:\'#bac3c1\',dash:true}],{label:\'Lịch sử giá \'+m.symbol,dates:m.history.map(r=>r.date)})}\nfunction runMarket(kind){const m=CONFIG.models[kind],v={},errors=[];[...marketMain,...marketExtra].forEach(k=>{let x=parseNumber($(kind+\'-\'+k).value);if(!Number.isFinite(x))errors.push(`${labels[k]} cần là số hợp lệ.`);if([\'Close\',\'Close_Lag1\',\'MA20\',\'MA60\'].includes(k)&&x<=0)errors.push(`${labels[k]} phải lớn hơn 0.`);if(k===\'RSI14\'&&(x<0||x>100))errors.push(\'RSI nằm trong 0–100.\');if([\'Volatility20\',\'Range_Pct\'].includes(k)&&x<0)errors.push(`${labels[k]} không được âm.`);if([\'Return_Lag1\',\'Volume_Change\',\'Open_Close_Pct\'].includes(k)&&x<=-100)errors.push(`${labels[k]} phải lớn hơn -100%.`);v[k]=percentFields.has(k)?x/100:x});$(kind+\'-errors\').hidden=!errors.length;$(kind+\'-errors\').textContent=errors.join(\' \');if(errors.length)return;v.Return_1D=v.Close/v.Close_Lag1-1;const y=predictModel(m,v);if(!Number.isFinite(y)||y<=0){$(kind+\'-errors\').hidden=false;$(kind+\'-errors\').textContent=\'Kịch bản tạo dự báo không hợp lệ. Hãy kiểm tra mức giá và các chỉ báo.\';return}const diff=y-v.Close,radius=m.interval.radius*v.Close,ood=outOfRange(m,v),scenario=m.features.some(k=>Math.abs(v[k]-m.latest[k])>Math.max(1e-5,Math.abs(m.latest[k])*1e-5));\n $(kind+\'-result\').innerHTML=`<div class="score-header"><span class="kicker">GIÁ ĐÓNG CỬA PHIÊN KẾ TIẾP</span><span class="badge">${scenario?\'Kịch bản tùy chỉnh\':\'Dữ liệu \'+m.as_of}</span></div><div class="score">${fmt(y,kind===\'stock\'?0:2)} <small>${m.currency}</small></div><p class="small">${diff>=0?\'+\':\'\'}${fmt(diff,2)} ${m.currency} · ${diff>=0?\'+\':\'\'}${fmt(diff/v.Close*100,2)}% so với đầu vào</p><div class="interval">Khoảng dự báo mục tiêu 90%: <strong>${fmt(Math.max(0,y-radius),2)} – ${fmt(y+radius,2)}</strong><br>Hiệu chỉnh từ ${m.interval.calibration_size} phiên gần đây (${m.interval.calibration_start} – ${m.interval.calibration_end}).</div><div class="callout"><b>${Math.abs(diff)<=radius?\'Biến động dự báo nằm trong vùng sai số\':\'Biến động dự báo vượt bán kính sai số lịch sử\'}</b><br>${m.name===\'Persistence\'?\'Cross-validation chọn giữ nguyên giá hiện tại. Mô hình không phát tín hiệu tăng/giảm.\':\'Đọc cùng sai số ngoài mẫu; không diễn giải chênh lệch nhỏ thành tín hiệu chắc chắn.\'}</div>${historyChart(m)}${ood.length?`<div class="callout watch">Ngoài phạm vi ${m.ood_space===\'relative\'?\'đặc trưng tương đối đã học\':\'train\'}: ${ood.map(k=>esc(labels[k]||k)).join(\', \')}. Khoảng dự báo không được kiểm chứng cho kịch bản này.</div>`:\'\'}<details class="model-details"><summary>Giải thích & chất lượng dự báo</summary><div class="stat-row"><span>Mô hình chọn theo CV</span><b>${esc(m.name)}</b></div><div class="stat-row"><span>MAE trên test lịch sử</span><b>${fmt(m.metrics.MAE,2)} ${m.currency}</b></div><div class="stat-row"><span>MAE giữ nguyên giá</span><b>${fmt(m.metrics.Persistence_MAE,2)}</b></div><div class="stat-row"><span>Cải thiện so với baseline</span><b>${fmt(m.metrics.Improvement_vs_Persistence_pct)}%</b></div>${trainingDetails(m)}<p class="hint">Nguồn: ${esc(m.source)}. Trên kịch bản tự nhập, chỉ số test và khoảng sai số lịch sử không phải cam kết chất lượng cho kịch bản.</p></details>`;\n}\nfunction trainingDetails(m){return `<div class="stat-row"><span>MAE bản trước → bản này</span><b>${fmt(m.previous.mae,2)} → ${fmt(m.metrics.MAE,2)}</b></div><p class="hint">Đánh giá cố định: ${m.metrics.test_start} – ${m.metrics.test_end}. Bao phủ lịch sử: ${fmt(m.evaluation_interval.test_coverage*100)}%. Bản dùng trên web được fit lại đến ${m.deployment.train_end}${m.deployment.window?\', dùng \'+m.deployment.window+\' phiên gần nhất\':\'\'}; khoảng dự báo được hiệu chỉnh lại, không dùng tỷ lệ bao phủ lịch sử làm cam kết cho bản refit.</p><p class="hint">Khoảng bootstrap 95% của mức cải thiện so với baseline: ${m.metrics.Improvement_CI95_pct.map(v=>fmt(v,2)+\'%\').join(\' đến \')}. Khoảng chứa 0 nghĩa là chưa có bằng chứng rõ ràng rằng sai số đã giảm.</p>`}\nfunction activateTab(kind,focus=false){document.querySelectorAll(\'.model-tab\').forEach(b=>{const active=b.dataset.kind===kind;b.classList.toggle(\'active\',active);b.setAttribute(\'aria-selected\',String(active));b.tabIndex=active?0:-1;$(\'panel-\'+b.dataset.kind).hidden=!active;if(active&&focus)b.focus()})}\nfunction renderQuality(){\n const titles={diabetes:\'Tiểu đường\',stock:\'Chứng khoán FPT\',gold:\'Vàng quốc tế\'};\n $(\'quality-cards\').innerHTML=Object.entries(CONFIG.models).map(([key,m])=>{\n  const historical=m.evaluation_interval||m.interval;\n  return `<article class="card quality-card"><span class="badge">${esc(m.name)}</span><h3>${titles[key]}</h3>\n   <div class="metric-value">${fmt(m.metrics.MAE,2)}</div><p class="metric-label">MAE lịch sử · ${key===\'diabetes\'?\'điểm\':m.currency}</p>\n   <div class="stat-row"><span>MAE bản trước</span><b>${fmt(m.previous.mae,2)}</b></div>\n   <div class="stat-row"><span>Giảm sai số so với bản trước</span><b>${fmt(m.previous.improvement_pct,2)}%</b></div>\n   <div class="stat-row"><span>R² trên test</span><b>${fmt(m.metrics.R2,3)}</b></div>\n   <div class="stat-row"><span>Train / Hiệu chỉnh / Test lịch sử</span><b>${m.metrics.train_size} / ${m.metrics.calibration_size} / ${m.metrics.test_size}</b></div>\n   <div class="stat-row"><span>Bao phủ khoảng dự báo lịch sử</span><b>${fmt(historical.test_coverage*100)}%</b></div>\n   ${key!==\'diabetes\'?`<div class="stat-row"><span>So với giữ nguyên giá</span><b>${fmt(m.metrics.Improvement_vs_Persistence_pct,2)}%</b></div><p class="hint">${m.metrics.Improvement_vs_Persistence_pct>0?\'Sai số test thấp hơn baseline trong giai đoạn đánh giá.\':m.name===\'Persistence\'?\'Baseline vẫn thắng CV sau khi mở rộng dữ liệu và tuning. Chưa có bằng chứng để thay bằng mô hình phức tạp.\':\'Đã giảm sai số so với bản trước nhưng vẫn chưa thắng baseline trên test.\'}</p><details><summary>Dữ liệu & độ bất định</summary>${trainingDetails(m)}</details>`:`<div class="stat-row"><span>MAE dự báo trung vị</span><b>${fmt(m.metrics.Baseline_MAE,2)}</b></div><p class="hint">${esc(m.source)}. Không suy rộng kết quả thành độ chính xác chẩn đoán.</p>`}\n  </article>`;\n }).join(\'\');\n $(\'model-comparison\').innerHTML=\'<div class="comparison-grid">\'+Object.entries(CONFIG.models).map(([key,m])=>`<div><h4>${titles[key]}</h4><table><thead><tr><th>Mô hình</th><th>CV MAE ↓</th></tr></thead><tbody>${[...m.comparison].sort((a,b)=>a.CV_MAE-b.CV_MAE).map(r=>`<tr class="${r[\'Mô hình\']===m.name?\'selected\':\'\'}"><td>${esc(r[\'Mô hình\'])}</td><td>${fmt(r.CV_MAE,2)}</td></tr>`).join(\'\')}</tbody></table><p class="hint">${m.comparison.reduce((sum,r)=>sum+r[\'Cấu hình đã thử\'],0)} cấu hình. ${key===\'diabetes\'?\'K-fold 5 phần; giữ nguyên 89 mẫu test.\':`TimeSeriesSplit 5 phần × ${m.metrics.cv_fold_size} phiên; gap=1. Lịch sử tải 5 năm; giữ nguyên giai đoạn test của bản trước.`}</p></div>`).join(\'\')+\'</div>\';\n $(\'build-date\').textContent=\'Bản \'+CONFIG.version+\' · \'+new Date(CONFIG.generated_at).toLocaleString(\'vi-VN\');\n}\nfunction updateContext(){const p=chat.profile,rows=clinical.contextRows(p,fmt);if(p.glucose)rows.unshift([\'Glucose\',`${fmt(p.glucose)} mg/dL`],[\'Thời điểm\',contextNames[p.context]||\'Chưa rõ\']);if(p.a1c)rows.push([\'HbA1c\',`${fmt(p.a1c)}%`]);if(p.bmi)rows.push([\'BMI\',fmt(p.bmi)]);if(p.pregnant)rows.push([\'Thai kỳ\',\'Đã ghi nhận\']);if(p.score!==null&&p.score!==undefined)rows.push([\'Điểm nghiên cứu\',fmt(p.score)]);for(const k of p.missing||[])rows.push([labels[k]||k,\'Chưa cung cấp\']);$(\'chat-context\').innerHTML=rows.length?rows.map(([k,v])=>`<div class="stat-row"><span>${esc(k)}</span><b>${esc(v)}</b></div>`).join(\'\'):\'Chưa có thông số sức khỏe. Bạn có thể hỏi trực tiếp mà không cần chạy dự đoán.\';}\nfunction addMessage(content,who=\'bot\',html=false){const row=document.createElement(\'div\');row.className=\'message \'+who;const bubble=document.createElement(\'div\');bubble.className=\'bubble\';if(html)bubble.innerHTML=content;else bubble.textContent=content;if(who===\'bot\'){const avatar=document.createElement(\'span\');avatar.className=\'mini-avatar\';avatar.textContent=\'✳\';row.append(avatar)}row.append(bubble);$(\'messages\').append(row);$(\'messages\').scrollTop=$(\'messages\').scrollHeight}\nfunction suggestions(items){$(\'quick-replies\').replaceChildren();items.forEach(text=>{const b=document.createElement(\'button\');b.textContent=text;b.addEventListener(\'click\',()=>sendQuestion(text));$(\'quick-replies\').append(b)})}\nfunction sendQuestion(raw){const question=raw.trim().slice(0,1200);if(!question)return;addMessage(question,\'user\');$(\'chat-input\').value=\'\';const reply=chat.respond(question);addMessage(reply.html,\'bot\',true);suggestions(reply.suggestions);updateContext()}\nfunction shareHealth(){if(!lastHealth)return;chat.share(lastHealth);addMessage(\'Mình muốn trao đổi về các chỉ số vừa phân tích.\',\'user\');const reply=chat.summaryReply();addMessage(\'<p>Mình đã nhận đầy đủ các thông số bạn chọn chia sẻ. Trường dùng trung vị chỉ phục vụ mô hình, không được coi là số đo của bạn.</p>\'+reply.html,\'bot\',true);suggestions(reply.suggestions);updateContext()}\nfunction clearChat(){chat.reset();$(\'messages\').replaceChildren();updateContext();addMessage(\'<h4>Chào bạn, mình sẵn sàng lắng nghe.</h4><p>Mình có thể giải thích đường huyết, HbA1c, BMI, huyết áp, cholesterol, HDL, LDL và triglyceride. Bạn có thể gửi nhiều chỉ số trong một tin nhắn; hãy ghi đơn vị cạnh từng số.</p><p class="hint">Ví dụ: “Huyết áp 120/90 mmHg; HDL 47 mg/dL”. Nếu thiếu loại chỉ số hoặc đơn vị, mình sẽ hỏi thêm.</p>\',\'bot\',true);suggestions([\'Huyết áp 90 có cao không?\',\'HDL 47 mg/dL\',\'Đường huyết 150 mg/dL\',\'Nhận xét tổng thể\'])}\ndocument.querySelectorAll(\'.model-tab\').forEach((b,i)=>{b.addEventListener(\'click\',()=>activateTab(b.dataset.kind));b.addEventListener(\'keydown\',e=>{const keys=[\'diabetes\',\'stock\',\'gold\'];if([\'ArrowRight\',\'ArrowLeft\',\'Home\',\'End\'].includes(e.key)){e.preventDefault();const next=e.key===\'Home\'?0:e.key===\'End\'?2:(i+(e.key===\'ArrowRight\'?1:2))%3;activateTab(keys[next],true)}})});\nrenderHealth();renderMarket(\'stock\');renderMarket(\'gold\');renderQuality();clearChat();\n$(\'chat-form\').addEventListener(\'submit\',e=>{e.preventDefault();sendQuestion($(\'chat-input\').value)});\n$(\'chat-input\').addEventListener(\'keydown\',e=>{if(e.key===\'Enter\'&&!e.shiftKey&&!e.isComposing){e.preventDefault();sendQuestion(e.target.value)}});\n$(\'clear-chat\').addEventListener(\'click\',clearChat);$(\'share-health\').addEventListener(\'click\',shareHealth);\ndocument.querySelectorAll(\'[data-question]\').forEach(b=>b.addEventListener(\'click\',()=>sendQuestion(b.dataset.question)));\n</script></body></html>\n'
html=html_template.replace("__MODEL_CONFIG__",json.dumps(web_config,ensure_ascii=False,allow_nan=False))
(WEB_DIR/"index.html").write_text(html,encoding="utf-8")
(WEB_DIR/"vercel.json").write_text(json.dumps({"rewrites":[{"source":"/(.*)","destination":"/index.html"}],"headers":[{"source":"/(.*)","headers":[{"key":"X-Content-Type-Options","value":"nosniff"},{"key":"Referrer-Policy","value":"strict-origin-when-cross-origin"}]}]},indent=2),encoding="utf-8")
print("Đã xuất Insight Lab 2.0:",{k:v["name"] for k,v in web_config["models"].items()})

# Sơ đồ bố cục minh họa, có nhãn rõ ràng; không giả làm ảnh chụp màn hình.
fig,ax=plt.subplots(figsize=(12,7),dpi=240)
ax.axis("off"); ax.set_xlim(0,12);ax.set_ylim(0,7)
def layout_box(x,y,w,h,title,subtitle):
    ax.add_patch(plt.Rectangle((x,y),w,h,facecolor="#f5f5f5",edgecolor="black",lw=.8))
    ax.text(x+.2,y+h-.35,title,fontweight="bold",fontsize=11,color="black")
    ax.text(x+.2,y+h-.75,subtitle,fontsize=9,color="black",va="top")
ax.text(.25,6.75,"INSIGHT LAB 2.0 - SƠ ĐỒ BỐ CỤC GIAO DIỆN",fontsize=15,fontweight="bold",color="black")
for x,title in [(0.25,"TIỂU ĐƯỜNG"),(4.15,"FPT.VN"),(8.05,"VÀNG GC=F")]:
    layout_box(x,5.4,3.7,.95,title,"Ba tab trong cùng khu Dự đoán")
layout_box(.25,3.4,5.55,1.75,"ĐẦU VÀO","Đơn vị rõ ràng · kiểm tra dữ liệu\nThông số bổ sung · bối cảnh xét nghiệm")
layout_box(6,3.4,5.75,1.75,"KẾT QUẢ","Dự báo · khoảng sai số · nhận xét\nSo sánh baseline · ngoài phạm vi train")
layout_box(.25,.35,7.4,2.75,"CHATBOT SỨC KHỎE ĐỘC LẬP","Truy xuất BM25 + bằng chứng KG\nHỏi tiếp đơn vị / thời điểm đo\nNhớ ngữ cảnh trong phiên · dẫn nguồn\nChỉ nhận hồ sơ khi người dùng chọn chia sẻ")
layout_box(7.9,.35,3.85,2.75,"NGỮ CẢNH","Glucose · HbA1c · BMI\nThời điểm đo · thai kỳ\nGợi ý câu hỏi tiếp theo")
fig.tight_layout();fig.savefig(FIG_DIR/"web_app_preview.png",dpi=240,bbox_inches="tight");plt.close(fig)


In [ ]:
# 5.2. Tạo báo cáo PDF chi tiết
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY, TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.platypus import (
    BaseDocTemplate, Frame, PageTemplate, Paragraph, Spacer, PageBreak,
    Image, Table, TableStyle, KeepTogether
)
from reportlab.platypus.tableofcontents import TableOfContents


FONT_DIR = Path("/usr/share/fonts/truetype/liberation")
pdfmetrics.registerFont(TTFont("TNR", str(FONT_DIR / "LiberationSerif-Regular.ttf")))
pdfmetrics.registerFont(TTFont("TNR-Bold", str(FONT_DIR / "LiberationSerif-Bold.ttf")))
pdfmetrics.registerFont(TTFont("TNR-Italic", str(FONT_DIR / "LiberationSerif-Italic.ttf")))

PAGE_W, PAGE_H = A4
ink, pale, line = colors.black, colors.HexColor("#f2f2f2"), colors.HexColor("#777777")
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="VNBody", parent=styles["BodyText"], fontName="TNR", fontSize=12,
                          leading=17, alignment=TA_JUSTIFY, spaceAfter=8, textColor=ink))
styles.add(ParagraphStyle(name="VNTitle", parent=styles["Title"], fontName="TNR-Bold", fontSize=22,
                          leading=28, alignment=TA_CENTER, textColor=ink, spaceAfter=12))
styles.add(ParagraphStyle(name="VNChapter", parent=styles["Heading1"], fontName="TNR-Bold", fontSize=16,
                          leading=21, textColor=ink, spaceAfter=12, keepWithNext=True))
styles.add(ParagraphStyle(name="VNHeading", parent=styles["Heading2"], fontName="TNR-Bold", fontSize=13,
                          leading=18, textColor=ink, spaceAfter=9, keepWithNext=True))
styles.add(ParagraphStyle(name="VNSmall", parent=styles["BodyText"], fontName="TNR", fontSize=9,
                          leading=12, textColor=ink))
styles.add(ParagraphStyle(name="VNCaption", parent=styles["BodyText"], fontName="TNR-Italic", fontSize=9,
                          leading=12, alignment=TA_CENTER, textColor=ink, spaceAfter=8))
styles.add(ParagraphStyle(name="VNQuote", parent=styles["BodyText"], fontName="TNR", fontSize=11,
                          leading=16, leftIndent=14, rightIndent=14, borderColor=ink, borderWidth=.6,
                          borderPadding=8, backColor=pale, spaceAfter=10, textColor=ink))


class ReportDoc(BaseDocTemplate):
    def __init__(self, filename):
        super().__init__(filename, pagesize=A4, rightMargin=1.8*cm, leftMargin=2.2*cm,
                         topMargin=1.7*cm, bottomMargin=1.8*cm,
                         title="Knowledge Graph và RAG", author=STUDENT_NAME)
        frame = Frame(self.leftMargin, self.bottomMargin, self.width, self.height, id="normal")
        cover_frame = Frame(2*cm, self.bottomMargin, PAGE_W-4*cm, self.height, id="cover")
        self.addPageTemplates([
            PageTemplate(id="cover", frames=cover_frame, onPage=self._header_footer, autoNextPageTemplate="body"),
            PageTemplate(id="body", frames=frame, onPage=self._header_footer),
        ])

    def _header_footer(self, canvas, doc):
        canvas.saveState()
        canvas.setFillColor(colors.black); canvas.setStrokeColor(colors.black)
        if doc.page == 1:
            canvas.setLineWidth(1.0)
            canvas.rect(0.9*cm, 0.9*cm, PAGE_W-1.8*cm, PAGE_H-1.8*cm)
            canvas.setLineWidth(.35)
            canvas.rect(1.1*cm, 1.1*cm, PAGE_W-2.2*cm, PAGE_H-2.2*cm)
            canvas.setFont("TNR-Bold", 28)
            canvas.drawCentredString(PAGE_W/2, PAGE_H-7.0*cm, "PTIT")
        else:
            canvas.setLineWidth(.35)
            canvas.line(2.2*cm, PAGE_H-1.2*cm, PAGE_W-1.8*cm, PAGE_H-1.2*cm)
            canvas.setFont("TNR", 8)
            canvas.drawString(2.2*cm, PAGE_H-0.95*cm, "CHƯƠNG 4 - KNOWLEDGE GRAPH VÀ RAG")
            canvas.drawRightString(PAGE_W-1.8*cm, PAGE_H-0.95*cm, f"{STUDENT_NAME} - {STUDENT_ID}")
            canvas.drawCentredString(PAGE_W/2, 0.9*cm, str(doc.page - 1))
        canvas.restoreState()

    def afterFlowable(self, flowable):
        if not isinstance(flowable, Paragraph):
            return
        title = flowable.getPlainText()
        style_name = flowable.style.name
        if style_name not in {"VNChapter", "VNHeading"}:
            return
        if title.startswith("4.") or title.startswith("PHỤ LỤC") or title == "TÀI LIỆU THAM KHẢO":
            level = 0 if style_name == "VNChapter" else 1
            self.notify("TOCEntry", (level, title, self.page - 1))

def P(text, style="VNBody"):
    clean = str(text).replace("—", "-").replace("–", "-").replace("‑", "-")
    return Paragraph(clean, styles[style])

def report_table(data, widths=None, header=True, font_size=9.0):
    table = Table(data, colWidths=widths, repeatRows=1 if header else 0, hAlign="LEFT")
    table.setStyle(TableStyle([
        ("FONTNAME", (0,0), (-1,-1), "TNR"), ("FONTSIZE", (0,0), (-1,-1), font_size),
        ("LEADING", (0,0), (-1,-1), font_size+3), ("TEXTCOLOR", (0,0), (-1,-1), colors.black),
        ("VALIGN", (0,0), (-1,-1), "TOP"), ("GRID", (0,0), (-1,-1), .45, colors.black),
        ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, colors.HexColor("#f7f7f7")]),
        ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#e5e5e5") if header else colors.white),
        ("FONTNAME", (0,0), (-1,0), "TNR-Bold"), ("LEFTPADDING", (0,0), (-1,-1), 6),
        ("RIGHTPADDING", (0,0), (-1,-1), 6), ("TOPPADDING", (0,0), (-1,-1), 6),
        ("BOTTOMPADDING", (0,0), (-1,-1), 6),
    ]))
    return table

def report_image(path, max_width=15.5*cm, max_height=11.2*cm):
    image = Image(str(path))
    image._restrictSize(max_width, max_height)
    return image

styles.add(ParagraphStyle(name="VNCoverBody", parent=styles["VNBody"], alignment=TA_CENTER))
styles.add(ParagraphStyle(name="VNCoverHeading", parent=styles["VNHeading"], alignment=TA_CENTER))
story = []

# Bìa
story += [Spacer(1, 1.1*cm), P(SCHOOL, "VNCoverHeading"), P(FACULTY, "VNCoverHeading"), Spacer(1, 2.2*cm),
          P("CHƯƠNG 04", "VNTitle"), P("KNOWLEDGE GRAPH VÀ RAG", "VNTitle"), Spacer(1, 1.6*cm),
          P(f"<b>Học phần:</b> {COURSE}", "VNCoverBody"), P("<b>Giảng viên hướng dẫn:</b> Thầy Trần Đình Quế", "VNCoverBody"),
          P(f"<b>Sinh viên thực hiện:</b> {STUDENT_NAME}", "VNCoverBody"), P(f"<b>Mã sinh viên:</b> {STUDENT_ID}", "VNCoverBody"),
              P(f"<b>GitHub:</b> <link href='{GITHUB_URL}' color='black'>{GITHUB_URL}</link>", "VNCoverBody"),
              P(f"<b>Web demo:</b> <link href='{DEPLOY_URL}' color='black'>{DEPLOY_URL}</link>", "VNCoverBody"),
          Spacer(1, 4.0*cm), P("Hà Nội — 2026", "VNCoverHeading"), PageBreak()]



story.append(P("MỤC LỤC", "VNChapter"))
toc = TableOfContents()
toc.levelStyles = [
    ParagraphStyle(name="TOC0", fontName="TNR-Bold", fontSize=11.5, leading=17,
                   leftIndent=0, firstLineIndent=0, textColor=colors.black, spaceBefore=4),
    ParagraphStyle(name="TOC1", fontName="TNR", fontSize=10.5, leading=15,
                   leftIndent=18, firstLineIndent=0, textColor=colors.black),
]
story += [toc, Spacer(1, .45*cm),
          P(f"<b>Repository:</b> <link href='{GITHUB_URL}' color='black'>{GITHUB_URL}</link>"),
          P(f"<b>Web demo:</b> <link href='{DEPLOY_URL}' color='black'>{DEPLOY_URL}</link>"), PageBreak()]

pages = [
    ("4.1. MỞ ĐẦU", [
        "Knowledge Graph (KG) và Retrieval-Augmented Generation (RAG) xuất phát từ hai truyền thống khác nhau. KG nhấn mạnh biểu diễn tường minh: thực thể nào tồn tại, chúng liên hệ ra sao và ràng buộc nào phải được giữ. RAG xuất phát từ nhu cầu cho mô hình sinh tiếp cận một kho tri thức ngoài tại thời điểm trả lời. Khi ghép hai hướng này, hệ thống vừa có khả năng tìm kiếm mềm theo ngôn ngữ, vừa có cấu trúc để kiểm tra đường suy luận.",
        "Bài làm lựa chọn kiến trúc có thể chạy trọn trong một notebook. TF-IDF đảm nhiệm truy xuất văn bản, NetworkX lưu đồ thị, còn bộ tổng hợp ghép bằng chứng tìm được thành câu trả lời. Dữ liệu FPT.VN và GC=F được tải trực tiếp khi chạy; dữ liệu tiểu đường lấy từ scikit-learn.",
        "Ba miền được chọn tạo ra phép thử đủ khác biệt. Y tế yêu cầu diễn đạt thận trọng; chứng khoán là chuỗi thời gian nội địa; vàng chịu tác động của yếu tố vĩ mô và được niêm yết quốc tế. Một pipeline dùng chung phải giữ được ngữ cảnh riêng của từng miền mà không trộn lẫn bằng chứng."], "architecture.png"),
    ("4.1.1. LỊCH SỬ PHÁT TRIỂN CỦA KNOWLEDGE GRAPH", [
        "Ý tưởng nối các khái niệm thành mạng ngữ nghĩa đã xuất hiện từ những nghiên cứu ban đầu về trí nhớ và biểu diễn tri thức. Sau đó, hệ chuyên gia đưa luật IF–THEN vào các miền hẹp. Bước ngoặt của Web ngữ nghĩa là việc chuẩn hóa RDF, RDFS, OWL và SPARQL, khiến bộ ba chủ thể–vị từ–đối tượng trở thành một ngôn ngữ trao đổi máy đọc được [1], [2].",
        "Năm 2012, cách gọi Knowledge Graph được phổ biến rộng rãi qua công cụ tìm kiếm của Google. Trong doanh nghiệp, KG tiếp tục được dùng cho danh mục sản phẩm, phát hiện gian lận, quản trị dữ liệu chủ và gợi ý nội dung. Điểm chung không phải một loại cơ sở dữ liệu cụ thể, mà là cách coi quan hệ giữa các thực thể là công dân hạng nhất.",
        "Ở bài này, đồ thị được cài bằng Labeled Property Graph đơn giản. Mỗi nút có type và domain; mỗi cạnh có nhãn quan hệ. Lựa chọn này gần với cách Neo4j tổ chức dữ liệu và thuận tiện cho trực quan hóa, dù chưa khai thác đầy đủ suy luận ontology như OWL."], None),
    ("4.1.1. LỊCH SỬ PHÁT TRIỂN CỦA RAG", [
        "Trước RAG, các hệ hỏi đáp mở thường tách bộ tìm kiếm tài liệu và bộ đọc hiểu. REALM đưa cơ chế truy xuất vào quá trình tiền huấn luyện [3]. Lewis và cộng sự sau đó định danh kiến trúc Retrieval-Augmented Generation, kết hợp bộ truy xuất dense với mô hình sinh seq2seq cho nhiệm vụ giàu tri thức [4].",
        "Sự phổ biến của mô hình ngôn ngữ lớn khiến RAG trở thành một mẫu triển khai thực dụng: tài liệu có thể cập nhật mà không cần huấn luyện lại toàn bộ mô hình, đồng thời câu trả lời giữ được liên kết về nguồn. Tuy nhiên, vector search thường ưu tiên đoạn giống câu hỏi và có thể bỏ qua quan hệ gián tiếp. GraphRAG giải quyết phần thiếu này bằng việc xây thực thể, quan hệ, cụm và tóm tắt theo cấu trúc [5].",
        "Nguyên mẫu trong notebook không cố tái tạo toàn bộ Microsoft GraphRAG. Nó giữ hai ý cốt lõi: truy xuất đồng thời từ văn bản và đồ thị, sau đó đưa cả hai nhóm bằng chứng vào bước tổng hợp."], None),
    ("4.1.2. VAI TRÒ VÀ Ý NGHĨA TRONG AI", [
        "KG giúp mô hình có từ điển thực thể thống nhất, quan hệ có tên và đường đi có thể kiểm tra. RAG giúp tri thức ngoài đi vào ngữ cảnh trả lời đúng thời điểm. Kết hợp hai thành phần làm giảm nguy cơ câu trả lời chỉ dựa trên tương đồng bề mặt, nhưng không đồng nghĩa loại bỏ hoàn toàn sai sót.",
        "Trong y tế, giá trị quan trọng nhất là provenance: người đọc phải biết nhận xét dựa trên biến nào và nguồn nào. Trong tài chính, dữ liệu có tính thời điểm nên ngày quan sát phải đi cùng con số. Trong cả hai trường hợp, câu trả lời nên phân biệt sự kiện đo được, quan hệ thường gặp và kết luận mang tính dự báo.",
        "Ý nghĩa học thuật của bài nằm ở khả năng chạy lại. Người chấm có thể Run All, xem dữ liệu thô đã lưu, đối chiếu bộ ba, kiểm tra tài liệu được truy xuất và mở PDF được sinh từ cùng một trạng thái chạy."], None),
    ("4.1.3. PHẠM VI, CÂU HỎI VÀ ĐÓNG GÓP", [
        "Bài đặt ba câu hỏi: (1) một cấu trúc KG thống nhất có mô tả được ba miền khác nhau không; (2) truy xuất lai có cung cấp nhiều bằng chứng hơn vector search đơn lẻ không; và (3) có thể đóng gói toàn bộ quy trình trong một notebook duy nhất không.",
        "Đóng góp thực hành gồm lớp KnowledgeGraphCore, bộ tải Yahoo Finance không cần API key, các bước tạo chỉ báo, VectorRetriever, truy xuất đồ thị theo miền, đánh giá precision@3, ứng dụng dự đoán, chatbot sức khỏe có nguồn và bộ sinh PDF. Mọi tệp phụ đều được tạo lại từ notebook.",
        "Ba nguồn dữ liệu được mô tả bằng số dòng, số cột, khoảng thời gian, đơn vị và biến đích. Bảng được sinh trực tiếp từ DataFrame sau xử lý nên khớp với lần chạy hiện tại."], "data_profile.png"),
    ("4.2. KNOWLEDGE GRAPH – KNOWLEDGE BASE", [
        "Knowledge Base trong bài gồm hai lớp. Lớp văn bản giữ các đoạn mô tả dữ liệu, phương pháp và kết quả theo từng miền. Lớp đồ thị giữ thực thể cùng các quan hệ có hướng. Hai lớp dùng chung nhãn domain để ngăn truy xuất nhầm giữa tiểu đường, cổ phiếu và vàng.",
        f"Đồ thị tại lần chạy này có {kg.graph.number_of_nodes()} nút và {kg.graph.number_of_edges()} cạnh. Kích thước nhỏ giúp người đọc kiểm tra thủ công. Khi mở rộng, cùng schema có thể chuyển sang Neo4j hoặc một triple store mà không thay đổi ý nghĩa bộ ba.",
        "Knowledge Base không chỉ chứa kiến thức tĩnh. Các nút quan sát như giá FPT hoặc giá vàng gần nhất được tạo từ dữ liệu tải về. Cách làm này nối câu trả lời với đúng trạng thái thị trường tại thời điểm chạy."], "knowledge_graph.png"),
    ("4.2.1. BỘ BA, THỰC THỂ VÀ QUAN HỆ", [
        "Một bộ ba có dạng (s, p, o), trong đó s là thực thể nguồn, p là quan hệ có tên và o là thực thể hoặc giá trị đích. Ví dụ: (MA20, MÔ_TẢ_XU_HƯỚNG_NGẮN_HẠN_CỦA, FPT.VN). Bộ ba này không khẳng định xu hướng sẽ tiếp diễn; nó chỉ diễn tả vai trò của chỉ báo.",
        "Thực thể được gắn type như Condition, Biomarker, Stock, TechnicalIndicator, Commodity hoặc MacroFactor. Thuộc tính domain tách ba không gian nghiệp vụ. Tên quan hệ dùng động từ viết hoa để người đọc có thể đọc thành câu.",
        "Việc chọn quan hệ là công đoạn có tính chủ quan. Báo cáo ưu tiên từ CÓ_THỂ_TÁC_ĐỘNG hoặc CÓ_QUAN_HỆ... thay vì GÂY_RA khi dữ liệu không đủ để chứng minh nhân quả."], None),
    ("4.2.2. KNOWLEDGE GRAPH CORE ENGINE", [
        "Lớp KnowledgeGraphCore bọc MultiDiGraph của NetworkX. add_entity quản lý nút và thuộc tính; add_relation thêm cạnh có hướng; neighbors trả về bằng chứng một bước; multi_hop tìm các đường đơn trong giới hạn số hop. Cấu trúc MultiDiGraph cho phép hai thực thể có nhiều loại quan hệ.",
        "Độ phức tạp của truy vấn lân cận tỷ lệ với bậc của nút. Truy vấn mọi đường đơn có thể tăng nhanh theo kích thước đồ thị, vì vậy max_hops phải được giới hạn. Với đồ thị sản xuất, cần chỉ mục, phân quyền và chiến lược truy vấn cụ thể thay vì duyệt toàn bộ.",
        "Toàn bộ cạnh được xuất thành knowledge_graph_triples.csv. Đây là điểm kiểm tra thuận tiện: mỗi dòng trong tệp phải đọc được như một phát biểu, không có nút mồ côi vô nghĩa và không chứa con số không rõ ngày."], "code_kg_core.png"),
    ("4.2.3. RDF VÀ LABELED PROPERTY GRAPH", [
        "RDF phù hợp khi cần URI toàn cục, liên kết dữ liệu và suy luận theo chuẩn W3C. Labeled Property Graph thuận tiện khi cạnh cần thuộc tính và truy vấn đường đi là trọng tâm. Bài chọn mô hình thứ hai vì mục tiêu là nguyên mẫu phân tích và trực quan hóa bằng Python.",
        "Nếu chuyển sang RDF, các entity_id cần ánh xạ thành IRI; relation thành predicate; giá trị số thành literal có datatype. Nếu chuyển sang Neo4j, type có thể thành node label, relation thành relationship type và các trường domain/source thành property.",
        "Không có mô hình nào tốt tuyệt đối. Tiêu chí chọn là khả năng tích hợp với hệ thống, yêu cầu suy luận, quy mô và kỹ năng vận hành của nhóm."], None),
    ("4.2.4. CHẤT LƯỢNG VÀ QUẢN TRỊ TRI THỨC", [
        "Một KG hữu ích cần hơn số lượng cạnh. Bốn kiểm tra tối thiểu gồm: tính duy nhất của định danh, tính hợp lệ của miền–phạm vi, nguồn gốc quan hệ và thời hạn của dữ liệu động. Các quan hệ y tế và tài chính phải có mức tin cậy hoặc câu chữ thể hiện độ bất định.",
        "Bài hiện lưu source ở lớp tài liệu và domain ở lớp đồ thị. Bản sản xuất nên bổ sung created_at, valid_from, valid_to, confidence và evidence_id cho từng cạnh. Khi đó, một quan hệ đã cũ có thể bị loại trước khi đưa vào prompt.",
        "Kiểm soát chất lượng cần vòng phản hồi của chuyên gia. Bác sĩ xác nhận ontology y tế; chuyên viên tài chính rà soát khái niệm và đơn vị; kỹ sư dữ liệu kiểm tra schema cùng pipeline cập nhật."], None),
    ("4.3. RAG — RETRIEVAL-AUGMENTED GENERATION", [
        "Pipeline RAG gồm ingestion, indexing, retrieval và generation. Văn bản được chia thành các đơn vị có nguồn; TF-IDF biến chúng thành vector; cosine similarity xếp hạng theo câu hỏi. Bước sinh ghép hai đoạn phù hợp nhất với các bộ ba liên quan.",
        "TF-IDF được chọn vì minh bạch, nhẹ và tái lập. Hạn chế là không hiểu đồng nghĩa sâu như embedding. Để giảm nhược điểm, câu hỏi được nhận diện miền bằng một từ điển alias và GraphRAG bổ sung các quan hệ từ đồ thị.",
        "Mỗi câu trả lời trả về cả documents, score, triples và domain. Cấu trúc này quan trọng hơn phần câu chữ: giao diện hoặc báo cáo có thể trình bày bằng chứng mà không phải phân tích ngược chuỗi văn bản."], "architecture.png"),
    ("4.3.1. NẠP, LẬP CHỈ MỤC VÀ TRUY XUẤT", [
        "Các tài liệu ngắn được xây từ mô tả bộ dữ liệu và kết quả thực nghiệm. Trong hệ thống lớn, ingestion cần đọc PDF, HTML hoặc cơ sở dữ liệu; chuẩn hóa Unicode; bỏ trùng; chia đoạn theo ngữ nghĩa; và gắn metadata trước khi embedding.",
        "TfidfVectorizer dùng unigram và bigram. Bigram giúp phân biệt các cụm như giá vàng, tiểu đường, lãi suất Fed. top_k=3 giữ ngữ cảnh ngắn và dễ kiểm tra. Precision@3 theo domain được dùng như phép đo đơn giản cho độ đúng của truy xuất.",
        "Điểm cosine chỉ phản ánh độ gần từ vựng, không phải độ đúng. Vì vậy báo cáo không diễn giải score như xác suất và không tự động loại bỏ nguồn chuyên môn chỉ vì score thấp."], "code_retriever.png"),
    ("4.3.2. TRUY XUẤT ĐỒ THỊ VÀ SUY LUẬN NHIỀU BƯỚC", [
        "graph_evidence nhận diện miền, chọn các nút cùng domain hoặc trùng từ khóa, rồi lấy quan hệ vào và ra. Phương án này đủ cho đồ thị nhỏ. Khi đồ thị lớn, cần entity linking, tìm kiếm theo embedding của nhãn và truy vấn Cypher có giới hạn.",
        "Suy luận nhiều bước được minh họa bởi chuỗi Rủi ro địa chính trị → Nhu cầu trú ẩn → Vàng quốc tế. Vector search có thể tìm một đoạn nói về địa chính trị nhưng không mặc nhiên biểu diễn hai quan hệ riêng. Đồ thị giúp giữ cấu trúc của lời giải thích.",
        "Đường đi chỉ là bằng chứng cấu trúc, không tự chứng minh nhân quả. Mọi kết luận vẫn phụ thuộc chất lượng cạnh và nguồn đã dùng để tạo cạnh."], "code_graphrag.png"),
    ("4.3.3. TỔNG HỢP CÂU TRẢ LỜI CÓ CĂN CỨ", [
        "Bộ sinh trong bài là deterministic synthesizer. Nó chọn hai tài liệu phù hợp, nối tối đa ba quan hệ đồ thị và thêm cảnh báo theo miền. Kết quả ít tự nhiên hơn LLM nhưng có lợi thế là chạy offline và không tự thêm sự kiện ngoài bằng chứng.",
        "Có thể thay bộ sinh bằng một LLM cục bộ hoặc dịch vụ API. Khi đó prompt phải tách rõ QUESTION, TEXT EVIDENCE, GRAPH EVIDENCE và yêu cầu từ chối nếu bằng chứng thiếu. Đầu ra nên ở dạng JSON để kiểm tra citation_id.",
        "Thiết kế production cần chống prompt injection từ tài liệu, lọc dữ liệu nhạy cảm, theo dõi phiên bản chỉ mục và ghi log các nguồn đã đưa cho mô hình."], None),
    ("4.3.4. ĐÁNH GIÁ RAG VÀ GRAPHRAG", [
        "Bộ kiểm thử gồm sáu câu hỏi, hai câu cho mỗi miền. vector_precision@3 đo tỷ lệ tài liệu thuộc đúng miền trong ba kết quả đầu. graph_evidence ghi nhận hệ thống có tìm được cạnh liên quan hay không. Điểm tổng hợp cộng một phần thưởng nhỏ khi cả hai kênh đều có bằng chứng.",
        f"Ở lần chạy này, precision@3 trung bình là {evaluation_df['vector_precision@3'].mean():.3f}; tỷ lệ câu hỏi có bằng chứng đồ thị là {evaluation_df['graph_evidence'].mean():.3f}; điểm bằng chứng lai là {evaluation_df['hybrid_evidence_score'].mean():.3f}.",
        "Đây là smoke test, chưa phải benchmark khoa học. Một đánh giá đầy đủ cần nhiều câu hỏi do người độc lập soạn, đáp án chuẩn, chấm faithfulness, answer relevance, citation correctness, độ trễ và chi phí."], "rag_benchmark.png"),

("4.4. KB VÀ RAG CHO BỆNH TIỂU ĐƯỜNG", [
    "Bộ dữ liệu diabetes của scikit-learn có 442 quan sát và 10 biến đầu vào ở thang đo gốc. Biến đích là chỉ số định lượng mức tiến triển bệnh sau một năm. Dữ liệu không chứa tên, mã hồ sơ hay thông tin nhận dạng cá nhân.",
    "Bốn cấu hình được so sánh bằng K-fold 5 phần trên tập train: Linear Regression, Ridge, Random Forest và Gradient Boosting. Tập test 20% được giữ riêng cho đến khi đã chọn mô hình theo CV MAE.",
    f"Mô hình được chọn là {diabetes_metrics['Model']}; CV MAE = {diabetes_metrics['CV_MAE']:.2f}, test MAE = {diabetes_metrics['MAE']:.2f}, RMSE = {diabetes_metrics['RMSE']:.2f}, R² = {diabetes_metrics['R2']:.3f}. Tham số: {best_params}."], "kg_diabetes.png"),
    ("4.4.1. ONTOLOGY VÀ XỬ LÝ DỮ LIỆU Y TẾ", [
        "Ontology tối giản gồm Condition, Biomarker, RiskFactor, Lifestyle, Monitoring và Professional. Các quan hệ như LIÊN_QUAN_ĐẾN hoặc HỖ_TRỢ_QUẢN_LÝ được chọn để tránh biến một tương quan thành chỉ định điều trị.",
        "Dữ liệu không có tên bệnh nhân hay định danh cá nhân. Trong tình huống thực, hồ sơ y tế cần mã hóa, phân quyền theo vai trò, nhật ký truy cập và quy trình chấp thuận. Không nên đưa thông tin nhận dạng vào dịch vụ embedding công cộng khi chưa có cơ sở pháp lý.",
        f"Mô hình {diabetes_metrics['Model']} được chọn theo MAE trung bình của 5 fold. Permutation importance được tính trên tập test để cùng một cách giải thích áp dụng được cho cả mô hình tuyến tính và mô hình cây."], "code_diabetes_training.png"),
    ("4.4.2. THỰC NGHIỆM HỎI ĐÁP Y TẾ", [
        "Câu hỏi mẫu hỏi các yếu tố liên quan đến quản lý bệnh tiểu đường. Vector retriever tìm mô tả theo dõi đường huyết, HbA1c, huyết áp, cân nặng, ăn uống và vận động. Đồ thị bổ sung các cạnh nối bệnh với theo dõi và chuyên gia.",
        f"Câu trả lời sinh ra trong lần chạy: {demo_answers[0]['answer']}",
        "Biểu đồ đối chiếu giá trị thực - dự đoán và feature importance cho thấy mức độ phù hợp cũng như biến nào được mô hình sử dụng nhiều nhất."], "diabetes_model.png"),
    ("4.4.3. GIỚI HẠN VÀ HƯỚNG MỞ RỘNG Y TẾ", [
        "Bộ dữ liệu nhỏ, không đại diện cho dân số Việt Nam và không chứa bối cảnh điều trị. Chỉ số đích mang ý nghĩa nghiên cứu, khác với chẩn đoán đái tháo đường trong thực hành. Vì vậy không thể suy rộng độ chính xác mô hình thành hiệu quả lâm sàng.",
        "Hướng mở rộng là dùng dữ liệu đã khử định danh, xác định rõ biến kết cục, đánh giá theo nhóm tuổi/giới và mời chuyên gia lâm sàng kiểm tra ontology. Ngoài độ chính xác, cần đo calibration, fairness và khả năng giải thích.",
        "Với RAG y tế, mọi đoạn văn cần version và ngày hiệu lực. Hướng dẫn cũ phải được thu hồi khỏi chỉ mục thay vì chỉ thêm tài liệu mới."], None),

("4.5. KB VÀ RAG CHO CHỨNG KHOÁN", [
    f"Dữ liệu FPT.VN được lấy từ {market_sources['stock']}. Khoảng thời gian {stock_summary['start']} - {stock_summary['end']} có {stock_summary['observations']} quan sát. Giá đóng cửa gần nhất là {stock_summary['latest_close']:,.0f} {stock_summary['currency']}.",
    f"Lợi suất toàn kỳ là {stock_summary['period_return_pct']:.2f}%, biến động năm hóa {stock_summary['volatility_pct']:.2f}%, RSI14 {stock_summary['rsi14']:.1f}. Mỗi con số được gắn ngày chạy và nguồn dữ liệu.",
    "Knowledge Graph nối FPT.VN với giá gần nhất, MA20, MA60, RSI, thanh khoản, kết quả kinh doanh và tâm lý thị trường. Các quan hệ mô tả vai trò chỉ báo, không tự động tạo khuyến nghị mua bán."], "kg_stock.png"),
("4.5.1. DỮ LIỆU VÀ TIỀN XỬ LÝ FPT", [
    "Chuỗi OHLCV được sắp xếp tăng dần theo ngày; dòng thiếu Close bị loại. Return_1D, MA20, MA60, volatility20 và RSI14 được tính từ dữ liệu lịch sử. Đặc trưng bổ sung gồm thay đổi khối lượng, biên độ High-Low, chênh lệch Open-Close và các độ trễ một phiên.",
    f"Tại điểm cuối, MA20 = {stock_summary['ma20']:,.0f}, MA60 = {stock_summary['ma60']:,.0f}. Bộ học có {stock_model_info['train_size']} mẫu train và {stock_model_info['test_size']} mẫu test theo thứ tự thời gian.",
    "Biến đích là giá Close của phiên kế tiếp. Việc dịch target một bước rồi drop dòng cuối giúp đặc trưng tại ngày t không chứa giá của ngày t+1."], "fpt_market.png"),
("4.5.2. HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH FPT", [
    "Ridge, Random Forest và Gradient Boosting được tuning bằng TimeSeriesSplit 5 phần. Khác K-fold ngẫu nhiên, mỗi fold chỉ huấn luyện trên quá khứ và xác nhận trên đoạn thời gian xảy ra sau đó. Tập test 20% cuối chuỗi không tham gia tuning.",
    f"Mô hình được chọn: {stock_model_info['Model']}; CV MAE = {stock_model_info['CV_MAE']:.2f}; test MAE = {stock_model_info['MAE']:.2f}; RMSE = {stock_model_info['RMSE']:.2f}; MAPE = {stock_model_info['MAPE']:.2f}%; R² = {stock_model_info['R2']:.3f}.",
    f"Độ đúng chiều biến động là {stock_model_info['Direction_Accuracy']:.1f}%. Persistence baseline có MAE {stock_model_info['Persistence_MAE']:.2f}; mức cải thiện của mô hình là {stock_model_info['Improvement_vs_Persistence_pct']:.2f}%. Báo cáo giữ nguyên kết quả kể cả khi baseline tốt hơn."], "stock_model.png"),
("4.5.3. HỎI ĐÁP CHỨNG KHOÁN VỚI GRAPHRAG", [
    f"Câu trả lời mẫu: {demo_answers[1]['answer']}",
    "Tài liệu cung cấp số liệu động; đồ thị giải thích vai trò của chỉ báo và liên kết multi-hop. Bộ lọc domain ngăn tài liệu vàng hoặc y tế đi vào câu trả lời chứng khoán.",
    "Kết quả mô hình được ghi thành tài liệu trong KB với model name, thời gian, MAE và R². Vì vậy RAG có thể dẫn lại đúng lần huấn luyện thay vì tạo ra độ chính xác không tồn tại."], "code_market_training.png"),
("4.5.4. RỦI RO VÀ HƯỚNG MỞ RỘNG", [
    "Corporate actions, thanh khoản, cách điều chỉnh giá và survivorship bias có thể làm sai phân tích. Yahoo Finance phù hợp bài tập nhưng nên đối chiếu nguồn chính thức nếu dùng trong nghiên cứu giao dịch.",
    "Dự báo Close phiên kế tiếp thường có R² cao do giá hôm nay gần giá ngày mai; vì vậy phải đọc cùng persistence baseline và directional accuracy. R² cao không đồng nghĩa chiến lược có lợi nhuận.",
    "Hướng mở rộng là walk-forward backtest có phí, trượt giá, khoảng tin cậy và kiểm tra thay đổi chế độ thị trường. Mọi đầu ra vẫn cần cảnh báo không phải khuyến nghị đầu tư."], None),

("4.6. KB VÀ RAG CHO GIÁ VÀNG", [
    f"Dữ liệu hợp đồng vàng GC=F được lấy từ {market_sources['gold']}. Khoảng {gold_summary['start']} - {gold_summary['end']} có {gold_summary['observations']} quan sát; giá gần nhất {gold_summary['latest_close']:,.2f} {gold_summary['currency']}/oz.",
    f"Lợi suất toàn kỳ {gold_summary['period_return_pct']:.2f}%, biến động năm hóa {gold_summary['volatility_pct']:.2f}%, RSI14 {gold_summary['rsi14']:.1f}. GC=F là tham chiếu quốc tế, không đồng nhất với giá vàng miếng SJC.",
    "Knowledge Graph mô tả lãi suất Fed, USD, lạm phát, địa chính trị và nhu cầu trú ẩn. Những cạnh này là khung giải thích có điều kiện, không phải quan hệ nhân quả cố định."], "kg_gold.png"),
("4.6.1. DỮ LIỆU, CHỈ BÁO VÀ KNOWLEDGE GRAPH VÀNG", [
    f"MA20 hiện là {gold_summary['ma20']:,.2f}, MA60 là {gold_summary['ma60']:,.2f}. Chuỗi giữ nguyên lịch phiên của nguồn; không nội suy ngày nghỉ. Biến đích là Close phiên kế tiếp.",
    "Đặc trưng dùng cùng một công thức với FPT để so sánh công bằng: giá và độ trễ, return, MA20, MA60, volatility20, RSI14, thay đổi volume, biên độ ngày và chênh lệch Open-Close.",
    "Chuỗi Rủi ro địa chính trị -> Nhu cầu trú ẩn -> Vàng quốc tế minh họa truy vấn nhiều bước. Quan hệ Vàng quốc tế -> Giá vàng SJC chỉ là tham chiếu, cần thêm tỷ giá và cơ chế thị trường trong nước để giải thích chênh lệch."], "gold_market.png"),
("4.6.2. HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH VÀNG", [
    "Ridge, Random Forest và Gradient Boosting được tuning bằng TimeSeriesSplit 5 phần trên 80% đầu chuỗi. 20% cuối được dùng duy nhất cho kiểm thử ngoài mẫu. Tiêu chí lựa chọn là CV MAE.",
    f"Mô hình được chọn: {gold_model_info['Model']}; CV MAE = {gold_model_info['CV_MAE']:.3f}; test MAE = {gold_model_info['MAE']:.3f}; RMSE = {gold_model_info['RMSE']:.3f}; MAPE = {gold_model_info['MAPE']:.2f}%; R² = {gold_model_info['R2']:.3f}.",
    f"Độ đúng chiều là {gold_model_info['Direction_Accuracy']:.1f}%. Persistence baseline có MAE {gold_model_info['Persistence_MAE']:.3f}; mức cải thiện là {gold_model_info['Improvement_vs_Persistence_pct']:.2f}%. Các chỉ số được tính trực tiếp từ phần test theo thời gian."], "gold_model.png"),
("4.6.3. HỎI ĐÁP VĨ MÔ VỚI GRAPHRAG", [
    f"Câu trả lời mẫu: {demo_answers[2]['answer']}",
    "Truy xuất văn bản nêu điều kiện và độ bất định; đồ thị cung cấp các cạnh thường gặp và đường đi qua nhu cầu trú ẩn. Hai nguồn bằng chứng cùng xuất hiện giúp câu trả lời tránh tuyệt đối hóa.",
    "Nếu thay bộ sinh mẫu bằng LLM, prompt phải cấm tự tạo mục tiêu giá hoặc xác suất tăng giảm khi evidence không cung cấp. Citation ID và timestamp nên là trường bắt buộc trong JSON đầu ra."], "code_graphrag.png"),
("4.6.4. GIỚI HẠN VÀ HƯỚNG MỞ RỘNG", [
    "Hợp đồng tương lai có ngày đáo hạn và cơ chế nối chuỗi. Spot XAU/USD, futures GC=F và vàng SJC là ba sản phẩm khác nhau; không thể trộn USD/oz với VND/lượng nếu chưa quy đổi.",
    "Mô hình hiện chưa dùng lãi suất thực, DXY, tỷ giá USD/VND hay vị thế ETF. Hướng mở rộng là gắn timestamp cho các yếu tố vĩ mô, thử độ trễ và lưu confidence cùng evidence trên mỗi cạnh.",
    "Tương tự chứng khoán, dự báo một bước chỉ minh họa quy trình machine learning có kiểm soát. Nó không phải khuyến nghị mua, bán hay nắm giữ vàng."], None),
    ("4.7. DEPLOYMENT", [
        "Notebook sinh ứng dụng web tương tác trong vercel_app. FPT và vàng có khu dự báo riêng; phần tiểu đường được đặt trực tiếp trong chatbot để tạo một luồng sử dụng liền mạch từ nhập thông số, chạy mô hình đến giải thích và hỏi đáp.",
        "Kết quả không phải số viết sẵn. Notebook xuất mean, scale, coefficient và intercept từ đúng pipeline Ridge đã huấn luyện sang JavaScript. Kho tri thức sức khỏe dựa trên WHO, CDC và ADA; mỗi câu trả lời tiếp theo hiển thị cả nguồn và quan hệ Knowledge Graph.",
        f"Ứng dụng đã triển khai tại <link href='{DEPLOY_URL}' color='black'>{DEPLOY_URL}</link>. Mã nguồn và notebook đặt tại <link href='{GITHUB_URL}' color='black'>{GITHUB_URL}</link>."], "web_app_preview.png"),
    ("4.7.1. SUY LUẬN MÔ HÌNH TRÊN TRÌNH DUYỆT", [
        "Phần compute nặng gồm tải dữ liệu, tạo đặc trưng, cross-validation và fit mô hình chạy trong notebook. Phần web nhận tham số cuối cùng của StandardScaler và Ridge. Với từng mẫu x, trình duyệt tính z=(x-mean)/scale, sau đó y=intercept+sum(coef*z). Vì vậy cùng đầu vào cho cùng kết quả như pipeline Python.",
        "Dữ liệu tiểu đường được nạp ở thang đo gốc để biểu mẫu có đơn vị dễ hiểu. Người dùng nhập tuổi, BMI, huyết áp, cholesterol toàn phần, HDL và glucose. Bốn biến chuyên sâu còn lại được giữ ở trung vị của bộ dữ liệu, nhưng mô hình vẫn sử dụng đủ 10 thuộc tính.",
        "Hai biểu mẫu thị trường hiển thị sáu biến dễ diễn giải; năm biến còn lại lấy từ quan sát mới nhất. Kiến trúc tĩnh không lưu API key và không gửi thông số biểu mẫu về máy chủ."], "code_deployment.png"),
    ("4.7.2. CHATBOT DỰ ĐOÁN, TƯ VẤN VÀ HỎI ĐÁP", [
        "Chatbot bắt đầu bằng thông số sức khỏe. Ngoài sáu biến đưa vào mô hình, biểu mẫu hỏi thời điểm đo glucose và HbA1c để nhận xét đúng bối cảnh. Khi người dùng bấm Phân tích và nhận lời khuyên, hệ thống chạy Ridge, so điểm dự đoán với các tứ phân vị của dữ liệu tham chiếu rồi chèn kết quả vào cuộc hội thoại.",
        "Phản hồi được chia thành bốn phần: nhận xét từng chỉ số, khuyến nghị đi khám, dinh dưỡng - sinh hoạt và cảnh báo. Glucose lúc đói, glucose sau hai giờ và HbA1c được đối chiếu với khoảng tham chiếu CDC/ADA; BMI và điểm mô hình được đánh giá riêng. Điểm mô hình là đầu ra hồi quy nghiên cứu, không phải xác suất mắc bệnh.",
        "Sau đánh giá, người dùng có thể hỏi tiếp. Nếu nhập trực tiếp một câu như đường huyết 150, chatbot giải thích rằng cần biết thời điểm đo và trình bày ý nghĩa theo từng bối cảnh thay vì kết luận máy móc. Câu hỏi khác được truy xuất từ kho tri thức và kèm đường bằng chứng. Chatbot không chẩn đoán, kê đơn hoặc thay thế bác sĩ."], None),
    ("ĐÁNH GIÁ TỔNG HỢP", [
        "Kết quả cho thấy một notebook có thể bao quát tải dữ liệu, tiền xử lý, KG, RAG, đánh giá, báo cáo và web. GraphRAG lai cung cấp thêm loại bằng chứng mà Vector RAG không có: quan hệ có tên và đường đi giữa các thực thể.",
        "Đổi lại, KG yêu cầu thiết kế ontology, kiểm soát quan hệ và cập nhật theo thời gian. Chi phí này chỉ đáng bỏ ra khi bài toán cần giải thích, liên kết đa bước hoặc quản trị tri thức dài hạn.",
        "Bài làm đạt mục tiêu nguyên mẫu tái lập. Hạn chế chính là corpus nhỏ, đánh giá ít câu hỏi, generator dạng mẫu và chưa có chuyên gia miền thẩm định. Những giới hạn này được ghi rõ thay vì suy rộng kết quả."], "rag_benchmark.png"),
    ("KẾT LUẬN", [
        "Knowledge Graph và RAG bổ trợ nhau theo cách thực dụng: văn bản đem lại ngữ cảnh linh hoạt, đồ thị đem lại cấu trúc và provenance. Trên ba miền thử nghiệm, cùng một pipeline vẫn giữ được nguồn, thời điểm và cảnh báo riêng.",
        "Điều quan trọng nhất rút ra là không nên coi GraphRAG như một nhãn công nghệ thay thế cho thiết kế dữ liệu. Chất lượng câu trả lời bắt đầu từ việc định nghĩa thực thể, quan hệ, nguồn và phạm vi kết luận.",
        "Notebook kèm theo là sản phẩm có thể chạy lại. PDF và ứng dụng web chỉ là hai cách trình bày cùng một trạng thái kết quả, nhờ đó giảm sai lệch giữa code và báo cáo."], None),
]

# INSIGHT_LAB_REPORT_UPDATES
page_updates = {
"4.4. KB VÀ RAG CHO BỆNH TIỂU ĐƯỜNG": [
    "Dữ liệu diabetes gồm 442 người đã mắc bệnh, 10 thuộc tính nền và điểm tiến triển sau một năm. Đầu ra là hồi quy nghiên cứu, không phải xác suất mắc tiểu đường.",
    f"Dữ liệu được chia thành {len(X_train)} mẫu train, {len(X_cal)} mẫu hiệu chỉnh khoảng dự báo và {len(X_test)} mẫu test. K-fold 5 phần chỉ chạy trên train; so sánh Median baseline, Linear, Ridge, ElasticNet, Huber, SVR RBF, Random Forest và Gradient Boosting. Thêm mô hình phi tuyến không đồng nghĩa chắc chắn tốt hơn; quyết định vẫn dựa trên CV MAE.",
    f"Mô hình được chọn theo CV: {diabetes_model_name}; CV MAE {diabetes_metrics['CV_MAE']:.2f}; test MAE {diabetes_metrics['MAE']:.2f}; R² {diabetes_metrics['R2']:.3f}. MAE của dự báo trung vị là {diabetes_metrics['Baseline_MAE']:.2f}. Khoảng dự báo mục tiêu 90% có độ bao phủ test {diabetes_interval['test_coverage']:.1%}."],
"4.5.2. HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH FPT": [
    "Lịch sử tải được mở rộng từ 2 năm lên 5 năm. Chín họ cấu hình gồm Persistence, Ridge giá, Median lợi suất, Ridge lợi suất, Huber, SVR RBF, Random Forest và hai dạng Gradient Boosting. Tuning độ co lợi suất, cửa sổ học gần nhất và siêu tham số trên TimeSeriesSplit 5 phần, mỗi phần xác nhận 63 phiên, gap=1. Tập test giữ nguyên mốc của bản 2.0.",
    f"Chia theo thời gian: {stock_model_info['train_size']} train, {stock_model_info['calibration_size']} hiệu chỉnh, {stock_model_info['test_size']} test; bỏ một quan sát tại mỗi ranh giới. Mô hình {stock_model_info['Model']} đạt MAE {stock_model_info['MAE']:.2f}, R² {stock_model_info['R2']:.3f}; baseline MAE {stock_model_info['Persistence_MAE']:.2f}. Cải thiện trên test {stock_model_info['Improvement_vs_Persistence_pct']:.2f}%.",
    f"Test lịch sử từ {stock_model_info['test_start']} đến {stock_model_info['test_end']}; MAE bản trước 728.13 VND. Bao phủ khoảng dự báo lịch sử: {market_details['Chứng khoán FPT']['interval']['test_coverage']:.1%}. Kết quả hiện tại chưa cho thấy mô hình học máy thắng Persistence. Dữ liệu chuỗi thời gian không thỏa giả định trao đổi như mẫu độc lập nên mức 90% chỉ là mục tiêu, không phải bảo đảm cho tương lai."],
"4.6.2. HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH VÀNG": [
    "Vàng dùng cùng quy trình lựa chọn có baseline và mô hình lợi suất như FPT. Không ép chọn mô hình phức tạp nếu Persistence có CV MAE thấp hơn. Train, hiệu chỉnh và test nằm theo đúng thứ tự thời gian, có khoảng cách một phiên ở ranh giới.",
    f"Mô hình được chọn: {gold_model_info['Model']}; CV MAE {gold_model_info['CV_MAE']:.3f}; test MAE {gold_model_info['MAE']:.3f}; R² {gold_model_info['R2']:.3f}. Baseline MAE {gold_model_info['Persistence_MAE']:.3f}; cải thiện {gold_model_info['Improvement_vs_Persistence_pct']:.2f}%.",
    f"Test từ {gold_model_info['test_start']} đến {gold_model_info['test_end']}; bản trước có MAE 50.4701. Mức giảm sai số so với bản trước là {(1-gold_model_info['MAE']/50.47005460824505)*100:.2f}%, nhưng vẫn cần đối chiếu baseline. Khoảng bootstrap khối 5 phiên, 2.000 lần lặp cho mức cải thiện so với baseline: {gold_model_info['Improvement_CI95_pct'][0]:.2f}% đến {gold_model_info['Improvement_CI95_pct'][1]:.2f}%. Khoảng chứa 0 chưa chứng minh cải thiện rõ ràng. GC=F không phải vàng SJC."],
"4.7. DEPLOYMENT": [
    "Insight Lab 2.0 chia thành khu Dự đoán với ba tab Tiểu đường, FPT, Vàng và khu Trợ lý sức khỏe riêng. Người dùng xem kết quả tại từng tab; khi muốn thảo luận, chủ động chọn chia sẻ kết quả sức khỏe cho chatbot.",
    "Web hiển thị biểu đồ lịch sử giá, dự báo điểm, khoảng sai số hiệu chỉnh và so sánh baseline. Đầu vào trống, sai đơn vị hoặc vượt giới hạn được kiểm tra trước suy luận. Giá trị ngoài phạm vi train được đánh dấu để tránh hiểu sai mức tin cậy.",
    f"Web: <link href='{DEPLOY_URL}' color='black'>{DEPLOY_URL}</link>. GitHub: <link href='{GITHUB_URL}' color='black'>{GITHUB_URL}</link>. Sơ đồ dưới đây minh họa bố cục, không phải ảnh chụp màn hình."],
"4.7.1. SUY LUẬN MÔ HÌNH TRÊN TRÌNH DUYỆT": [
    "Notebook xuất cấu trúc estimator được chọn: scaler, hệ số tuyến tính, vector hỗ trợ và kernel RBF của SVR, cấu trúc cây hoặc công thức Persistence/lợi suất. JavaScript dùng cùng đặc trưng và tham số. Các vector đối chiếu Python/JavaScript được lưu để kiểm tra sai số đóng gói.",
    "Mô hình sức khỏe có sáu trường chính và bốn trường bổ sung. Trường bổ sung trống dùng trung vị tập train và được công bố ở kết quả. HbA1c và bối cảnh đo dùng cho nhận xét xét nghiệm riêng, không bị đưa nhầm vào mô hình chưa học chúng. Điểm tiến triển không được dùng quyết định chẩn đoán hoặc cấp cứu.",
    "Cấu hình thị trường được chọn trên dữ liệu trước test rồi fit lại cho web, chừa 63 phiên gần nhất để hiệu chỉnh khoảng dự báo và bỏ cách một phiên. Chỉ số test lịch sử thuộc bản fit trước đó, không phải đánh giá ngoài mẫu của bản refit. Web hiển thị riêng hai giai đoạn. Cảnh báo ngoài miền của mô hình lợi suất xét đặc trưng tương đối, không chỉ mức giá tuyệt đối. Web không gửi hồ sơ sức khỏe ra ngoài."],
"4.7.2. CHATBOT DỰ ĐOÁN, TƯ VẤN VÀ HỎI ĐÁP": [
    "Chatbot nhận đường huyết, HbA1c, BMI, tuổi, huyết áp tâm thu/tâm trương, huyết áp trung bình, cholesterol toàn phần, LDL, HDL, triglyceride và các đặc trưng nghiên cứu s4/s5. Câu ‘huyết áp 90’ được hỏi lại loại số đo; thiếu đơn vị thì hỏi tiếp. Một tin nhắn có thể chứa nhiều chỉ số, đơn vị được gắn riêng với từng giá trị để tránh lấy nhầm mmol/L của HDL cho glucose.",
    "Các ngưỡng xét nghiệm tham khảo NIDDK/ADA. Đường huyết thấp được xử lý trước bảng phân loại đường cao. Sau bữa ăn thông thường không bị đánh đồng với nghiệm pháp 75 g glucose. Thai kỳ và trẻ em cần bối cảnh riêng. Câu hỏi kê liều thuốc được chuyển thành hướng dẫn trao đổi với bác sĩ.",
    "Kho tri thức mở rộng thành 20 chủ đề, bổ sung nguồn AHA về huyết áp, MedlinePlus về mỡ máu và Mayo Clinic về xét nghiệm cholesterol. Nội dung, đường quan hệ KG và danh mục nguồn được xuất ra health_knowledge.json. BM25 truy xuất kết hợp bộ nhận diện số đo; chưa kết nối LLM ngoài. Khi chia sẻ form, chatbot nhận đủ các trường đã nhập, nhưng loại giá trị trung vị thay thế khỏi nhận xét y khoa. Mã giới tính 1/2 không tự dùng chọn ngưỡng HDL; s5 không tự quy đổi thành xét nghiệm triglyceride. Hồ sơ chỉ giữ trong phiên."],
}
pages = [(title,page_updates.get(title,paras),fig) for title,paras,fig in pages]

for title, paragraphs, figure in pages:
    story.append(P(title, "VNChapter" if title.count(".") <= 1 else "VNHeading"))
    for paragraph in paragraphs:
        story.append(P(paragraph))
    if figure:
        img_path = FIG_DIR / figure
        if img_path.exists():
            img = report_image(img_path)
            story += [Spacer(1, .15*cm), img, P(f"Hình: {title}", "VNCaption")]
    story.append(PageBreak())


# Bảng kết quả chi tiết
story += [P("PHỤ LỤC A - TỔNG HỢP HUẤN LUYỆN BA MIỀN", "VNChapter")]
training_rows = [["Miền", "Mô hình được chọn", "CV MAE", "Test MAE", "Test R²"]] + [
    ["Tiểu đường", diabetes_metrics["Model"], f"{diabetes_metrics['CV_MAE']:.3f}",
     f"{diabetes_metrics['MAE']:.3f}", f"{diabetes_metrics['R2']:.3f}"],
    ["Chứng khoán FPT", stock_model_info["Model"], f"{stock_model_info['CV_MAE']:.3f}",
     f"{stock_model_info['MAE']:.3f}", f"{stock_model_info['R2']:.3f}"],
    ["Giá vàng GC=F", gold_model_info["Model"], f"{gold_model_info['CV_MAE']:.3f}",
     f"{gold_model_info['MAE']:.3f}", f"{gold_model_info['R2']:.3f}"],
]
story += [report_table(training_rows, [3.2*cm, 4.0*cm, 2.7*cm, 2.7*cm, 2.7*cm]),
          Spacer(1, .45*cm),
          P("Mô hình được chọn theo CV MAE trên tập train. Chỉ số test được tính đúng một lần trên phần dữ liệu giữ lại."),
          PageBreak()]

story += [P("PHỤ LỤC B - KẾT QUẢ TRUY XUẤT", "VNChapter")]

eval_data = [["Câu hỏi", "Miền", "P@3", "Graph", "Hybrid"]] + [
    [P(r["query"], "VNSmall"), r["expected_domain"], f"{r['vector_precision@3']:.2f}", str(r["graph_evidence"]), f"{r['hybrid_evidence_score']:.2f}"]
    for _, r in evaluation_df.iterrows()
]
story += [report_table(eval_data, [8.4*cm, 2.0*cm, 1.6*cm, 1.6*cm, 1.8*cm], font_size=7.5),
          Spacer(1, .5*cm), P("Bảng này là phép kiểm tra hồi quy cho pipeline. Nó không thay thế đánh giá câu trả lời bởi chuyên gia."), PageBreak()]

story += [P("PHỤ LỤC C - MẪU BỘ BA TRI THỨC", "VNChapter")]
triple_data = [["Subject", "Predicate", "Object"]] + [
    [P(str(r.Subject), "VNSmall"), P(str(r.Predicate), "VNSmall"), P(str(r.Object), "VNSmall")]
    for r in triples_df.head(20).itertuples()
]
story += [report_table(triple_data, [4.9*cm, 5.2*cm, 5.4*cm], font_size=7.2), PageBreak()]

references = [
    "[1] W3C, RDF 1.1 Concepts and Abstract Syntax, W3C Recommendation, 2014.",
    "[2] T. Berners-Lee, J. Hendler, O. Lassila, The Semantic Web, Scientific American, 2001.",
    "[3] K. Guu et al., REALM: Retrieval-Augmented Language Model Pre-Training, ICML, 2020.",
    "[4] P. Lewis et al., Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks, NeurIPS, 2020.",
    "[5] D. Edge et al., From Local to Global: A Graph RAG Approach to Query-Focused Summarization, 2024.",
    "[6] scikit-learn developers, Diabetes dataset and TfidfVectorizer documentation.",
    "[7] NetworkX developers, MultiDiGraph documentation.",
    "[8] Yahoo Finance chart endpoint, mã FPT.VN và GC=F; truy cập tại thời điểm chạy notebook.",
    "[9] World Health Organization, Diabetes fact sheet, https://www.who.int/news-room/fact-sheets/detail/diabetes.",
    "[10] Centers for Disease Control and Prevention, Diabetes Meal Planning, https://www.cdc.gov/diabetes/healthy-eating/diabetes-meal-planning.html.",
    "[11] American Diabetes Association, Eating Healthy, https://diabetes.org/food-nutrition/eating-healthy.",
    "[12] Centers for Disease Control and Prevention, Diabetes Testing, https://www.cdc.gov/diabetes/diabetes-testing/index.html.",
    "[13] American Diabetes Association, Diabetes Diagnosis, https://diabetes.org/about-diabetes/diagnosis.",
    "[14] Vercel Documentation, Static Deployments and CLI.",
    "[18] American Heart Association, Understanding Blood Pressure Readings (2025), https://www.heart.org/en/health-topics/high-blood-pressure/understanding-blood-pressure-readings.",
    "[19] MedlinePlus, Cholesterol Levels: What You Need to Know, https://medlineplus.gov/cholesterollevelswhatyouneedtoknow.html.",
    "[20] Mayo Clinic, Cholesterol test, https://www.mayoclinic.org/tests-procedures/cholesterol-test/about/pac-20384601.",
    "[17] NIDDK, Diabetes Tests and Diagnosis, https://www.niddk.nih.gov/health-information/diabetes/overview/tests-diagnosis.",
    f"[15] Mã nguồn bài làm: <link href='{GITHUB_URL}' color='black'>{GITHUB_URL}</link>.",
    f"[16] Web triển khai: <link href='{DEPLOY_URL}' color='black'>{DEPLOY_URL}</link>.",
]
story += [P("TÀI LIỆU THAM KHẢO", "VNChapter")]
references.append("[21] ACOG, Preeclampsia and High Blood Pressure During Pregnancy, https://www.acog.org/womens-health/faqs/preeclampsia-and-high-blood-pressure-during-pregnancy.")
references.sort(key=lambda value:int(value.split("]")[0][1:]))
styles.add(ParagraphStyle(name="VNReference",parent=styles["VNBody"],fontSize=10.5,leading=14,spaceAfter=5,alignment=TA_LEFT))
story += [P(ref,"VNReference") for ref in references]
story += [Spacer(1, .5*cm), P(f"Báo cáo được sinh tự động từ notebook lúc {RUN_AT:%d/%m/%Y %H:%M:%S %Z}. Dữ liệu nguồn: FPT — {market_sources['stock']}; vàng — {market_sources['gold']}.", "VNSmall")]

doc = ReportDoc(str(REPORT_PATH))
doc.multiBuild(story)
print("Đã tạo báo cáo:", REPORT_PATH)

In [10]:
# 6. KIỂM TRA ĐẦU RA VÀ TRIỂN KHAI TÙY CHỌN
assert REPORT_PATH.exists() and REPORT_PATH.stat().st_size > 100_000
assert (WEB_DIR / "index.html").exists()
assert (DATA_DIR / "knowledge_graph_triples.csv").exists()

try:
    from pypdf import PdfReader
except Exception:
    bundled_site = Path("/home/jellalaz/.cache/codex-runtimes/codex-primary-runtime/dependencies/python")
    if bundled_site.exists() and str(bundled_site) not in sys.path:
        sys.path.append(str(bundled_site))
    from pypdf import PdfReader
page_count = len(PdfReader(str(REPORT_PATH)).pages)
print(f"PDF hợp lệ: {page_count} trang, {REPORT_PATH.stat().st_size/1024/1024:.2f} MB")
print("Web package:", WEB_DIR)

# Để deploy thực sự: đặt biến môi trường VERCEL_TOKEN và chuyển cờ thành True.
# Token tuyệt đối không được ghi trực tiếp vào notebook.
DEPLOY_TO_VERCEL = False
if DEPLOY_TO_VERCEL:
    token = os.getenv("VERCEL_TOKEN")
    if not token:
        raise RuntimeError("Thiếu biến môi trường VERCEL_TOKEN")
    subprocess.run(
        ["npx", "vercel", "--prod", "--yes", "--token", token],
        cwd=WEB_DIR, check=True
    )
else:
    print("Production URL cấu hình trong báo cáo:", DEPLOY_URL)

PDF hợp lệ: 41 trang, 3.68 MB
Web package: /home/jellalaz/Documents/Jellalaz/DATA_CODE/PYTHON/RAG/vercel_app
Production URL cấu hình trong báo cáo: https://rag-httm-chatbot.vercel.app
